# The Enrolled ICMS Debt Portfolio of São Paulo: Stock, Recovery and Macroeconomic Conditions
## Supplementary Material — Exploratory Data Analysis

This notebook is the supplementary material of the paper. It documents, step by step, the data preparation and every exploratory and associational analysis reported in the main text, and it generates all figures (`fig_*`) and tables (`tab_*`) under the labels used in the manuscript.

The analysis proceeds in five parts. **Part A** describes the full portfolio of enrolled tax debt across tax types and motivates the restriction of the study to ICMS. **Part B** characterises the ICMS portfolio: its evolution, concentration, age structure, recovery, payment behaviour, seasonality and the effect of settlement programmes. **Part C** examines the statistical properties of the series that determine the appropriate measures of association. **Part D** relates the portfolio to macroeconomic conditions. **Part E** discusses limitations, ethical aspects and directions for future research.

## Research questions
- **RQ1.** How do volume, monetary value, age and recovery differ across tax types, and do these differences justify an ICMS-only analysis?
- **RQ2.** How are the ICMS stock, its age structure and its payments distributed over the life of the debt and over calendar time, including the windows of the 2019 Special Instalment Programme (PEP) and the 2024 Acordo Paulista?
- **RQ3.** Which statistical properties — distribution, linearity, stationarity and persistence — condition the choice of association measures?
- **RQ4.** Are monthly and quarterly lagged macroeconomic indicators significantly associated with the ICMS stock, payments and recovery?
  - *H4a.* Associations are monotone but non-linear, so that rank correlation is preferred to linear correlation.
  - *H4b.* The Selic rate is positively associated with both the stock and recovery, consistent with firms using unpaid taxes as a source of financing.

## Glossary
Terms are grouped by theme. Each entry gives a plain definition and, where relevant, how the quantity is measured in this dataset.

**Institutional and data terms**

| Term | Definition | Measurement in this dataset |
|---|---|---|
| CDA (*Certidão de Dívida Ativa*) | Certificate that formally enrols an unpaid tax debt in the state's active debt, after which it can be collected administratively or in court. | One row per CDA identifier (`ID_DEBITO`) in each monthly snapshot of `DEBITO`. |
| Active debt (*dívida ativa*) | The set of tax debts that have been enrolled and are collected by the State Attorney General's Office (PGE-SP). | All CDAs present in `DEBITO`. |
| Snapshot | A monthly image of the portfolio listing every CDA outstanding in that month. | `DEBITO` rows with a given extraction month (`MES_EXTRACAO`), January 2016 to March 2026. |
| ICMS | State tax on the circulation of goods and on interstate and inter-municipal transport and communication services; the main state tax in Brazil. | Raw debt types whose name contains "ICMS". |
| IPVA, ITCMD | State taxes on motor-vehicle ownership (IPVA) and on inheritance and donations (ITCMD). | Raw debt types grouped by a text rule (A.3). |
| Raw debt type | The debt category recorded in the source (`TIPO_DEBITO`). | Five raw types for ICMS. |
| ICMS Declarado | ICMS that the firm declared itself but did not pay. | Raw type *ICMS Declarado*; origin *declared*. |
| ICMS Autuação / AIIM | ICMS assessed by the tax authority through an infraction notice (*Auto de Infração e Imposição de Multa*, AIIM), usually after an audit. | Raw type *ICMS Autuação*; origin *assessed*. |
| Simples Nacional | Simplified joint tax regime for small firms; *ICMS Autuação Simples Nacional* is ICMS assessed on firms in this regime. | Raw type *ICMS Autuação Simples Nacional*; origin *assessed*. |
| Fecoep | State fund for poverty alleviation financed by an ICMS surcharge on selected goods. | Raw type *ICMS Fecoep*; origin *other*. |
| GIA (*Guia de Informação e Apuração do ICMS*) | Periodic ICMS return; *Alteração de GIA* refers to debts arising from amendments to that return. | Raw type *ICMS Alteração de GIA*; origin *other*. |
| Origin | Grouping of the raw ICMS types into *declared*, *assessed* and *other*, justified in B.2. | Variable `ICMS_ORIGIN`. |
| GARE | State payment slip through which CDA payments are made. | Rows of `ARRECADACAO` linked to a CDA by `ID_DEBITO`; amount `VALOR_RECEITA`, date `DATA_ARRECADACAO_GARE`. |
| Judicial filing (*ajuizamento*) | Filing of a tax-enforcement lawsuit (*execução fiscal*) to collect a CDA in court. | `STATUS_AJUIZAMENTO_DEBITO`: filed, cleared for filing, awaiting clearance. |
| Administrative situation | Administrative status of the CDA (for example enrolled, suspended, stayed). | `SITUACAO_DEBITO`. |
| Settlement programme | Time-limited programme offering discounts or instalments on enrolled debt: the 2019 Special Instalment Programme (PEP) and the 2024 Acordo Paulista. | Programme windows July–December 2019 and April–December 2024. |
| CNPJ, CNPJ root | National register number of a legal entity; the first eight digits (root) identify the firm, the full number an establishment. | `CNPJ_ROOT`; stored only as a salted hash in outputs. |
| CPF | Register number of a natural person. | CDAs with a CPF are excluded (1.3.1). |

**Portfolio measures**

| Term | Definition | Measurement in this dataset |
|---|---|---|
| Value of a CDA | Amount owed, net of fees. | `VALOR_SEM_HONORARIOS`, in R$. |
| Stock | Total value of outstanding CDAs at a date. | Sum of CDA values in a snapshot; annual stock = last snapshot of the year. |
| Collection | Amount paid on CDAs during a period. | Sum of `VALOR_RECEITA` in `ARRECADACAO` over the period. |
| New enrolments | CDAs whose enrolment date falls in the period. | Count by `DATA_INSCRICAO`. |
| Enrolment date, age | Date on which the debt was enrolled; age is the time elapsed since then. | `DATA_INSCRICAO`; age in years at the reference date. Implausible dates are set to missing. |
| Age band | Grouping of age into ≤1, 1–3, 3–5, 5–10 and >10 years. | Right-closed intervals of age. |
| Enrolment cohort | All CDAs enrolled in the same calendar year. | Year of `DATA_INSCRICAO`. |
| Recovery rate (CDA level) | Share of a CDA's stock paid in a year. | Amount paid in the year divided by year-end stock, per CDA-year. |
| Portfolio recovery rate | Recovery rate of the portfolio as a whole, weighting CDAs by their stock. | Total collection divided by total stock, per year or month. |
| Non-accumulated recovery | Recovery within a single period. | As above, for one year or month. |
| Accumulated recovery | Cumulative recovery of a CDA or cohort since enrolment. | Cumulative amount paid up to horizon *h* divided by the enrolled value. |
| Payment indicator | Whether a CDA records at least one payment in a period. | `paid_flag` (CDA-year) or `ever_paid` (since enrolment). |
| Recoverable stock | Stock excluding debts deemed irrecoverable under an operational criterion. | Enrolled CDAs, excluding those older than ten years with no payment ever. |
| CDA × year panel | Dataset with one row per CDA and year. | Built in 1.4. |
| Top 1% (top set) | The 1% of CDAs with the largest values. | Ranked by value in the March 2026 snapshot (B.5.1); alternative sets in B.5.1. |

**Statistical terms**

| Term | Definition | Use in this notebook |
|---|---|---|
| Logarithmic scale | Axis on which equal distances represent equal ratios. | Values and recovery rates spanning several orders of magnitude. |
| ECDF | Empirical cumulative distribution function: share of observations not exceeding each value. | Value distributions (A.4.5, B.2.2). |
| Interquartile range (IQR) | Range between the 25th and 75th percentiles. | Box plots and dot-and-range plots. |
| Lorenz curve, Gini coefficient | Curve of the cumulative share of a total held by the cumulative share of units; the Gini (0 to 1) measures inequality. | Concentration of the stock (B.4, B.5.13). |
| Bootstrap | Resampling with replacement to approximate the sampling distribution of a statistic. | Confidence intervals (B.1.4); moving-block bootstrap for time series (D.5). |
| Kruskal–Wallis, Mann–Whitney *U*, Dunn test | Rank-based tests of differences between groups, without assuming normality. | Comparisons across raw types and groups (B.2, B.3, B.5.22). |
| Rank-biserial correlation | Effect size for Mann–Whitney: difference between the probabilities that one group exceeds the other and vice versa (−1 to 1). | Pairwise comparisons. |
| Epsilon-squared | Effect size for Kruskal–Wallis: share of rank variance explained by the groups. | B.2.8. |
| Kolmogorov–Smirnov, Anderson–Darling | Tests comparing whole distributions. | B.2.8. |
| Chi-square test, Fisher exact test, Cramér's *V* | Tests of independence between categorical variables and a strength measure (0 to 1). | B.2.9, B.5.22. |
| Benjamini–Hochberg *q*-value | *p*-value adjusted to control the false discovery rate across many tests. | All multiple comparisons; threshold *q* < 0.05. |
| Kaplan–Meier estimator, censoring, log-rank test | Nonparametric estimate of time-to-event when some events are not yet observed (censored), and a test comparing two curves. | Time to first payment (B.5.9). |
| Pearson *r*, Spearman ρ, Kendall τ | Linear correlation, rank correlation, and concordance-based rank correlation. | Choice rule in C.2. |
| LOWESS | Locally weighted regression smoother. | C.2.1. |
| RESET, Harvey–Collier | Tests of linear functional form. | C.2.2. |
| Stationarity, unit root, ADF, KPSS, Zivot–Andrews | A stationary series has constant mean and variance; a unit root implies persistent shocks. ADF and KPSS test opposite null hypotheses; Zivot–Andrews allows one break. | C.3.1. |
| ACF, PACF, Ljung–Box | Autocorrelation and partial autocorrelation functions; joint test of no autocorrelation. | C.3.3, C.4.2. |
| Pre-whitening | Filtering series by an autoregressive model before computing cross-correlations. | C.3.4, D.1.2. |
| Seasonality, STL | Regular pattern repeating every year; STL decomposes a series into trend, seasonal and remainder components. | B.10. |
| Cross-correlation function (CCF) | Correlation between one series and lagged values of another. | D.1.2. |
| Newey–West (HAC) standard errors | Standard errors robust to heteroskedasticity and autocorrelation. | D.5, D.6, D.7. |
| Logit model, odds ratio | Regression for a binary outcome; the odds ratio is the multiplicative change in the odds per unit of a regressor. | D.6.1, D.6.4. |
| Fixed effects (year) | Dummy variables absorbing everything common to a year. | D.6.1, D.6.2. |
| Variance inflation factor (VIF) | Measure of collinearity among regressors. | D.6.4. |
| Principal component analysis (PCA) | Linear combinations of variables capturing most of their joint variance. | B.5.18, D.6.5. |
| AR(1), ARDL | First-order autoregression; autoregressive distributed-lag model. | D.6.3, D.7.1. |
| Granger causality | Past values of one series improve the prediction of another. | D.7.2. |
| Gower distance | Dissimilarity for mixed numerical and categorical features (0 to 1). | B.5.18. |
| t-SNE | Nonlinear projection preserving local neighbourhoods, for visualisation only. | B.5.18. |
| Agglomerative clustering, HDBSCAN | Hierarchical merging of clusters; density-based clustering that flags noise. | B.5.19. |
| Silhouette coefficient, adjusted Rand index | Cluster separation (−1 to 1); agreement between two partitions corrected for chance. | B.5.19. |
| Medoid | The member of a group with the smallest total distance to the others. | B.5.20, B.5.21. |
| Dynamic time warping (DTW) | Distance between sequences that allows stretching of the time axis. | B.5.21. |
| Intraclass correlation (ICC) | Share of variance lying between groups (here, debtors). | B.5.15. |
| Jaccard index | Size of the intersection of two sets divided by the size of their union. | B.5.2. |

**Computational terms**

| Term | Definition | Use in this notebook |
|---|---|---|
| Parquet | Columnar file format that allows reading only the columns and row groups needed. | Storage of extracts and cached results. |
| DuckDB | Embedded analytical database that processes data larger than memory by spilling to disk. | Extraction and large aggregations. |
| Lazy evaluation, streaming | A query is planned before execution, so filters and column selection are pushed to the data source; streaming processes data in batches. | Polars queries throughout. |
| Memory-mapped file | File on disk accessed as if it were an array in memory, loading only the parts that are used. | Distance matrices in B.5. |
| Cache key | Identifier of a stored result derived from the step, its parameters, its code and the source data. | Reuse of results across runs (0.3). |
| Random seed | Fixed starting value of the random-number generator, which makes random draws reproducible. | All sampling and resampling. |

## 0. Computational environment

*Terms introduced in this section:* Parquet, DuckDB, lazy evaluation, streaming, memory-mapped file, cache key, random seed. Definitions are given in the Glossary.

#### 0.1 Paths and configuration
All locations are read from environment variables, optionally loaded from a `.env` file (see `paper_eda_v1.env.example`), so that the notebook runs unchanged on any machine. Raw and intermediate data reside on an external drive; figures and tables are written inside the repository, next to the manuscript. Execution stops with an explicit message when a variable is missing or a drive is not mounted.

In [ ]:
from __future__ import annotations
import os, sys, gc, random, time, json, hashlib, secrets
from pathlib import Path
from datetime import datetime, timezone, date, timedelta
from typing import Any, Callable
os.environ.setdefault("POLARS_MAX_THREADS", "2")  # must be set before polars is imported
import numpy as np
import polars as pl
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

def env_path(name: str, create: bool = False) -> Path:
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"Environment variable {name} is not set (see paper_eda_v1.env.example).")
    path = Path(value).expanduser()
    if create:
        path.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        raise RuntimeError(f"{name}={path} does not exist; is the external drive mounted?")
    return path

def find_repo(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists() and (p / "projects").exists():
            return p
    raise RuntimeError("Repository root not found.")

REPO = find_repo()

def repo_path(name: str, default: str) -> Path:
    path = Path(os.environ.get(name, default))
    path = path if path.is_absolute() else REPO / path
    path.mkdir(parents=True, exist_ok=True)
    return path

DATA_ROOT    = env_path("CEMEPI_DATA_ROOT")
DUCKDB_MAIN  = DATA_ROOT / os.environ.get("CEMEPI_DUCKDB_MAIN", "api_ctf/cemepi.duckdb")
DUCKDB_MACRO = DATA_ROOT / os.environ.get("CEMEPI_DUCKDB_MACRO", "api_ctf/macro.duckdb")
INTERIM      = env_path("CEMEPI_INTERIM", create=True)
CACHE        = env_path("CEMEPI_CACHE", create=True)
FIG_DIR      = repo_path("CEMEPI_FIG_DIR", "projects/macro/manuscript/paper_eda_v1/figures")
TAB_DIR      = repo_path("CEMEPI_TAB_DIR", "projects/macro/manuscript/paper_eda_v1/tables")
ICMS_DIR     = INTERIM / "icms"
GARE_DIR     = INTERIM / "gare_icms"
ALL_DIR      = INTERIM / "all_taxes_snapshot"
TMP          = CACHE / "tmp"
for p in (ICMS_DIR, GARE_DIR, ALL_DIR, TMP):
    p.mkdir(parents=True, exist_ok=True)
for f in (DUCKDB_MAIN, DUCKDB_MACRO):
    if not f.exists():
        raise RuntimeError(f"Database not found: {f}")
print(f"data: {DATA_ROOT}\ninterim: {INTERIM}\ncache: {CACHE}\nfigures: {FIG_DIR}\ntables: {TAB_DIR}")

#### 0.2 Memory management
The raw debt table contains about 800 million monthly snapshot rows, whereas the analysis machine has 8 GB of RAM. All queries are therefore lazy (projection and predicate push-down), aggregations use the streaming engine, extraction is performed one calendar year at a time, and intermediate results are written to disk with `sink_parquet` rather than held in memory.

In [ ]:
os.environ.setdefault("POLARS_TEMP_DIR", str(TMP))

def collect(lf: pl.LazyFrame) -> pl.DataFrame:
    return lf.collect(engine="streaming")

def duck(db: Path = DUCKDB_MAIN):
    import duckdb
    con = duckdb.connect(str(db), read_only=True)
    con.execute(f"SET memory_limit='1500MB'; SET threads=2; SET temp_directory='{TMP}'; "
                "SET preserve_insertion_order=false")
    return con

#### 0.3 Computational reproducibility and result caching
Every extraction and intermediate result is stored once on the external drive and reused in later runs. Each stored object is identified by the identifier of the analysis step and a hash of (i) its parameters, (ii) the text of its query or the source code of the function that produces it, (iii) a version tag, and (iv) a fingerprint of the source data (size and modification time of the database files). Any change in parameters, code or data therefore yields a new key and triggers recomputation, while unchanged steps are read from disk. Data frames are stored as Parquet and read back lazily; test results and small tables are stored as JSON or Parquet. A manifest records, for each object, its key, inputs, creation time and run time. Recomputation can be forced globally (`CEMEPI_FORCE_REFRESH=1`) or for selected steps (`CEMEPI_FORCE_STEPS=B.6.3,D.7.1`). Figures are not cached: they are re-rendered from cached data, which is inexpensive.

In [ ]:
CACHE_DIR = CACHE / "results"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = CACHE_DIR / "manifest.jsonl"
CACHE_VERSION = "v2"
FORCE_ALL = os.environ.get("CEMEPI_FORCE_REFRESH", "0") == "1"
FORCE_STEPS = {s.strip() for s in os.environ.get("CEMEPI_FORCE_STEPS", "").split(",") if s.strip()}

def source_fingerprint() -> dict:
    return {f.name: [f.stat().st_size, int(f.stat().st_mtime)] for f in (DUCKDB_MAIN, DUCKDB_MACRO)}

SOURCE_FP = source_fingerprint()

def _canonical(obj) -> str:
    return json.dumps(obj or {}, sort_keys=True, separators=(",", ":"), ensure_ascii=True, default=str)

def _sha(text: str | None) -> str:
    return hashlib.sha256(text.encode()).hexdigest() if text else ""

def key_from_parts(step: str, params: dict | None, version: str, sql_sha: str,
                   cache_version: str = CACHE_VERSION, source: dict | None = None) -> str:
    # Explicit inputs only: step id, canonical params, step version, SQL hash, global version, source fingerprint
    payload = _canonical({"step": step, "params": params or {}, "version": str(version), "sql": sql_sha,
                          "cache_version": cache_version, "source": source or SOURCE_FP})
    return f"{step.replace('.', '_').replace(':', '-')}__{hashlib.sha256(payload.encode()).hexdigest()[:16]}"

def cache_key(step: str, params: dict | None = None, version: str = "1", sql: str | None = None) -> str:
    return key_from_parts(step, params, version, _sha(sql))

def _must_refresh(step: str, refresh: bool = False) -> bool:
    return refresh or FORCE_ALL or step.split(":")[0] in FORCE_STEPS

def _log(step: str, key: str, path: Path, params: dict | None, version: str, sql: str | None, seconds: float) -> None:
    with MANIFEST.open("a") as fh:
        fh.write(json.dumps({"step": step, "key": key, "path": str(path), "params": params or {},
                             "version": str(version), "sql_sha": _sha(sql), "cache_version": CACHE_VERSION,
                             "source": SOURCE_FP, "seconds": round(seconds, 2),
                             "created": datetime.now(timezone.utc).isoformat()}, default=str) + "\n")

def _report(step: str, hit: bool) -> None:
    print(f"[cache] {step}: {'HIT' if hit else 'MISS, computing'}", flush=True)

def cached(step: str, fn: Callable[[], pl.LazyFrame | pl.DataFrame], params: dict | None = None,
           version: str = "1", sql: str | None = None, refresh: bool = False, allow_empty: bool = False) -> pl.LazyFrame:
    key = cache_key(step, params, version, sql)
    path = CACHE_DIR / f"{key}.parquet"
    hit = path.exists() and not _must_refresh(step, refresh)
    _report(step, hit)
    if hit:
        return pl.scan_parquet(path)
    t0, tmp = time.time(), path.with_suffix(".tmp.parquet")
    obj = fn()
    if isinstance(obj, pl.LazyFrame):
        obj.sink_parquet(tmp)
    else:
        obj.write_parquet(tmp)
    if not allow_empty and pl.scan_parquet(tmp).select(pl.len()).collect().item() == 0:
        tmp.unlink()
        raise ValueError(f"{step}: empty result, not cached (pass allow_empty=True if expected)")
    tmp.replace(path)
    _log(step, key, path, params, version, sql, time.time() - t0)
    return pl.scan_parquet(path)

def cached_json(step: str, fn: Callable[[], Any], params: dict | None = None,
                version: str = "1", refresh: bool = False) -> Any:
    key = cache_key(step, params, version)
    path = CACHE_DIR / f"{key}.json"
    hit = path.exists() and not _must_refresh(step, refresh)
    _report(step, hit)
    if not hit:
        t0 = time.time()
        obj = fn()
        tmp = path.with_suffix(".tmp.json")
        tmp.write_text(json.dumps(obj, default=str, indent=1))
        tmp.replace(path)
        _log(step, key, path, params, version, None, time.time() - t0)
    return json.loads(path.read_text())

def cached_partitioned(step: str, out_dir: Path, years, build_year: Callable[[int, Path], None],
                       params: dict | None = None, version: str = "1", refresh: bool = False) -> pl.LazyFrame:
    for y in years:
        part = out_dir / f"year={y}"
        key = cache_key(f"{step}:{y}", params, version)
        marker = part / f"_KEY.{CACHE_VERSION}"
        hit = marker.exists() and marker.read_text() == key and not _must_refresh(step, refresh)
        _report(f"{step}:{y}", hit)
        if hit:
            continue
        part.mkdir(parents=True, exist_ok=True)
        for f in part.glob("*"):
            f.unlink()
        t0 = time.time()
        build_year(y, part)
        marker.write_text(key)
        _log(f"{step}:{y}", key, part, params, version, None, time.time() - t0)
        print(f"{step} year {y}: {time.time() - t0:,.1f} s")
    return pl.scan_parquet(out_dir / "**/*.parquet", hive_partitioning=True)

def cached_memmap(step: str, n: int, build: Callable[[np.memmap], None], params: dict | None = None,
                  version: str = "1", refresh: bool = False) -> np.memmap:
    params = {**(params or {}), "n": n}
    key = cache_key(step, params, version)
    path = CACHE_DIR / f"{key}.f32"
    hit = path.exists() and not _must_refresh(step, refresh)
    _report(step, hit)
    if not hit:
        tmp = path.with_suffix(".tmp")
        mm = np.memmap(tmp, dtype=np.float32, mode="w+", shape=(n, n))
        t0 = time.time()
        build(mm)
        mm.flush()
        del mm
        tmp.replace(path)
        _log(step, key, path, params, version, None, time.time() - t0)
    return np.memmap(path, dtype=np.float32, mode="r", shape=(n, n))

def query(step: str, sql: str, params: dict | None = None, db: Path | None = None, version: str = "1",
          allow_empty: bool = False) -> pl.LazyFrame:
    def run() -> pl.DataFrame:
        con = duck(db or DUCKDB_MAIN)
        try:
            return con.execute(sql).pl()
        finally:
            con.close()
    return cached(step, run, params, version=version, sql=sql, allow_empty=allow_empty)

def cached_sql_parts(step: str, parts: dict[str, str], params: dict | None = None,
                     version: str = "1", refresh: bool = False, allow_empty: bool = False) -> pl.LazyFrame:
    # Each named DuckDB query is written to its own Parquet file; finished parts are kept, so an
    # interrupted build resumes. Memory is bounded by the largest single part.
    sql_all = "\n;\n".join(f"-- {k}\n{v}" for k, v in sorted(parts.items()))
    out = cache_path(step, sql_all, params, version)
    done = out / "_DONE"
    hit = done.exists() and not _must_refresh(step, refresh)
    _report(step, hit)
    if not hit:
        import duckdb
        out.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        for name, sql in parts.items():
            part = out / f"part-{name}.parquet"
            if part.exists() and not _must_refresh(step, refresh):
                continue
            t1, tmp = time.time(), part.with_suffix(".tmp")
            con = duckdb.connect(config={"memory_limit": "1500MB", "threads": 2, "temp_directory": str(TMP),
                                         "preserve_insertion_order": False})
            try:
                con.execute(f"COPY ({sql}) TO '{tmp}' (FORMAT PARQUET, COMPRESSION ZSTD)")
            finally:
                con.close()
                gc.collect()
            tmp.replace(part)
            print(f"  {step} [{name}] {time.time() - t1:,.1f} s", flush=True)
        if not allow_empty and pl.scan_parquet(out / "part-*.parquet").select(pl.len()).collect().item() == 0:
            raise ValueError(f"{step}: empty result, not cached (pass allow_empty=True if expected)")
        done.write_text(cache_key(step, params, version, sql_all))
        _log(step, cache_key(step, params, version, sql_all), out, params, version, sql_all, time.time() - t0)
    return pl.scan_parquet(out / "part-*.parquet")

def cache_path(step: str, sql: str, params: dict | None = None, version: str = "1") -> Path:
    return CACHE_DIR / cache_key(step, params, version, sql)

def parts_glob(lf_parts_step: str, parts: dict[str, str], params: dict | None = None, version: str = "1") -> str:
    sql_all = "\n;\n".join(f"-- {k}\n{v}" for k, v in sorted(parts.items()))
    return str(cache_path(lf_parts_step, sql_all, params, version) / "part-*.parquet")

def cache_status(show: bool = True) -> pl.DataFrame:
    # Recomputes each recorded step's key from the manifest under the current CACHE_VERSION and source; computes nothing
    latest = {}
    if MANIFEST.exists():
        for line in MANIFEST.read_text().splitlines():
            e = json.loads(line)
            if "version" in e:
                latest[e["step"]] = e
    rows = []
    for step, e in sorted(latest.items()):
        key = key_from_parts(step, e["params"], e["version"], e["sql_sha"])
        path = Path(e["path"])
        marker = path / f"_KEY.{CACHE_VERSION}"
        if path.is_dir() and (path / "_DONE").exists():
            marker = path / "_DONE"
        exists = marker.exists() and marker.read_text() == key if path.is_dir() else path.exists()
        rows.append({"step": step, "status": "HIT" if key == e["key"] and exists else "MISS",
                     "key": key, "seconds_to_build": e["seconds"]})
    out = pl.DataFrame(rows, schema={"step": pl.Utf8, "status": pl.Utf8, "key": pl.Utf8, "seconds_to_build": pl.Float64})
    if show:
        print(out.group_by("status").len())
    return out

def scan_icms() -> pl.LazyFrame:
    return pl.scan_parquet(ICMS_DIR / "**/*.parquet", hive_partitioning=True)

def scan_gare() -> pl.LazyFrame:
    return pl.scan_parquet(GARE_DIR / "**/*.parquet", hive_partitioning=True)
cache_status()

#### 0.4 Random seed and extraction
A fixed seed governs every random draw. The data correspond to a single frozen extraction (March 2026).

In [ ]:
NOTEBOOK_BUILD = "e9907bb2870c"
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
EXTRACT = os.environ.get("CEMEPI_EXTRACT", "2026-03")
SNAP_YEAR, SNAP_MONTH = (int(x) for x in EXTRACT.split("-"))
ENROL_MIN = date(1960, 1, 1)
ENROL_MAX = (date(SNAP_YEAR + SNAP_MONTH // 12, SNAP_MONTH % 12 + 1, 1) - timedelta(days=1))

def enrol_date_sql(col: str = "DATA_INSCRICAO") -> str:
    # Enrolment dates outside [1960-01-01, last snapshot] are data-entry errors and are set to NULL
    d = f"TRY_CAST(LEFT(CAST({col} AS VARCHAR), 10) AS DATE)"
    return f"CASE WHEN {d} BETWEEN DATE '{ENROL_MIN}' AND DATE '{ENROL_MAX}' THEN {d} END"

def plausible_enrol(lf: pl.LazyFrame) -> pl.LazyFrame:
    ok = pl.col("DATA_INSCRICAO").is_between(pl.lit(ENROL_MIN), pl.lit(ENROL_MAX))
    cols = [c for c in ("age", "enrol_year", "age_years") if c in lf.collect_schema().names()]
    return lf.with_columns(pl.when(ok).then(pl.col("DATA_INSCRICAO")).alias("DATA_INSCRICAO"),
                           *[pl.when(ok).then(pl.col(c)).alias(c) for c in cols])
YEARS = list(range(2016, SNAP_YEAR + 1))
ENV_INFO = {"python": sys.version.split()[0], "polars": pl.__version__, "extract": EXTRACT, "notebook_build": NOTEBOOK_BUILD}
ENV_INFO

#### 0.5 Figure style
Figures are produced with Plotly using a custom template. The palette follows the CEMEPI visual identity — deep teal `#154C56`, sand `#BD9869`, teal `#186D7F`, green `#5EA65B` and mid teal `#267586`, in an order that maximises contrast, combined with distinct markers and dash patterns for readers with colour-vision deficiency; cream `#FFF4E5` is reserved for shaded windows. Sequential scales run from white to deep teal; correlation maps use a diverging sand–white–teal scale fixed on [−1, 1]. Typography is Montserrat. Figures carry no titles, because captions are given in the manuscript, and no logos. Gridlines are light and horizontal. Every figure with more than one series has a legend, placed above the plotting area; single-series figures are identified by their axis titles, which state the variable and its unit. Colour bars of heat maps are titled with the quantity they encode. Monetary values are expressed in R$ billion, and recovery rates are shown on logarithmic axes. Categories of CDA age are always ordered from the youngest to the oldest band. Each figure is displayed interactively and exported as PDF and SVG (vector) and as PNG at six times the nominal resolution, at single-column (3.5 in) or double-column (7.0 in) width.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

CEMEPI = {"deep_teal": "#154C56", "teal": "#186D7F", "mid_teal": "#267586", "sand": "#BD9869",
          "green": "#5EA65B", "cream": "#FFF4E5", "grey": "#8A9499"}
CAT_ORDER = [CEMEPI[k] for k in ("deep_teal", "sand", "teal", "green", "mid_teal")]
DASHES = ["solid", "dash", "dot", "dashdot", "longdash"]
SEQ = [[0.0, "#FFFFFF"], [1.0, CEMEPI["deep_teal"]]]
DIV = [[0.0, CEMEPI["sand"]], [0.5, "#FFFFFF"], [1.0, CEMEPI["deep_teal"]]]
PX_PER_IN, SINGLE_IN, DOUBLE_IN, RATIO, EXPORT_SCALE = 96, 3.5, 7.0, 0.62, 6

pio.templates["cemepi"] = go.layout.Template(layout=go.Layout(
    font=dict(family="Montserrat, Arial, sans-serif", size=11, color=CEMEPI["deep_teal"]),
    colorway=CAT_ORDER, paper_bgcolor="white", plot_bgcolor="white", showlegend=False,
    margin=dict(l=10, r=30, t=15, b=10, autoexpand=True),
    xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor=CEMEPI["deep_teal"], ticks="outside",
               automargin=True),
    yaxis=dict(showgrid=True, gridcolor="#E6E6E6", gridwidth=0.5, zeroline=False,
               showline=True, linecolor=CEMEPI["deep_teal"], ticks="outside", automargin=True),
    colorscale=dict(sequential=SEQ, diverging=DIV),
    annotationdefaults=dict(font=dict(size=10)),
    legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0)))
pio.templates.default = "cemepi"

def bn(x):
    return x / 1e9

def color(i: int) -> str:
    return CAT_ORDER[i % len(CAT_ORDER)]

FILING_LABELS = {"Ajuizado": "Filed", "Liberado para ajuizamento": "Cleared for filing",
                 "Aguardando liberação para ajuizar": "Awaiting clearance"}

def filing_label(col: str = "STATUS_AJUIZAMENTO_DEBITO") -> pl.Expr:
    return pl.col(col).replace_strict(FILING_LABELS, default="Unrecognised code", return_dtype=pl.Utf8).alias(col)

def wrap(text: str, width: int) -> str:
    if not text or len(text) <= width or "<br>" in text:
        return text
    words, out, line = text.split(" "), [], ""
    for w in words:
        if line and len(line) + 1 + len(w) > width:
            out.append(line)
            line = w
        else:
            line = f"{line} {w}".strip()
    return "<br>".join(out + [line])

def _axis_values(fig: go.Figure, name: str) -> list:
    kind = name[0]
    ref = kind + name[5:]
    out = []
    for t in fig.data:
        if (getattr(t, kind + "axis", None) or kind) == ref and getattr(t, kind, None) is not None:
            out.extend(t[kind])
    return out

def _is_year(values: list) -> bool:
    nums = [v for v in values if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)]
    return bool(nums) and len(nums) == len(values) and all(1950 <= float(v) <= 2100 and float(v) == int(v) for v in nums)

def polish(fig: go.Figure) -> go.Figure:
    lay = fig.layout
    labels = globals().get("AGE_LABELS")
    for name in [k for k in lay.to_plotly_json() if k.startswith(("xaxis", "yaxis"))] or ["xaxis", "yaxis"]:
        ax = lay[name]
        values = _axis_values(fig, name)
        title = ax.title.text or ""
        if name.startswith("xaxis") and not title and ax.showticklabels is not False and _is_year(values):
            title = "Year"
        if title:
            ax.title.text = wrap(title, 34 if name.startswith("yaxis") else 60)
        if name.startswith("yaxis") and ax.type == "category":
            continue
        present = {str(v) for v in values if v is not None}
        if labels and present and present <= set(labels):
            ax.categoryorder, ax.categoryarray = "array", list(labels)
    if lay.showlegend and lay.legend.orientation == "h" and lay.legend.y is not None and lay.legend.y < 0:
        lay.legend.update(y=1.02, yanchor="bottom", x=0, xanchor="left")
    lay.legend.itemsizing = "constant"
    return fig

def save_fig(fig: go.Figure, label: str, width: str = "single", ratio: float = RATIO,
             height: int | None = None) -> go.Figure:
    polish(fig)
    if width == "single" and fig.layout.showlegend and len(fig.data) > 2:
        width, ratio = "double", min(ratio, 0.5)
    w_in = SINGLE_IN if width == "single" else DOUBLE_IN
    meta = fig.layout.meta if isinstance(fig.layout.meta, dict) else {}
    w = int(w_in * PX_PER_IN)
    h = height or max(int(w_in * ratio * PX_PER_IN), int(meta.get("min_height", 0)))
    stem = FIG_DIR / label
    fig.write_image(stem.with_suffix(".pdf"), width=w, height=h)
    fig.write_image(stem.with_suffix(".svg"), width=w, height=h)
    fig.write_image(stem.with_suffix(".png"), width=w, height=h, scale=EXPORT_SCALE)
    return fig

def save_tab(df: pl.DataFrame, label: str, fmt: str = "{:,.2f}") -> pl.DataFrame:
    def cell(v):
        if isinstance(v, float):
            return "" if v != v else fmt.format(v)
        return str(v).replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")
    header = " & ".join(c.replace("_", r"\_") for c in df.columns) + r" \\"
    rows = [" & ".join(cell(v) for v in r) + r" \\" for r in df.iter_rows()]
    body = [r"\begin{tabular}{" + "l" * df.width + "}", r"\toprule", header, r"\midrule",
            *rows, r"\bottomrule", r"\end{tabular}"]
    (TAB_DIR / f"{label}.tex").write_text("\n".join(body))
    return df

def lines(df: pl.DataFrame, x: str, y: str, group: str | None = None, log_y: bool = False,
          y_title: str = "", x_title: str = "", markers: bool = True, end_labels: bool = False,
          order: list | None = None, names: dict | None = None) -> go.Figure:
    fig = go.Figure()
    groups = [(None, df)] if group is None else [(k[0], d) for k, d in df.sort(group).group_by(group, maintain_order=True)]
    if order is not None:
        rank = {v: i for i, v in enumerate(order)}
        groups = sorted(groups, key=lambda gd: rank.get(gd[0], len(rank)))
    if names is not None:
        groups = [(names.get(g, g), d) for g, d in groups]
    ends = []
    for i, (g, d) in enumerate(groups):
        d = d.sort(x)
        fig.add_trace(go.Scatter(x=d[x], y=d[y], mode="lines+markers" if markers else "lines", name=str(g),
                                 line=dict(color=color(i), dash=DASHES[i % 5], width=1.6), marker=dict(size=4)))
        if g is not None and d.height:
            ends.append((i, g, d[x][-1], float(d[y][-1])))
    multi = group is not None and len(groups) > 1
    if len(y_title) > 28 and "<br>" not in y_title:
        cut = y_title.rfind(" ", 0, len(y_title) // 2 + 6)
        y_title = y_title[:cut] + "<br>" + y_title[cut + 1:] if cut > 0 else y_title
    fig.update_layout(xaxis_title=x_title, yaxis_title=y_title, showlegend=multi,
                      legend=dict(orientation="h", yanchor="top", y=-0.22, x=0.5, xanchor="center", title_text=""))
    if log_y:
        fig.update_yaxes(type="log", dtick=1)
    if end_labels and multi and ends:
        pos = sorted((np.log10(v) if log_y and v > 0 else v) for *_, v in ends)
        span = (max(pos) - min(pos)) or 1.0
        if all(b - a > 0.06 * span for a, b in zip(pos, pos[1:])):
            for i, g, xe, ye in ends:
                fig.add_annotation(x=xe, y=ye, text=str(g), showarrow=False, xanchor="left", xshift=5,
                                   font=dict(color=color(i), size=10))
            longest = max(len(str(g)) for _, g, *_ in ends)
            fig.update_layout(margin=dict(r=max(30, 12 + 6 * longest)))
    return fig

def bars(df: pl.DataFrame, cat: str, val: str, horizontal: bool = True, log: bool = False,
         val_title: str = "", text_fmt: str = "{:,.1f}") -> go.Figure:
    d = df.sort(val)
    text = [text_fmt.format(v) for v in d[val]]
    common = dict(text=text, textposition="outside", cliponaxis=False, marker_color=CEMEPI["deep_teal"])
    tr = (go.Bar(x=d[val], y=d[cat], orientation="h", **common) if horizontal else go.Bar(x=d[cat], y=d[val], **common))
    fig = go.Figure(tr)
    axis = "xaxis" if horizontal else "yaxis"
    vals = d[val].cast(pl.Float64).to_numpy()
    vmax = float(np.nanmax(vals)) if vals.size else 1.0
    if log:
        pos = vals[vals > 0]
        lo = np.log10(pos.min()) if pos.size else 0.0
        hi = np.log10(vmax) if vmax > 0 else 1.0
        rng = [np.floor(lo) if lo > 0 else lo - 0.2, hi + 0.5 * max(hi - lo, 1.0) + 0.3]
        fig.update_layout(**{axis: dict(type="log", range=rng, dtick=1)})
    else:
        fig.update_layout(**{axis: dict(range=[0, vmax * 1.3])})
    fig.update_layout(**{f"{axis}_title": val_title})
    if horizontal:
        fig.update_layout(xaxis=dict(showgrid=True, gridcolor="#E6E6E6"), yaxis=dict(showgrid=False),
                          meta={"min_height": 60 + 24 * d.height})
    return fig

def heatmap(mat: pl.DataFrame, row: str, diverging: bool = True, zmin=None, zmax=None,
            fmt: str = "{:.2f}", colorbar_title: str = "Correlation", text: list | None = None) -> go.Figure:
    cols = [c for c in mat.columns if c != row]
    z = mat.select(cols).to_numpy().astype(float)
    if text is None:
        text = [[("" if np.isnan(v) else fmt.format(v)) for v in r] for r in z]
    fig = go.Figure(go.Heatmap(z=z, x=cols, y=[str(v) for v in mat[row].to_list()], text=text, texttemplate="%{text}",
                               colorscale=DIV if diverging else SEQ,
                               zmin=-1 if diverging and zmin is None else zmin,
                               zmax=1 if diverging and zmax is None else zmax,
                               zmid=0 if diverging else None, colorbar=dict(title=colorbar_title)))
    fig.update_layout(yaxis=dict(showgrid=False, autorange="reversed", type="category"),
                      xaxis=dict(showgrid=False, tickangle=-35, type="category"),
                      meta={"min_height": 80 + 26 * len(z)})
    return fig

def ecdf(df: pl.DataFrame, value: str, group: str, log_x: bool = True, x_title: str = "",
         max_points: int = 2000) -> go.Figure:
    fig = go.Figure()
    for i, (k, d) in enumerate(df.sort(group).group_by(group, maintain_order=True)):
        v = np.sort(d[value].drop_nulls().to_numpy())
        v = v[v > 0] if log_x else v
        if v.size == 0:
            continue
        q = np.linspace(0, 1, min(max_points, v.size))
        xs = np.quantile(v, q)
        fig.add_trace(go.Scatter(x=xs, y=q, mode="lines", name=str(k[0]),
                                 line=dict(color=color(i), dash=DASHES[i % 5])))
    fig.update_layout(xaxis_title=x_title, yaxis_title="Cumulative share", showlegend=True,
                      legend=dict(x=0.02, y=0.98))
    if log_x:
        fig.update_xaxes(type="log")
    return fig

def shade_window(fig: go.Figure, x0, x1, text: str | None = None) -> go.Figure:
    fig.add_vrect(x0=x0, x1=x1, fillcolor=CEMEPI["cream"], opacity=1.0, layer="below", line_width=0,
                  annotation_text=text, annotation_position="bottom left", annotation_font_size=9)
    return fig

PROGRAMMES = [("PEP 2019", "2019-07-01", "2019-12-31"), ("Acordo Paulista 2024", "2024-04-01", "2024-12-31")]

def shade_programmes(fig: go.Figure) -> go.Figure:
    for name, a, b in PROGRAMMES:
        shade_window(fig, a, b, name)
    return fig

#### 0.6 Statistical utilities
Non-parametric tests, effect sizes, contingency-table statistics, correlation with a selectable method and concentration measures used throughout the analysis are defined once here.

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

def kruskal_test(df: pl.DataFrame, value: str, group: str) -> dict:
    samples = [d[value].drop_nulls().to_numpy() for _, d in df.group_by(group)]
    samples = [s for s in samples if s.size > 0]
    h, p = stats.kruskal(*samples)
    n, k = sum(s.size for s in samples), len(samples)
    return {"variable": value, "H": float(h), "df": k - 1, "p_value": float(p),
            "epsilon_sq": float((h - k + 1) / (n - k)) if n > k else float("nan")}

def mann_whitney(a: np.ndarray, b: np.ndarray) -> dict:
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"U": float(u), "p_value": float(p), "rank_biserial": float(1 - 2 * u / (a.size * b.size)),
            "n_a": int(a.size), "n_b": int(b.size)}

def pairwise_tests(df: pl.DataFrame, value: str, group: str, max_n: int = 200_000) -> pl.DataFrame:
    rng = np.random.default_rng(SEED)
    data = {}
    for k, d in df.group_by(group):
        v = d[value].drop_nulls().to_numpy().astype(float)
        data[k[0]] = rng.choice(v, size=max_n, replace=False) if v.size > max_n else v
    keys, rows = sorted(data), []
    for i, a in enumerate(keys):
        for b in keys[i + 1:]:
            if data[a].size == 0 or data[b].size == 0:
                continue
            mw = mann_whitney(data[a], data[b])
            ks = stats.ks_2samp(data[a], data[b])
            ad = stats.anderson_ksamp([data[a], data[b]], method=stats.PermutationMethod(n_resamples=199, random_state=SEED)) \
                if min(data[a].size, data[b].size) < 1000 else stats.anderson_ksamp([data[a], data[b]])
            rows.append({"variable": value, "group_a": a, "group_b": b, **mw,
                         "ks_stat": float(ks.statistic), "ks_p": float(ks.pvalue),
                         "ad_stat": float(ad.statistic), "ad_p": float(ad.pvalue)})
    out = pl.DataFrame(rows)
    if out.height:
        for col in ("p_value", "ks_p", "ad_p"):
            out = out.with_columns(pl.Series(f"{col}_bh", multipletests(out[col].to_numpy(), method="fdr_bh")[1]))
    return out

def dunn_test(df: pl.DataFrame, value: str, group: str) -> pl.DataFrame:
    d = df.select(value, group).drop_nulls()
    ranks = stats.rankdata(d[value].to_numpy())
    d = d.with_columns(pl.Series("_r", ranks))
    n = d.height
    _, counts = np.unique(ranks, return_counts=True)
    tie = 1 - (counts ** 3 - counts).sum() / (n ** 3 - n)
    g = d.group_by(group).agg(pl.col("_r").mean().alias("mr"), pl.len().alias("n")).sort(group)
    rows = []
    for i in range(g.height):
        for j in range(i + 1, g.height):
            se = np.sqrt(tie * n * (n + 1) / 12 * (1 / g["n"][i] + 1 / g["n"][j]))
            z = (g["mr"][i] - g["mr"][j]) / se
            rows.append({"variable": value, "group_a": g[group][i], "group_b": g[group][j],
                         "z": float(z), "p_value": float(2 * stats.norm.sf(abs(z)))})
    out = pl.DataFrame(rows)
    return out.with_columns(pl.Series("p_bh", multipletests(out["p_value"].to_numpy(), method="fdr_bh")[1]))

def chi2_table(ct: np.ndarray) -> dict:
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    res = {"chi2": float(chi2), "df": int(dof), "p_value": float(p), "cramers_v": float(v)}
    if ct.shape == (2, 2) and (expected < 5).any():
        res["fisher_p"] = float(stats.fisher_exact(ct)[1])
    return res

def corr_pair(x: np.ndarray, y: np.ndarray, method: str = "spearman") -> tuple[float, float]:
    m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 4:
        return float("nan"), float("nan")
    r = stats.spearmanr(x[m], y[m]) if method == "spearman" else stats.pearsonr(x[m], y[m])
    return float(r[0]), float(r[1])

def lorenz(values: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    v = np.sort(np.asarray(values, dtype=float))[::-1]
    cum = np.cumsum(v) / v.sum()
    share = np.arange(1, v.size + 1) / v.size
    gini = 1 - 2 * np.trapezoid(np.cumsum(v[::-1]) / v.sum(), dx=1 / v.size)
    return share, cum, float(gini)

def top_shares(values: np.ndarray, fractions=(0.001, 0.01, 0.1)) -> dict:
    v = np.sort(np.asarray(values, dtype=float))[::-1]
    return {f"top_{f * 100:g}pct": float(v[: max(1, int(round(f * v.size)))].sum() / v.sum()) for f in fractions}

## 1. Data

*Terms introduced in this section:* CDA, active debt, snapshot, raw debt type, origin, GARE, CNPJ root, CPF, stock, collection, new enrolments, enrolment date, age, CDA × year panel, recovery rate. Definitions are given in the Glossary.

#### 1.1 Sources

**Method.** Inventory of the data sources and of the role each one plays in the analysis.

**Data.** All tables of the frozen March 2026 extraction of PGE-SP active debt, plus the SGS (Central Bank of Brazil) and IBGE macroeconomic series. `DEBITO` holds monthly snapshots of every CDA (January 2016 to March 2026); `ARRECADACAO` holds GARE payments; `DEBITO_RECEITA`, `SOLICITACAOPARCELAMENTO`, `STATUS_PROTESTO`, `AJUIZAMENTO` and `FATURAMENTO_MASCARADO` hold debt components, instalment requests, protest, lawsuit filing and masked firm revenue.

**How to read.** One row per source, with its unit of observation, period and use.

**Interpretation.** The table delimits what can and cannot be measured: for example, settlement is not recorded directly and is approximated in B.8.5.

*Output:* Table `tab_sources`.

In [ ]:
def table_inventory() -> pl.DataFrame:
    con = duck()
    try:
        tables = con.execute("SELECT table_name FROM duckdb_tables() WHERE NOT starts_with(table_name, '_') ORDER BY 1").fetchall()
        rows = []
        for (t,) in tables:
            n = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
            ncol = con.execute(f"SELECT COUNT(*) FROM information_schema.columns WHERE table_name = '{t}'").fetchone()[0]
            rows.append({"table": t, "rows": n, "columns": ncol})
    finally:
        con.close()
    return pl.DataFrame(rows)

inventory = collect(cached("1.1", table_inventory))
DESCRIPTION = {
    "DEBITO": "Monthly snapshots of enrolled debt certificates (CDA)",
    "ARRECADACAO": "Payments through state payment slips (GARE)",
    "DEBITO_RECEITA": "Debt components (principal, interest, fines)",
    "SOLICITACAOPARCELAMENTO": "Instalment requests",
    "STATUS_PROTESTO": "Protest status of CDAs",
    "AJUIZAMENTO": "Lawsuit filing events",
    "FATURAMENTO_MASCARADO": "Masked firm revenue",
}
sources = inventory.with_columns(pl.col("table").replace_strict(DESCRIPTION, default="").alias("content"))
save_tab(sources.select("table", "content", "rows", "columns"), "tab_sources", fmt="{:,.0f}")
sources

#### 1.1.1 Snapshot logic

`DEBITO` is a stock table: every outstanding CDA is repeated in each monthly snapshot until it leaves the portfolio. Summing values across snapshots would therefore count the same debt many times. The stock at a given date is taken from the corresponding snapshot after deduplication by CDA identifier, and the annual stock is the stock in the last snapshot of each year. Flows (payments, new enrolments) are measured from event dates rather than from differences between snapshots.

#### 1.2 Cross-sectional snapshot of all tax types

**Method.** Extraction of a single cross-sectional snapshot, sufficient for comparing tax types at one date.

**Data.** `DEBITO`, March 2026 snapshot, all debt types; fields kept: CDA identifier, raw debt type, value net of fees, enrolment date and debtor CNPJ root.

**How to read.** This step produces the dataset used in Part A; its output is a summary of the extracted rows.

**Interpretation.** All Part A comparisons refer to this date and therefore describe the portfolio as it stood in March 2026.

In [ ]:
BUCKET_SQL = """CASE WHEN TIPO_DEBITO ILIKE '%ICMS%'  THEN 'ICMS'
                     WHEN TIPO_DEBITO ILIKE '%IPVA%'  THEN 'IPVA'
                     WHEN TIPO_DEBITO ILIKE '%ITCMD%' THEN 'ITCMD'
                     ELSE 'Fees and other' END"""
IDENT_SQL = """CASE WHEN COALESCE(CPF_DEVEDOR, '') <> '' THEN 'CPF'
                    WHEN COALESCE(CNPJ_DEVEDOR, '') <> '' THEN 'CNPJ' ELSE 'None' END"""

SNAPSHOT_SQL = f"""
SELECT ID_DEBITO, TIPO_DEBITO, {BUCKET_SQL} AS bucket, {IDENT_SQL} AS identifier,
       TRY_CAST(LEFT(DATA_INSCRICAO, 10) AS DATE) AS DATA_INSCRICAO,
       SITUACAO_DEBITO, STATUS_AJUIZAMENTO_DEBITO, TRY_CAST(VALOR_SEM_HONORARIOS AS DOUBLE) AS value,
       DATE_DIFF('day', TRY_CAST(LEFT(DATA_INSCRICAO, 10) AS DATE), DATE '{SNAP_YEAR}-{SNAP_MONTH:02d}-01') / 365.25 AS age_years
FROM DEBITO
WHERE ANO_EXTRACAO = {SNAP_YEAR} AND MES_EXTRACAO = {SNAP_MONTH}
"""
snapshot_all = plausible_enrol(query("1.2", SNAPSHOT_SQL, params={"snapshot": EXTRACT}))
collect(snapshot_all.group_by("bucket").agg(pl.len().alias("n_cda"), pl.col("value").sum().alias("stock")).sort("bucket"))

#### 1.3 ICMS extract

**Method.** Year-by-year extraction of the ICMS records into a Parquet dataset partitioned by year, which bounds memory use by the size of a single year.

**Data.** `DEBITO` and `ARRECADACAO`, 2016–2026, restricted to the five raw ICMS types and to debtors identified by a CNPJ. An origin variable (declared, assessed, other) is derived from the raw type. Payments are linked to CDAs through the debt identifier.

**How to read.** The table gives the number of rows extracted per year and source.

**Interpretation.** Abrupt changes in yearly counts would point to incomplete partitions rather than to changes in the portfolio.

*Output:* Table `tab_extraction_counts`.

In [ ]:
IDENT_SQL = """CASE WHEN COALESCE(CPF_DEVEDOR, '') <> '' THEN 'CPF'
                    WHEN COALESCE(CNPJ_DEVEDOR, '') <> '' THEN 'CNPJ' ELSE 'None' END"""
ICMS_TYPES = ["ICMS Declarado", "ICMS Autuação", "ICMS Autuação Simples Nacional",
              "ICMS Fecoep", "ICMS Alteração de GIA"]
ICMS_TYPES_SQL = ", ".join(f"'{t}'" for t in ICMS_TYPES)
PJ_FILTER = "COALESCE(CPF_DEVEDOR, '') = '' AND COALESCE(CNPJ_DEVEDOR, '') <> ''"

def extract_icms_year(year: int, part: Path) -> None:
    con = duck()
    try:
        con.execute(f"""
        COPY (
            SELECT ANO_EXTRACAO, MES_EXTRACAO, ID_DEBITO, CDA_COMPLETA,
                   TRY_CAST(LEFT(DATA_INSCRICAO, 10) AS DATE) AS DATA_INSCRICAO,
                   SITUACAO_DEBITO, TIPO_DEBITO,
                   CASE WHEN TIPO_DEBITO = 'ICMS Declarado' THEN 'declared'
                        WHEN TIPO_DEBITO LIKE 'ICMS Autua%' THEN 'assessed'
                        ELSE 'other' END AS ICMS_ORIGIN,
                   STATUS_AJUIZAMENTO_DEBITO,
                   TRY_CAST(VALOR_SEM_HONORARIOS AS DOUBLE) AS VALOR_SEM_HONORARIOS,
                   TRY_CAST(VALOR_VH AS DOUBLE) AS VALOR_VH,
                   REGEXP_REPLACE(CNPJ_DEVEDOR, '[^0-9]', '', 'g') AS CNPJ,
                   LEFT(REGEXP_REPLACE(CNPJ_DEVEDOR, '[^0-9]', '', 'g'), 8) AS CNPJ_ROOT
            FROM DEBITO
            WHERE TIPO_DEBITO IN ({ICMS_TYPES_SQL})
              AND ANO_EXTRACAO = {year} AND MES_EXTRACAO BETWEEN 1 AND 12
              AND {PJ_FILTER}
        ) TO '{part}/part-0.parquet' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)""")
    finally:
        con.close()

def extract_gare_year(year: int, part: Path) -> None:
    con = duck()
    try:
        con.execute(f"""
        COPY (
            SELECT a.ID_DEBITO, a.ID_GARE_SEFAZ, CAST(a.DATA_ARRECADACAO_GARE AS DATE) AS DATA_ARRECADACAO_GARE,
                   a.VALOR_RECEITA, a.VALOR_TOTAL_GARE, a.JUROS_MORA, a.MULTA_MORA, a.STATUS_AJUIZAMENTO_DEBITO
            FROM ARRECADACAO a
            SEMI JOIN (SELECT DISTINCT ID_DEBITO FROM read_parquet('{ICMS_DIR}/**/*.parquet')) i
                 ON a.ID_DEBITO = i.ID_DEBITO
            WHERE YEAR(a.DATA_ARRECADACAO_GARE) = {year}
        ) TO '{part}/part-0.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    finally:
        con.close()

EXTRACT_PARAMS = {"types": ICMS_TYPES, "snapshot": EXTRACT, "pj_only": True}
icms = cached_partitioned("1.3", ICMS_DIR, YEARS, extract_icms_year, params=EXTRACT_PARAMS)
gare = cached_partitioned("1.3-gare", GARE_DIR, YEARS, extract_gare_year, params=EXTRACT_PARAMS)

SOURCE_COUNTS_SQL = f"""
SELECT ANO_EXTRACAO AS year, {IDENT_SQL} AS identifier, COUNT(*) AS rows
FROM DEBITO
WHERE TIPO_DEBITO IN ({ICMS_TYPES_SQL}) AND ANO_EXTRACAO BETWEEN {YEARS[0]} AND {YEARS[-1]}
  AND MES_EXTRACAO BETWEEN 1 AND 12
GROUP BY ALL
"""
source_counts = collect(query("1.3:source", SOURCE_COUNTS_SQL, params=EXTRACT_PARAMS)).pivot(
    on="identifier", index="year", values="rows").fill_null(0)
extracted = collect(cached("1.3:extracted_counts", lambda: icms.group_by("year").agg(
    pl.len().alias("rows_extracted"),
    pl.col("ID_DEBITO").n_unique().alias("distinct_cda"),
    pl.col("MES_EXTRACAO").n_unique().alias("months"),
    (pl.col("VALOR_SEM_HONORARIOS").sum() / 1e9).alias("value_sum_bn")), params=EXTRACT_PARAMS))
gare_counts = collect(cached("1.3:gare_counts", lambda: gare.group_by("year").agg(
    pl.len().alias("gare_rows"), (pl.col("VALOR_RECEITA").sum() / 1e9).alias("gare_bn")), params=EXTRACT_PARAMS))
extraction_counts = (source_counts.join(extracted, on="year").join(gare_counts, on="year", how="left")
                     .with_columns((pl.col("rows_extracted") == pl.col("CNPJ")).alias("matches_source"))
                     .sort("year"))
save_tab(extraction_counts, "tab_extraction_counts", fmt="{:,.1f}")
extraction_counts

#### 1.3.1 Debtor identification and exclusion of natural persons

**Method.** Classification of each CDA by the identifier of its debtor (CNPJ, CPF or none) and exclusion of natural persons.

**Data.** ICMS extract (1.3), March 2026 snapshot, all debtors; unit of observation: the CDA.

**How to read.** Rows give CDA counts and stock by identifier class, in absolute terms and as shares of the total, for ICMS as a whole and for each raw type.

**Interpretation.** Small shares for CPF and missing identifiers mean that restricting the study to legal entities changes the results little; the exclusion is also required for the protection of personal data.

*Output:* Table `tab_debtor_identifier`.

In [ ]:
DEBTOR_ID_SQL = f"""
WITH snap AS (
    SELECT TIPO_DEBITO AS raw_type,
           CASE WHEN COALESCE(CPF_DEVEDOR, '') <> '' THEN 'CPF (natural person)'
                WHEN COALESCE(CNPJ_DEVEDOR, '') <> '' THEN 'CNPJ (legal entity)'
                ELSE 'No identifier' END AS identifier,
           TRY_CAST(VALOR_SEM_HONORARIOS AS DOUBLE) AS value
    FROM DEBITO
    WHERE ANO_EXTRACAO = {SNAP_YEAR} AND MES_EXTRACAO = {SNAP_MONTH} AND TIPO_DEBITO IN ({ICMS_TYPES_SQL})
), stacked AS (
    SELECT * FROM snap
    UNION ALL
    SELECT 'ICMS (all types)', identifier, value FROM snap
)
SELECT raw_type, identifier,
       COUNT(*) AS n_cda,
       SUM(value) / 1e9 AS stock_bn,
       100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY raw_type) AS pct_cda,
       100 * SUM(value) / SUM(SUM(value)) OVER (PARTITION BY raw_type) AS pct_stock
FROM stacked
GROUP BY raw_type, identifier
ORDER BY raw_type, identifier
"""
debtor_id = collect(query("1.3.1", DEBTOR_ID_SQL, params={"snapshot": EXTRACT}))
save_tab(debtor_id, "tab_debtor_identifier", fmt="{:,.3f}")

total = debtor_id.filter(pl.col("raw_type") == "ICMS (all types)")
excluded = total.filter(pl.col("identifier") != "CNPJ (legal entity)")
kept_source = int(total.filter(pl.col("identifier") == "CNPJ (legal entity)")["n_cda"].sum())
kept_extract = collect(cached("1.3.1:extract_snapshot", lambda: icms.filter(
    (pl.col("year") == SNAP_YEAR) & (pl.col("MES_EXTRACAO") == SNAP_MONTH)).select(pl.len().alias("n")),
    params=EXTRACT_PARAMS))[0, 0]
print(f"Excluded: {excluded['n_cda'].sum():,} CDAs ({excluded['pct_cda'].sum():.3f}% of CDAs, "
      f"{excluded['pct_stock'].sum():.3f}% of the ICMS stock).")
print(f"Legal-entity CDAs in the source snapshot: {kept_source:,}; in the extract: {kept_extract:,}.")
debtor_id

#### 1.4 CDA × year panel

**Method.** Construction of an unbalanced panel with one observation per CDA and year. The panel is built in DuckDB one year at a time and then combined by buckets of CDA identifiers, so that no step holds the full table in memory.

**Data.** ICMS extract (1.3), 2016–2026. Variables: year-end stock, amount paid through GARE in the year, recovery rate (paid over stock), payment indicator, age, enrolment year and recovery rate lagged one to three years. Enrolment dates before 1960 or after the last snapshot are treated as data-entry errors and set to missing, together with the derived age.

**How to read.** This step produces the panel used in most of Part B and in D.4 and D.6; the cell output reports its size.

**Interpretation.** The panel is unbalanced because CDAs enter at enrolment and leave when settled or cancelled; averages over the panel therefore weight years by the number of CDAs outstanding.

In [ ]:
# Built in DuckDB from one yearly file at a time, then combined in ID buckets, so no step holds the full extract
def icms_year(y: int) -> str:
    return f"read_parquet('{ICMS_DIR}/year={y}/*.parquet')"

ATTR_YEAR_SQL = """
SELECT ID_DEBITO,
       MIN(DATA_INSCRICAO) AS DATA_INSCRICAO,
       MIN(snap) AS first_snapshot, MAX(snap) AS last_snapshot,
       MIN(snap) FILTER (WHERE STATUS_AJUIZAMENTO_DEBITO = 'Ajuizado') AS first_filed_snapshot,
       MAX(VALOR_SEM_HONORARIOS) AS max_value,
       arg_max(TIPO_DEBITO, snap) AS TIPO_DEBITO, arg_max(ICMS_ORIGIN, snap) AS ICMS_ORIGIN,
       arg_max(CNPJ, snap) AS CNPJ, arg_max(CNPJ_ROOT, snap) AS CNPJ_ROOT
FROM (SELECT *, ANO_EXTRACAO * 100 + MES_EXTRACAO AS snap FROM {src})
GROUP BY ID_DEBITO
"""
ATTR_YEAR = {str(y): ATTR_YEAR_SQL.format(src=icms_year(y)) for y in YEARS}
attr_year = cached_sql_parts("1.4:attributes_by_year", ATTR_YEAR, params=EXTRACT_PARAMS)

BUCKETS = 8
ATTR_SQL = """
SELECT ID_DEBITO, MIN(DATA_INSCRICAO) AS DATA_INSCRICAO,
       arg_max(TIPO_DEBITO, last_snapshot) AS TIPO_DEBITO, arg_max(ICMS_ORIGIN, last_snapshot) AS ICMS_ORIGIN,
       arg_max(CNPJ, last_snapshot) AS CNPJ, arg_max(CNPJ_ROOT, last_snapshot) AS CNPJ_ROOT,
       MIN(first_snapshot) AS first_snapshot, MAX(last_snapshot) AS last_snapshot,
       MIN(first_filed_snapshot) AS first_filed_snapshot, MAX(max_value) AS max_value
FROM read_parquet('{src}') WHERE ID_DEBITO % {n} = {k}
GROUP BY ID_DEBITO
"""
ATTR = {f"b{k}": ATTR_SQL.format(src=parts_glob("1.4:attributes_by_year", ATTR_YEAR, EXTRACT_PARAMS), n=BUCKETS, k=k)
        for k in range(BUCKETS)}
attributes = plausible_enrol(cached_sql_parts("1.4:attributes", ATTR, params=EXTRACT_PARAMS))

YEAR_END_SQL = """
SELECT CAST({y} AS INTEGER) AS year, ID_DEBITO,
       arg_max(VALOR_SEM_HONORARIOS, MES_EXTRACAO) AS stock,
       arg_max(SITUACAO_DEBITO, MES_EXTRACAO) AS SITUACAO_DEBITO,
       arg_max(STATUS_AJUIZAMENTO_DEBITO, MES_EXTRACAO) AS STATUS_AJUIZAMENTO_DEBITO,
       MAX(MES_EXTRACAO) = 12 AS present_in_december
FROM {src}
GROUP BY ID_DEBITO
"""
YEAR_END = {str(y): YEAR_END_SQL.format(y=y, src=icms_year(y)) for y in YEARS}
year_end = cached_sql_parts("1.4:year_end", YEAR_END, params=EXTRACT_PARAMS)

GARE_YEAR = {"all": f"""
SELECT CAST(year AS INTEGER) AS year, ID_DEBITO,
       SUM(VALOR_RECEITA) AS paid, COUNT(*) AS n_payments,
       MIN(DATA_ARRECADACAO_GARE) AS first_payment_in_year
FROM read_parquet('{GARE_DIR}/**/*.parquet', hive_partitioning = true)
GROUP BY ALL
"""}
gare_year = cached_sql_parts("1.4:gare_year", GARE_YEAR, params=EXTRACT_PARAMS)

PANEL_SQL = """
WITH base AS (
    SELECT y.year, y.ID_DEBITO, y.stock, y.SITUACAO_DEBITO, y.STATUS_AJUIZAMENTO_DEBITO, y.present_in_december,
           COALESCE(g.paid, 0.0) AS paid, CAST(COALESCE(g.n_payments, 0) AS INTEGER) AS n_payments,
           g.first_payment_in_year, a.DATA_INSCRICAO, a.TIPO_DEBITO, a.ICMS_ORIGIN, a.CNPJ_ROOT
    FROM (SELECT * FROM read_parquet('{ye}') WHERE ID_DEBITO % {n} = {k}) y
    LEFT JOIN (SELECT * FROM read_parquet('{gy}') WHERE ID_DEBITO % {n} = {k}) g USING (year, ID_DEBITO)
    LEFT JOIN (SELECT * FROM read_parquet('{at}') WHERE ID_DEBITO % {n} = {k}) a USING (ID_DEBITO)
), rated AS (
    SELECT *, paid / NULLIF(stock, 0) AS rate,
           CAST(paid > 0 AS TINYINT) AS paid_flag,
           DATE_DIFF('day', DATA_INSCRICAO, MAKE_DATE(year, 12, 31)) / 365.25 AS age,
           YEAR(DATA_INSCRICAO) AS enrol_year
    FROM base
)
SELECT *,
       CASE WHEN LAG(year, 1) OVER w = year - 1 THEN LAG(rate, 1) OVER w END AS rate_l1,
       CASE WHEN LAG(year, 2) OVER w = year - 2 THEN LAG(rate, 2) OVER w END AS rate_l2,
       CASE WHEN LAG(year, 3) OVER w = year - 3 THEN LAG(rate, 3) OVER w END AS rate_l3
FROM rated
WINDOW w AS (PARTITION BY ID_DEBITO ORDER BY year)
"""
PANEL = {f"b{k}": PANEL_SQL.format(ye=parts_glob("1.4:year_end", YEAR_END, EXTRACT_PARAMS),
                                   gy=parts_glob("1.4:gare_year", GARE_YEAR, EXTRACT_PARAMS),
                                   at=parts_glob("1.4:attributes", ATTR, EXTRACT_PARAMS), n=BUCKETS, k=k)
         for k in range(BUCKETS)}
panel = plausible_enrol(cached_sql_parts("1.4:panel", PANEL, params=EXTRACT_PARAMS))
panel.collect_schema()

#### 1.5 Monthly series

**Method.** Aggregation of the panel and of the payments into monthly time series.

**Data.** ICMS extract (1.3), January 2016 to March 2026, for the pooled portfolio and for each origin. Series: amount collected, number of paying CDAs, number of new CDAs and outstanding stock, with first differences.

**How to read.** The cell output lists the series and their length.

**Interpretation.** These series are the input of the seasonal analysis (B.10), the diagnostics of Part C and the macroeconomic associations of Part D.

In [ ]:
def monthly_series() -> pl.DataFrame:
    stock = (scan_icms().group_by("ANO_EXTRACAO", "MES_EXTRACAO", "ICMS_ORIGIN")
             .agg(pl.col("VALOR_SEM_HONORARIOS").sum().alias("stock"), pl.len().alias("n_cda")))
    stock = collect(stock).with_columns(pl.date(pl.col("ANO_EXTRACAO"), pl.col("MES_EXTRACAO"), 1).alias("month"))
    new = collect(attributes.with_columns(pl.col("DATA_INSCRICAO").dt.truncate("1mo").alias("month"))
                  .group_by("month", "ICMS_ORIGIN").agg(pl.len().alias("n_new"),
                                                         pl.col("max_value").sum().alias("value_new")))
    pay = collect(scan_gare().join(attributes.select("ID_DEBITO", "ICMS_ORIGIN"), on="ID_DEBITO", how="left")
                  .with_columns(pl.col("DATA_ARRECADACAO_GARE").dt.truncate("1mo").alias("month"))
                  .group_by("month", "ICMS_ORIGIN").agg(pl.col("VALOR_RECEITA").sum().alias("collection"),
                                                         pl.col("ID_DEBITO").n_unique().alias("n_payers")))
    by_origin = (stock.select("month", "ICMS_ORIGIN", "stock", "n_cda")
                 .join(new, on=["month", "ICMS_ORIGIN"], how="full", coalesce=True)
                 .join(pay, on=["month", "ICMS_ORIGIN"], how="full", coalesce=True)
                 .filter(pl.col("month").is_between(pl.date(YEARS[0], 1, 1), pl.date(SNAP_YEAR, SNAP_MONTH, 1))))
    total = by_origin.group_by("month").agg(pl.exclude("ICMS_ORIGIN").sum()).with_columns(pl.lit("all").alias("ICMS_ORIGIN"))
    out = pl.concat([by_origin, total.select(by_origin.columns)]).sort("ICMS_ORIGIN", "month")
    return out.with_columns([pl.col(c).diff().over("ICMS_ORIGIN").alias(f"d_{c}")
                             for c in ("stock", "n_new", "collection", "n_payers")])

monthly = collect(cached("1.5", monthly_series, params=EXTRACT_PARAMS))
monthly_all = monthly.filter(pl.col("ICMS_ORIGIN") == "all")
monthly_all.tail(6)

#### 1.6 Macroeconomic series

**Method.** Assembly of the macroeconomic covariates at monthly, quarterly and annual frequency.

**Data.** SGS and IBGE series: Selic, IPCA, IPCA-15, IGP-M, PTAX, IBC-Br (log), credit to legal entities (log), unemployment rate, employed persons, household indebtedness and household income commitment. Quarterly and annual values are period averages for rates and indices and end-of-period values for stocks.

**How to read.** The table gives the source code, unit, frequency and transformation of each series.

**Interpretation.** Indicators measured as stocks and as rates are aggregated differently, which matters when comparing frequencies.

*Output:* Table `tab_macro_dictionary`.

In [ ]:
MACRO_SERIES = {
    432: ("selic", "Selic target rate (% p.a.)", "mean"),
    433: ("ipca", "IPCA inflation (% per month)", "mean"),
    7478: ("ipca15", "IPCA-15 inflation (% per month)", "mean"),
    189: ("igpm", "IGP-M inflation (% per month)", "mean"),
    1: ("ptax", "PTAX exchange rate (R$/US$)", "mean"),
    4380: ("gdp_monthly", "Monthly GDP, current prices (log)", "log"),
    20540: ("credit_firms", "Credit to legal entities (log of balance)", "log"),
    24369: ("unemployment", "Unemployment rate (%)", "mean"),
    24379: ("employed", "Employed persons (thousand)", "mean"),
    29037: ("hh_indebtedness", "Household indebtedness (% of income)", "mean"),
    29034: ("hh_commitment", "Household income commitment (%, s.a.)", "mean"),
}

def macro_monthly() -> pl.DataFrame:
    codes = ", ".join(str(c) for c in MACRO_SERIES)
    con = duck(DUCKDB_MACRO)
    try:
        raw = con.execute(f"SELECT codigo_serie, data, valor FROM MACRO WHERE codigo_serie IN ({codes})").pl()
    finally:
        con.close()
    raw = raw.with_columns(pl.col("data").dt.truncate("1mo").alias("month"),
                           pl.col("codigo_serie").replace_strict({k: v[0] for k, v in MACRO_SERIES.items()}).alias("var"))
    wide = raw.group_by("month", "var").agg(pl.col("valor").mean()).pivot(on="var", index="month", values="valor").sort("month")
    wide = wide.with_columns(pl.all().exclude("month").forward_fill())
    logs = [v[0] for v in MACRO_SERIES.values() if v[2] == "log"]
    wide = wide.with_columns([pl.col(c).log() for c in logs])
    return wide.filter(pl.col("month").is_between(pl.date(YEARS[0] - 4, 1, 1), pl.date(SNAP_YEAR, SNAP_MONTH, 1)))

macro_m = collect(cached("1.6:monthly", macro_monthly))
MACRO_VARS = [v[0] for v in MACRO_SERIES.values()]
LOG_VARS = [v[0] for v in MACRO_SERIES.values() if v[2] == "log"]
macro_q = (macro_m.with_columns(pl.col("month").dt.truncate("1q").alias("quarter"))
           .group_by("quarter").agg([pl.col(c).last() if c in LOG_VARS else pl.col(c).mean() for c in MACRO_VARS])
           .sort("quarter"))
macro_dictionary = pl.DataFrame([{"variable": v[0], "description": v[1], "sgs_code": k,
                                  "monthly_aggregation": "log of monthly mean" if v[2] == "log" else "monthly mean",
                                  "quarterly_aggregation": "end of quarter" if v[2] == "log" else "quarterly mean"}
                                 for k, v in MACRO_SERIES.items()])
save_tab(macro_dictionary, "tab_macro_dictionary")
macro_dictionary

#### 1.7 Data dictionary and glossary

**Method.** Reference documentation of the derived variables.

**Data.** All variables constructed in 1.2 to 1.6.

**How to read.** One row per variable, with definition, unit and transformation.

**Interpretation.** The dictionary complements the Glossary at the start of the notebook.

*Output:* Table `tab_data_dictionary`.

In [ ]:
data_dictionary = pl.DataFrame([
    ("ID_DEBITO", "Integer", "Debt identifier (one per CDA)"),
    ("DATA_INSCRICAO", "Date", "Date of enrolment in the dívida ativa"),
    ("TIPO_DEBITO", "Text", "Raw debt type recorded in the source"),
    ("ICMS_ORIGIN", "Category", "declared, assessed (infraction notice, AIIM) or other"),
    ("SITUACAO_DEBITO", "Category", "Administrative situation of the CDA"),
    ("STATUS_AJUIZAMENTO_DEBITO", "Category", "Judicial filing status"),
    ("stock", "R$", "Value without fees in the last snapshot of the year"),
    ("paid", "R$", "Sum of GARE revenue paid on the CDA in the year"),
    ("rate", "Ratio", "Non-accumulated recovery rate: paid / stock"),
    ("paid_flag", "0/1", "At least one payment in the year"),
    ("age", "Years", "Time from enrolment to 31 December of the year"),
    ("enrol_year", "Year", "Enrolment cohort"),
    ("CNPJ_ROOT", "Text", "First eight digits of the CNPJ (firm); hashed in outputs"),
    ("collection", "R$", "Monthly GARE revenue on ICMS CDAs"),
    ("n_payers", "Count", "Monthly number of CDAs with a payment"),
    ("n_new", "Count", "Monthly number of newly enrolled CDAs"),
], schema=["variable", "type", "definition"], orient="row")
glossary = pl.DataFrame([
    ("CDA", "Certidão de Dívida Ativa: certificate of enrolled tax debt"),
    ("GARE", "Guia de Arrecadação Estadual: state payment slip"),
    ("AIIM", "Auto de Infração e Imposição de Multa: tax infraction notice"),
    ("Stock", "Outstanding value of enrolled debt at a given snapshot"),
    ("Cohort", "Set of CDAs enrolled in the same calendar year"),
], schema=["term", "definition"], orient="row")
save_tab(data_dictionary, "tab_data_dictionary")
data_dictionary

## Part A. The portfolio across tax types
This part describes the whole portfolio and assembles the evidence on which the restriction to ICMS rests (A.7).

*Terms introduced in this section:* ICMS, IPVA, ITCMD, logarithmic scale. Definitions are given in the Glossary.

### A.1 Volume of the database

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### A.1.1 Rows per table

**Method.** Count of the rows of each source table.

**Data.** All tables of the extraction.

**How to read.** Horizontal bars, one per table; the horizontal axis gives the number of rows on a logarithmic scale and the exact count is printed at the end of each bar.

**Interpretation.** The longest bar shows which table dominates the data volume; the repetition of CDAs in monthly snapshots explains the size of `DEBITO` and motivates the out-of-core processing of Section 0.

*Output:* Figure `fig_volume_tabelas`; Table `tab_inventory`.

In [ ]:
fig = bars(inventory, "table", "rows", log=True, val_title="Rows (log scale)", text_fmt="{:,.0f}")
save_fig(fig, "fig_volume_tabelas", width="double", ratio=0.5)
save_tab(inventory.sort("rows", descending=True), "tab_inventory", fmt="{:,.0f}")
fig

#### A.1.2 Schema

**Method.** Listing of the columns and storage types of each table.

**Data.** All tables of the extraction.

**How to read.** One row per column, with its table and type.

**Interpretation.** The schema documents the raw structure from which all derived variables are built.

*Output:* Table `tab_schema`.

In [ ]:
def schema_table() -> pl.DataFrame:
    con = duck()
    try:
        return con.execute("""SELECT table_name AS "table", column_name AS "column", data_type AS type
                              FROM information_schema.columns WHERE NOT starts_with(table_name, '_')
                              ORDER BY table_name, ordinal_position""").pl()
    finally:
        con.close()

schema = collect(cached("A.1.2", schema_table))
save_tab(schema, "tab_schema")
schema

#### A.1.3 Uniqueness of CDAs in the annual panel

**Method.** Consistency check of the deduplication of snapshots.

**Data.** Annual panel, 2016–2026.

**How to read.** For each year, the number of rows and the number of distinct CDAs.

**Interpretation.** Equal values confirm one observation per CDA and year; a larger row count would indicate duplicated snapshots.

*Output:* Table `tab_sanity_panel`.

In [ ]:
sanity = collect(year_end.group_by("year").agg(pl.len().alias("rows"), pl.col("ID_DEBITO").n_unique().alias("distinct_cda"))
                 .with_columns((pl.col("rows") == pl.col("distinct_cda")).alias("one_row_per_cda")).sort("year"))
save_tab(sanity, "tab_sanity_panel", fmt="{:,.0f}")
sanity

#### A.1.4 Structure of a single raw snapshot

**Method.** Profile of one raw monthly snapshot, computed on a subquery restricted to that month so that other months do not contaminate the profile.

**Data.** `DEBITO`, a single monthly snapshot.

**How to read.** Summary of the records in the snapshot.

**Interpretation.** The profile illustrates what a single snapshot contains before any aggregation.

*Output:* Table `tab_probe_month`.

In [ ]:
# USING SAMPLE binds to the FROM clause, so the snapshot filter goes into a subquery first
probe = collect(query("A.1.4", f"""SELECT * FROM (
                                      SELECT * EXCLUDE (NOME_DEVEDOR, CPF_DEVEDOR, CNPJ_DEVEDOR) FROM DEBITO
                                      WHERE ANO_EXTRACAO = {SNAP_YEAR} AND MES_EXTRACAO = {SNAP_MONTH}
                                        AND COALESCE(CPF_DEVEDOR, '') = '')
                                  USING SAMPLE 10 ROWS (reservoir, {SEED})""", params={"snapshot": EXTRACT}))
save_tab(probe.select(pl.all().cast(pl.Utf8)), "tab_probe_month")
probe

### A.2 Missing values

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### A.2.1 Fields with missing values

**Method.** Share of missing values per field, distinguishing structural gaps (field not applicable) from genuine gaps.

**Data.** All tables of the extraction.

**How to read.** Horizontal bars, one per field, sorted from the most to the least incomplete; the horizontal axis gives the percentage of missing values.

**Interpretation.** High shares in fields such as administrative fees are structural and need no treatment; gaps in key fields such as enrolment date or value limit what can be measured.

*Output:* Figure `fig_nulos_top`; Table `tab_nulls`.

In [ ]:
def null_shares() -> pl.DataFrame:
    con = duck()
    rows = []
    try:
        cols = con.execute("""SELECT table_name, column_name FROM information_schema.columns
                              WHERE NOT starts_with(table_name, '_')""").fetchall()
        by_table: dict[str, list[str]] = {}
        for t, c in cols:
            by_table.setdefault(t, []).append(c)
        for t, cs in by_table.items():
            where = f"WHERE ANO_EXTRACAO = {SNAP_YEAR} AND MES_EXTRACAO = {SNAP_MONTH}" if t == "DEBITO" else ""
            exprs = ", ".join(f"""AVG(CASE WHEN "{c}" IS NULL OR CAST("{c}" AS VARCHAR) = '' THEN 1 ELSE 0 END) AS "{c}" """ for c in cs)
            res = con.execute(f'SELECT {exprs} FROM "{t}" {where}').fetchone()
            rows += [{"table": t, "column": c, "null_share": float(v)} for c, v in zip(cs, res)]
    finally:
        con.close()
    return pl.DataFrame(rows)

nulls = collect(cached("A.2.1", null_shares, params={"snapshot": EXTRACT}))
top_nulls = nulls.filter(pl.col("null_share") > 0).sort("null_share", descending=True).head(20) \
                 .with_columns((pl.col("table") + "." + pl.col("column")).alias("field"), (100 * pl.col("null_share")).alias("pct"))
fig = bars(top_nulls, "field", "pct", val_title="Missing values (%)")
save_fig(fig, "fig_nulos_top", width="double")
save_tab(nulls.sort("table", "null_share", descending=[False, True]), "tab_nulls", fmt="{:.4f}")
fig

#### A.2.2 Missing values by tax type

**Method.** Share of missing values in key fields by tax type.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** Rows are tax types; columns give the percentage missing for enrolment date, administrative situation, filing status and value.

**Interpretation.** Marked differences across tax types indicate that data quality itself may justify treating some types separately (criterion v of A.7).

*Output:* Table `tab_nulls_by_tax`.

In [ ]:
key_fields = ["DATA_INSCRICAO", "SITUACAO_DEBITO", "STATUS_AJUIZAMENTO_DEBITO", "value"]
nulls_by_tax = collect(snapshot_all.group_by("bucket").agg(
    [(pl.col(c).is_null().mean() * 100).alias(c) for c in key_fields]
    + [((pl.col("identifier") == "None").mean() * 100).alias("debtor_identifier")]).sort("bucket"))
save_tab(nulls_by_tax, "tab_nulls_by_tax", fmt="{:.3f}")
nulls_by_tax

### A.3 Grouping of debt types

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### A.3.1 Classification rule

**Method.** Assignment of raw debt types to tax groups (ICMS, IPVA, ITCMD, fees and other) by a text rule applied to the type name.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** One row per raw type, with its assigned group.

**Interpretation.** The rule is transparent and can be checked type by type.

*Output:* Table `tab_tipo_debito_buckets`.

In [ ]:
bucket_table = collect(snapshot_all.group_by("TIPO_DEBITO", "bucket").agg(
    pl.len().alias("n_cda"), (pl.col("value").sum() / 1e9).alias("stock_bn")).sort("bucket", "n_cda", descending=[False, True]))
bucket_table = bucket_table.with_columns(pl.col("bucket").replace_strict(
    {"ICMS": "contains 'ICMS'", "IPVA": "contains 'IPVA'", "ITCMD": "contains 'ITCMD'"}, default="none of the above").alias("rule"))
save_tab(bucket_table, "tab_tipo_debito_buckets", fmt="{:,.3f}")
bucket_table

#### A.3.2 Composition of the ITCMD group

**Method.** Listing of the raw types forming the ITCMD group.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** One row per raw type in the group.

**Interpretation.** The table documents the scope of the ITCMD group in the cross-tax comparisons.

*Output:* Table `tab_itcmd_types`.

In [ ]:
itcmd = bucket_table.filter(pl.col("bucket") == "ITCMD")
save_tab(itcmd, "tab_itcmd_types", fmt="{:,.3f}")
itcmd

### A.4 Frequency and monetary value

*Terms introduced in this section:* Value of a CDA, ECDF. Definitions are given in the Glossary.

#### A.4.1 Number of CDAs by tax type

**Method.** Count of outstanding CDAs by tax group.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** Horizontal bars; the horizontal axis gives the number of CDAs in millions, with the value printed at the end of each bar.

**Interpretation.** Compare with A.4.2: a group that ranks high here but low there consists of many small debts.

*Output:* Figure `fig_bucket_n_id`.

In [ ]:
composition = collect(snapshot_all.group_by("bucket").agg(
    pl.len().alias("n_cda"), pl.col("value").sum().alias("stock"),
    pl.col("value").mean().alias("mean_value"), pl.col("value").median().alias("median_value")))
composition = composition.with_columns((100 * pl.col("n_cda") / pl.col("n_cda").sum()).alias("pct_cda"),
                                       (100 * pl.col("stock") / pl.col("stock").sum()).alias("pct_stock"),
                                       (pl.col("stock") / 1e9).alias("stock_bn"))
fig = bars(composition.with_columns((pl.col("n_cda") / 1e6).alias("n_cda_m")), "bucket", "n_cda_m",
           val_title="CDAs (million)", text_fmt="{:,.2f}")
save_fig(fig, "fig_bucket_n_id")
fig

#### A.4.2 Stock by tax type

**Method.** Outstanding stock by tax group.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** Horizontal bars; the horizontal axis gives the stock in R$ billion.

**Interpretation.** A group that dominates the stock but not the count holds fewer, larger debts.

*Output:* Figure `fig_bucket_valor`.

In [ ]:
fig = bars(composition, "bucket", "stock_bn", val_title="Stock (R$ bn)")
save_fig(fig, "fig_bucket_valor")
fig

#### A.4.3 Share of CDAs versus share of value

**Method.** Paired comparison of each group's share of the number of CDAs and of the stock.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** One row per tax group; the circle marks the share of CDAs and the diamond the share of stock, both in percent on a common horizontal axis, joined by a grey line; values are printed next to the markers.

**Interpretation.** A long line indicates that count and value are strongly out of proportion. A diamond to the right of the circle identifies a group whose debts are larger than average. This is the main evidence for criterion (i) of A.7.

*Output:* Figure `fig_bucket_pct_qtd_valor`; Table `tab_composition`.

In [ ]:
d = composition.sort("pct_stock")
fig = go.Figure()
for b, a, s in zip(d["bucket"], d["pct_cda"], d["pct_stock"]):
    fig.add_trace(go.Scatter(x=[a, s], y=[b, b], mode="lines", line=dict(color="#C9CED1", width=3),
                             hoverinfo="skip", showlegend=False))
for i, (col, other, name, sym) in enumerate((("pct_cda", "pct_stock", "Share of CDAs", "circle"),
                                              ("pct_stock", "pct_cda", "Share of stock", "diamond"))):
    pos = ["middle right" if v >= o else "middle left" for v, o in zip(d[col], d[other])]
    fig.add_trace(go.Scatter(x=d[col], y=d["bucket"], mode="markers+text", name=name, cliponaxis=False,
                             marker=dict(color=color(i), size=11, symbol=sym), text=[f"  {v:.1f}%  " for v in d[col]],
                             textposition=pos, textfont=dict(color=color(i), size=9)))
fig.update_layout(xaxis=dict(title="Share of the portfolio (%)", range=[-12, 115], showgrid=True, gridcolor="#E6E6E6"),
                  yaxis=dict(showgrid=False), showlegend=True,
                  legend=dict(orientation="h", y=-0.28, x=0.5, xanchor="center"),
                  meta={"min_height": 120 + 45 * d.height})
save_fig(fig, "fig_bucket_pct_qtd_valor", width="double", ratio=0.45)
save_tab(composition.select("bucket", "n_cda", "stock_bn", "pct_cda", "pct_stock").sort("pct_stock", descending=True),
         "tab_composition")
fig

#### A.4.4 Average value per CDA

**Method.** Mean and median value per CDA by tax group.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** One row per group; columns give mean and median value in R$.

**Interpretation.** Because value distributions are skewed, a mean far above the median indicates that a few very large debts dominate the group's total.

*Output:* Table `tab_ticket_by_tax`.

In [ ]:
ticket = composition.select("bucket", "mean_value", "median_value").sort("mean_value", descending=True)
save_tab(ticket, "tab_ticket_by_tax", fmt="{:,.0f}")
ticket

#### A.4.5 Distribution of CDA values

**Method.** Empirical cumulative distribution functions (ECDF) of CDA value. An ECDF needs no choice of bins and shows the whole distribution, which makes it preferable to histograms for comparing several groups.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA. CDAs with positive value.

**How to read.** The horizontal axis gives CDA value in R$ on a logarithmic scale; the vertical axis gives the cumulative share of CDAs with value up to that point; one line per tax group, identified in the legend.

**Interpretation.** A curve further to the right corresponds to larger debts throughout the distribution; a steep curve indicates values concentrated in a narrow range; crossing curves indicate that the groups differ in dispersion rather than only in level.

*Output:* Figure `fig_value_dist_by_tax`.

In [ ]:
value_sample = collect(snapshot_all.select("bucket", "value").filter(pl.col("value") > 0)
                       .group_by("bucket").agg(pl.col("value").sample(n=200_000, with_replacement=True, seed=SEED))
                       .explode("value"))
fig = ecdf(value_sample, "value", "bucket", x_title="CDA value (R$, log scale)")
save_fig(fig, "fig_value_dist_by_tax")
fig

### A.5 Stock and age

*Terms introduced in this section:* Interquartile range. Definitions are given in the Glossary.

#### A.5.1 Stock by tax type over time

**Method.** Year-end stock by tax group over time.

**Data.** Year-end snapshots of `DEBITO`, all tax types, 2016–2026; unit of observation: the group-year.

**How to read.** The horizontal axis gives the year; the vertical axis gives stock in R$ billion on a logarithmic scale; one line per group, identified in the legend.

**Interpretation.** On a logarithmic axis, parallel lines mean equal proportional growth and therefore a stable composition; converging or diverging lines indicate a change in composition.

*Output:* Figure `fig_stock_by_tax_year`.

In [ ]:
STOCK_BY_TAX_SQL = f"""
WITH last_month AS (
    SELECT ANO_EXTRACAO, MAX(MES_EXTRACAO) AS m FROM DEBITO
    WHERE ANO_EXTRACAO BETWEEN {YEARS[0]} AND {YEARS[-1]} AND MES_EXTRACAO BETWEEN 1 AND 12 GROUP BY 1)
SELECT d.ANO_EXTRACAO AS year, {BUCKET_SQL} AS bucket, COUNT(*) AS n_cda, SUM(VALOR_SEM_HONORARIOS) AS stock
FROM DEBITO d JOIN last_month l ON d.ANO_EXTRACAO = l.ANO_EXTRACAO AND d.MES_EXTRACAO = l.m
GROUP BY ALL
"""
stock_by_tax = collect(query("A.5.1", STOCK_BY_TAX_SQL, params={"snapshot": EXTRACT}))
fig = lines(stock_by_tax.with_columns((pl.col("stock") / 1e9).alias("stock_bn")), "year", "stock_bn", "bucket",
            log_y=True, y_title="Year-end stock (R$ bn, log scale)")
save_fig(fig, "fig_stock_by_tax_year")
fig

#### A.5.2 Age of CDAs by tax type

**Method.** Median, mean and interquartile range of CDA age by tax group.

**Data.** March 2026 snapshot of `DEBITO` for all tax types (1.2); legal entities only; unit of observation: the CDA.

**How to read.** One row per group; the dot marks the median, the diamond the mean and the grey bar the interquartile range, all in years; the legend identifies the markers.

**Interpretation.** A mean well to the right of the median indicates a long tail of old debts; a wide bar indicates heterogeneous enrolment dates.

*Output:* Figure `fig_idade_cda_bucket`; Table `tab_age_by_tax`.

In [ ]:
age_by_tax = collect(snapshot_all.group_by("bucket").agg(
    pl.col("age_years").mean().alias("mean_age"), pl.col("age_years").median().alias("median_age"),
    pl.col("age_years").quantile(0.25).alias("q25"), pl.col("age_years").quantile(0.75).alias("q75")).sort("mean_age"))
d = age_by_tax.sort("median_age")
fig = go.Figure()
for b, lo, hi in zip(d["bucket"], d["q25"], d["q75"]):
    fig.add_trace(go.Scatter(x=[lo, hi], y=[b, b], mode="lines", line=dict(color="#C9CED1", width=4),
                             hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#C9CED1", width=4), name="Interquartile range"))
fig.add_trace(go.Scatter(x=d["median_age"], y=d["bucket"], mode="markers+text", name="Median", cliponaxis=False,
                         marker=dict(color=CEMEPI["deep_teal"], size=10), text=[f"{v:.1f}" for v in d["median_age"]],
                         textposition="top center", textfont=dict(size=9, color=CEMEPI["deep_teal"])))
fig.add_trace(go.Scatter(x=d["mean_age"], y=d["bucket"], mode="markers", name="Mean",
                         marker=dict(color=CEMEPI["sand"], size=9, symbol="diamond-open", line=dict(width=1.5))))
fig.update_layout(xaxis=dict(title="Age of CDA (years)", rangemode="tozero", showgrid=True, gridcolor="#E6E6E6"),
                  yaxis=dict(showgrid=False), showlegend=True,
                  legend=dict(orientation="h", y=-0.3, x=0.5, xanchor="center"), meta={"min_height": 120 + 45 * d.height})
save_fig(fig, "fig_idade_cda_bucket", width="double", ratio=0.45)
save_tab(age_by_tax, "tab_age_by_tax")
fig

### A.6 Recovery and payments

*Terms introduced in this section:* Non-accumulated recovery, payment indicator. Definitions are given in the Glossary.

#### A.6.1 Recovery rate by tax type

**Method.** Annual non-accumulated recovery rate by tax group: amount paid in the year divided by year-end stock.

**Data.** Year-end snapshots of `DEBITO` and GARE payments, all tax types, 2016–2026; unit of observation: the group-year.

**How to read.** The horizontal axis gives the year; the vertical axis gives the rate in percent on a logarithmic scale; one line per group, identified in the legend.

**Interpretation.** A persistent vertical gap between lines reflects a structural difference in recovery between tax groups; lines that rise together in the same years reflect common shocks such as settlement programmes.

*Output:* Figure `fig_recovery_by_tax`.

In [ ]:
RECOVERY_BY_TAX_SQL = f"""
WITH last_month AS (
    SELECT ANO_EXTRACAO, MAX(MES_EXTRACAO) AS m FROM DEBITO
    WHERE ANO_EXTRACAO BETWEEN {YEARS[0]} AND {YEARS[-1]} AND MES_EXTRACAO BETWEEN 1 AND 12 GROUP BY 1),
ye AS (
    SELECT d.ANO_EXTRACAO AS year, d.ID_DEBITO, {BUCKET_SQL} AS bucket, d.VALOR_SEM_HONORARIOS AS stock
    FROM DEBITO d JOIN last_month l ON d.ANO_EXTRACAO = l.ANO_EXTRACAO AND d.MES_EXTRACAO = l.m),
pay AS (
    SELECT YEAR(DATA_ARRECADACAO_GARE) AS year, ID_DEBITO, SUM(VALOR_RECEITA) AS paid
    FROM ARRECADACAO GROUP BY ALL)
SELECT ye.year, ye.bucket, COUNT(*) AS n_cda, SUM(ye.stock) AS stock,
       SUM(COALESCE(pay.paid, 0)) AS paid, AVG(CASE WHEN pay.paid > 0 THEN 1 ELSE 0 END) AS share_paying
FROM ye LEFT JOIN pay ON ye.year = pay.year AND ye.ID_DEBITO = pay.ID_DEBITO
GROUP BY ALL
"""
recovery_by_tax = collect(query("A.6.1", RECOVERY_BY_TAX_SQL, params={"snapshot": EXTRACT})) \
    .with_columns((100 * pl.col("paid") / pl.col("stock")).alias("rate_pct"))
fig = lines(recovery_by_tax.filter(pl.col("rate_pct") > 0), "year", "rate_pct", "bucket", log_y=True,
            y_title="Recovery rate (% of year-end stock, log scale)")
save_fig(fig, "fig_recovery_by_tax")
fig

#### A.6.2 Share of paying CDAs by tax type

**Method.** Annual share of CDAs with at least one payment, by tax group.

**Data.** As in A.6.1.

**How to read.** Rows are tax groups and years; the value is the percentage of CDAs paying.

**Interpretation.** This separates whether debts are paid at all (extensive margin) from how much is recovered (A.6.1).

*Output:* Table `tab_payers_by_tax`.

In [ ]:
payers_by_tax = recovery_by_tax.with_columns((100 * pl.col("share_paying")).alias("pct_paying")) \
    .pivot(on="bucket", index="year", values="pct_paying").sort("year")
save_tab(payers_by_tax, "tab_payers_by_tax", fmt="{:.2f}")
payers_by_tax

#### A.6.3 Monthly enrolment frequency by tax type

**Method.** Monthly count of new CDAs by tax group.

**Data.** `DEBITO`, all tax types, enrolment dates from January 2016 to March 2026, with implausible dates removed; unit of observation: the group-month.

**How to read.** The horizontal axis gives the month; the vertical axis gives the number of new CDAs on a logarithmic scale; one line per group, identified in the legend.

**Interpretation.** Groups with many small debts produce large and irregular counts that would dominate any analysis based on event frequencies (criterion iii of A.7).

*Output:* Figure `fig_enrolment_freq_by_tax`.

In [ ]:
ENROLMENT_BY_TAX_SQL = f"""
WITH ids AS (
    SELECT DISTINCT ID_DEBITO, {BUCKET_SQL} AS bucket, {enrol_date_sql()} AS enrol
    FROM DEBITO WHERE ANO_EXTRACAO BETWEEN {YEARS[0]} AND {YEARS[-1]} AND MES_EXTRACAO BETWEEN 1 AND 12)
SELECT DATE_TRUNC('month', enrol) AS month, bucket, COUNT(*) AS n_new
FROM ids WHERE enrol >= DATE '{YEARS[0]}-01-01' GROUP BY ALL
"""
enrol_by_tax = collect(query("A.6.3", ENROLMENT_BY_TAX_SQL, params={"snapshot": EXTRACT}))
fig = lines(enrol_by_tax.filter(pl.col("n_new") > 0), "month", "n_new", "bucket", log_y=True, markers=False,
            y_title="New CDAs per month (log scale)")
save_fig(fig, "fig_enrolment_freq_by_tax", width="double")
fig

### A.7 Segmentation decision

*Terms introduced in this section:* ICMS Declarado, ICMS Autuação (AIIM). Definitions are given in the Glossary.

#### A.7.1 Criteria for an ICMS-only analysis

**Method.** Structured assessment of the restriction of Parts B to D to ICMS against six explicit criteria: share of the stock, value per CDA and its dispersion, distortion of event frequencies, age and recovery dynamics, data completeness and institutional relevance.

**Data.** Results of A.2 to A.6.

**How to read.** One row per criterion, with the evidence, its implication and what is lost by excluding the other tax groups.

**Interpretation.** The decision rests on the joint weight of the criteria rather than on any single one.

*Output:* Table `tab_segmentation_criteria`.

In [ ]:
comp = {r["bucket"]: r for r in composition.iter_rows(named=True)}
ages = {r["bucket"]: r for r in age_by_tax.iter_rows(named=True)}
rec = recovery_by_tax.group_by("bucket").agg((100 * pl.col("paid").sum() / pl.col("stock").sum()).alias("r"))
rec = dict(zip(rec["bucket"], rec["r"]))
enrol_share = enrol_by_tax.group_by("bucket").agg(pl.col("n_new").sum()).with_columns(
    (100 * pl.col("n_new") / pl.col("n_new").sum()).alias("pct"))
enrol_share = dict(zip(enrol_share["bucket"], enrol_share["pct"]))
miss = dict(zip(nulls_by_tax["bucket"], nulls_by_tax["debtor_identifier"]))
segmentation = pl.DataFrame([
    ("Share of the stock", f"ICMS {comp['ICMS']['pct_stock']:.1f}% vs IPVA {comp['IPVA']['pct_stock']:.1f}%", "ICMS dominates the monetary portfolio"),
    ("Share of CDAs", f"ICMS {comp['ICMS']['pct_cda']:.1f}% vs IPVA {comp['IPVA']['pct_cda']:.1f}%", "IPVA dominates counts but not value"),
    ("Mean value per CDA (R$)", f"ICMS {comp['ICMS']['mean_value']:,.0f} vs IPVA {comp['IPVA']['mean_value']:,.0f}", "Ticket sizes differ by orders of magnitude"),
    ("Share of new enrolments", f"ICMS {enrol_share.get('ICMS', float('nan')):.1f}% vs IPVA {enrol_share.get('IPVA', float('nan')):.1f}%", "Pooling would let IPVA drive event frequencies"),
    ("Median age (years)", f"ICMS {ages['ICMS']['median_age']:.1f} vs IPVA {ages['IPVA']['median_age']:.1f}", "Distinct age structures"),
    ("Recovery rate 2016-2026 (%)", f"ICMS {rec['ICMS']:.2f} vs IPVA {rec['IPVA']:.2f}", "Distinct recovery dynamics"),
    ("Missing debtor identifier (%)", f"ICMS {miss['ICMS']:.3f} vs IPVA {miss['IPVA']:.3f}", "Data completeness by tax"),
    ("Institutional relevance", "Collection policy of PGE-SP centred on ICMS", "ICMS is the policy-relevant portfolio"),
], schema=["criterion", "evidence", "implication"], orient="row")
save_tab(segmentation, "tab_segmentation_criteria")
segmentation

#### A.7.2 Declared versus assessed ICMS

**Method.** Breakdown of the ICMS stock by raw type.

**Data.** March 2026 snapshot of `DEBITO` restricted to ICMS (1.2); legal entities only; unit of observation: the CDA.

**How to read.** One row per raw ICMS type, with CDA count, stock and value per CDA.

**Interpretation.** A sharp contrast between declared and assessed debts in number and unit value would justify carrying the origin of the debt through the later analyses; the formal tests are in B.2.

*Output:* Table `tab_icms_types`.

In [ ]:
icms_types = bucket_table.filter(pl.col("bucket") == "ICMS").with_columns(
    (100 * pl.col("n_cda") / pl.col("n_cda").sum()).alias("pct_cda"),
    (100 * pl.col("stock_bn") / pl.col("stock_bn").sum()).alias("pct_stock"),
    (pl.col("stock_bn") * 1e9 / pl.col("n_cda")).alias("mean_value")).drop("bucket", "rule")
save_tab(icms_types, "tab_icms_types")
icms_types

## Part B. The ICMS portfolio

*Terms introduced in this section:* Portfolio recovery rate. Definitions are given in the Glossary.

### B.1 Evolution of the portfolio

*Terms introduced in this section:* Bootstrap, settlement programme. Definitions are given in the Glossary.

#### B.1.1 Annual aggregates

**Method.** Annual aggregates of the ICMS portfolio. The portfolio recovery rate weights CDAs by their stock (total paid over total stock, computed only when the stock is positive); the mean CDA rate gives each CDA equal weight.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** One row per year: year-end stock, amount collected, portfolio recovery rate, mean CDA recovery rate and share of CDAs paying.

**Interpretation.** A gap between the portfolio rate and the mean CDA rate indicates that recovery differs between large and small debts.

*Output:* Table `tab_serie_agregada_ano`.

In [ ]:
annual = collect(panel.group_by("year").agg(
    pl.len().alias("n_cda"), pl.col("stock").sum().alias("stock"), pl.col("paid").sum().alias("collected"),
    pl.col("rate").filter(pl.col("stock") > 0).mean().alias("mean_cda_rate"),
    pl.col("paid_flag").mean().alias("share_paying")).sort("year"))
annual = annual.with_columns(pl.when(pl.col("stock") > 0).then(pl.col("collected") / pl.col("stock")).alias("portfolio_rate"),
                             (pl.col("stock") / 1e9).alias("stock_bn"), (pl.col("collected") / 1e9).alias("collected_bn"))
save_tab(annual.select("year", "n_cda", "stock_bn", "collected_bn",
                       (100 * pl.col("portfolio_rate")).alias("portfolio_rate_pct"),
                       (100 * pl.col("mean_cda_rate")).alias("mean_cda_rate_pct"),
                       (100 * pl.col("share_paying")).alias("share_paying_pct")), "tab_serie_agregada_ano", fmt="{:,.3f}")
annual

#### B.1.2 Stock by year

**Method.** Year-end ICMS stock.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** Vertical bars, one per year; the vertical axis gives the stock in R$ billion, printed above each bar.

**Interpretation.** Year-to-year changes combine new enrolments, payments and exits; a sudden fall may reflect cancellations or a settlement programme.

*Output:* Figure `fig_estoque_agregado_ano`.

In [ ]:
fig = go.Figure(go.Bar(x=annual["year"], y=annual["stock_bn"], marker_color=CEMEPI["deep_teal"],
                       text=[f"{v:,.0f}" for v in annual["stock_bn"]], textposition="outside"))
fig.update_layout(yaxis_title="Year-end ICMS stock (R$ bn)")
save_fig(fig, "fig_estoque_agregado_ano")
fig

#### B.1.3 Stock and collection

**Method.** Joint display of stock and collection in two aligned panels with a common time axis, which avoids the arbitrary visual scaling of a dual axis.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** Upper panel: year-end stock in R$ billion; lower panel: annual collection in R$ billion; the horizontal axis gives the year.

**Interpretation.** Compare the timing of movements across panels rather than their heights.

*Output:* Figure `fig_serie_agregada_icms`.

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_trace(go.Scatter(x=annual["year"], y=annual["stock_bn"], mode="lines+markers", line_color=CEMEPI["deep_teal"]), 1, 1)
fig.add_trace(go.Bar(x=annual["year"], y=annual["collected_bn"], marker_color=CEMEPI["sand"]), 2, 1)
fig.update_yaxes(title_text="Stock (R$ bn)", row=1, col=1)
fig.update_yaxes(title_text="Collected (R$ bn)", row=2, col=1)
save_fig(fig, "fig_serie_agregada_icms", ratio=0.9)
fig

#### B.1.4 Mean CDA recovery rate by year

**Method.** Annual mean of CDA-level recovery rates with a 95% confidence interval from a nonparametric bootstrap (200 resamples of CDAs with replacement within each year; interval from the 2.5th and 97.5th percentiles). The bootstrap is used because rates are skewed and zero-inflated, which makes normal-theory intervals unreliable.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year. CDA-years with positive stock.

**How to read.** The horizontal axis gives the year; the line gives the mean rate in percent and the shaded band its 95% bootstrap interval, as identified in the legend.

**Interpretation.** A wide band indicates that the mean is driven by a few CDAs with very high rates; means above 100% arise when payments in a year exceed the year-end stock, for instance when a debt is largely paid off during the year.

*Output:* Figure `fig_painel_taxa_media_ano`.

In [ ]:
rates = collect(panel.filter(pl.col("stock") > 0).select("year", "rate"))
rng = np.random.default_rng(SEED)
rows = []
for (y,), d in rates.group_by("year"):
    v = d["rate"].to_numpy()
    boots = [rng.choice(v, size=v.size).mean() for _ in range(200)]
    rows.append({"year": y, "mean": v.mean() * 100, "lo": np.percentile(boots, 2.5) * 100, "hi": np.percentile(boots, 97.5) * 100})
ci = pl.DataFrame(rows).sort("year")
fig = go.Figure([
    go.Scatter(x=ci["year"].to_list() + ci["year"].to_list()[::-1], y=ci["hi"].to_list() + ci["lo"].to_list()[::-1],
               fill="toself", fillcolor="rgba(21,76,86,0.15)", line_width=0, hoverinfo="skip", name="95% bootstrap interval"),
    go.Scatter(x=ci["year"], y=ci["mean"], mode="lines+markers", line_color=CEMEPI["deep_teal"], name="Mean")])
fig.update_layout(yaxis_title="Mean CDA recovery rate (%)", xaxis_title="Year", showlegend=True,
                  legend=dict(orientation="h", y=-0.3))
save_fig(fig, "fig_painel_taxa_media_ano")
fig

#### B.1.5 New enrolments

**Method.** Number and value of newly enrolled ICMS CDAs per year.

**Data.** ICMS extract (1.3), enrolment years 2016–2026; unit of observation: the enrolment year.

**How to read.** Left panel: new CDAs in thousands; right panel: enrolled value in R$ billion; the horizontal axis gives the year.

**Interpretation.** Years with many enrolments but little value indicate an inflow of small debts; the final year covers only January to March.

*Output:* Figure `fig_new_enrolments_year`.

In [ ]:
new_year = collect(attributes.with_columns(pl.col("DATA_INSCRICAO").dt.year().alias("year"))
                   .filter(pl.col("year") >= YEARS[0]).group_by("year")
                   .agg(pl.len().alias("n_new"), (pl.col("max_value").sum() / 1e9).alias("value_bn")).sort("year"))
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
fig.add_trace(go.Bar(x=new_year["year"], y=new_year["n_new"] / 1e3, marker_color=CEMEPI["deep_teal"]), 1, 1)
fig.add_trace(go.Bar(x=new_year["year"], y=new_year["value_bn"], marker_color=CEMEPI["sand"]), 1, 2)
fig.update_yaxes(title_text="New CDAs (thousand)", row=1, col=1)
fig.update_yaxes(title_text="Value enrolled (R$ bn)", row=1, col=2)
save_fig(fig, "fig_new_enrolments_year", width="double")
fig

#### B.1.6 Monthly overview

**Method.** Monthly overview of stock, new enrolments and collection against the calendar of settlement programmes.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** Three panels on a common monthly axis: stock (R$ billion), new CDAs (thousand) and collection (R$ billion); cream bands mark the PEP 2019 and Acordo Paulista 2024 windows.

**Interpretation.** Peaks inside the bands are associated with programmes; isolated spikes or dips in the stock outside them may reflect data events such as batch cancellations.

*Output:* Figure `fig_icms_monthly_overview`.

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05)
fig.add_trace(go.Scatter(x=monthly_all["month"], y=bn(monthly_all["stock"]), line_color=CEMEPI["deep_teal"]), 1, 1)
fig.add_trace(go.Scatter(x=monthly_all["month"], y=monthly_all["n_new"] / 1e3, line_color=CEMEPI["teal"]), 2, 1)
fig.add_trace(go.Scatter(x=monthly_all["month"], y=bn(monthly_all["collection"]), line_color=CEMEPI["sand"]), 3, 1)
for r, t in enumerate(("Stock (R$ bn)", "New CDAs (thousand)", "Collection (R$ bn)"), start=1):
    fig.update_yaxes(title_text=t, row=r, col=1)
shade_programmes(fig)
save_fig(fig, "fig_icms_monthly_overview", width="double", ratio=0.8)
fig

### B.2 Raw ICMS debt types and the case for grouping
Before any grouping, the five raw ICMS debt types recorded in the source are analysed separately, and their homogeneity is tested formally. The grouping adopted in B.3 rests on this evidence.

*Terms introduced in this section:* Simples Nacional, Fecoep, GIA, judicial filing, Kruskal–Wallis, Mann–Whitney *U*, Dunn test, rank-biserial correlation, epsilon-squared, Kolmogorov–Smirnov, Anderson–Darling, chi-square test, Fisher exact test, Cramér's *V*, Benjamini–Hochberg *q*-value. Definitions are given in the Glossary.

#### B.2.1 Counts and stock by raw type

**Method.** Counts and stock by raw ICMS type.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA. Annual counts from the year-end snapshots.

**How to read.** Left panel: number of CDAs; right panel: stock in R$ billion; both horizontal axes are logarithmic.

**Interpretation.** Raw types with very few CDAs cannot sustain separate analysis and are candidates for pooling.

*Output:* Figure `fig_icms_raw_counts_value`; Table `tab_icms_raw_summary`.

In [ ]:
snap_icms = (scan_icms().filter((pl.col("year") == SNAP_YEAR) & (pl.col("MES_EXTRACAO") == SNAP_MONTH))
             .with_columns(((pl.date(SNAP_YEAR, SNAP_MONTH, 1) - pl.col("DATA_INSCRICAO")).dt.total_days() / 365.25).alias("age")))
raw_summary = collect(snap_icms.group_by("TIPO_DEBITO").agg(
    pl.len().alias("n_cda"), (pl.col("VALOR_SEM_HONORARIOS").sum() / 1e9).alias("stock_bn")).sort("n_cda", descending=True))
raw_by_year = collect(panel.group_by("year", "TIPO_DEBITO").agg(pl.len().alias("n_cda"),
                                                                (pl.col("stock").sum() / 1e9).alias("stock_bn")))
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.18)
fig.add_trace(go.Bar(y=raw_summary["TIPO_DEBITO"], x=raw_summary["n_cda"], orientation="h", marker_color=CEMEPI["deep_teal"]), 1, 1)
fig.add_trace(go.Bar(y=raw_summary["TIPO_DEBITO"], x=raw_summary["stock_bn"], orientation="h", marker_color=CEMEPI["sand"]), 1, 2)
fig.update_xaxes(type="log", title_text="CDAs (log)", row=1, col=1)
fig.update_xaxes(type="log", title_text="Stock, R$ bn (log)", row=1, col=2)
fig.update_yaxes(showticklabels=False, row=1, col=2)
save_fig(fig, "fig_icms_raw_counts_value", width="double")
save_tab(raw_summary.join(raw_by_year.pivot(on="year", index="TIPO_DEBITO", values="n_cda"), on="TIPO_DEBITO", how="left"),
         "tab_icms_raw_summary", fmt="{:,.2f}")
fig

#### B.2.2 Value per CDA by raw type

**Method.** Location, dispersion and ECDF of CDA value by raw type.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** The table gives mean, median and interquartile range; in the figure the horizontal axis gives value in R$ on a logarithmic scale and the vertical axis the cumulative share; one line per raw type, identified in the legend.

**Interpretation.** Close curves indicate similar value distributions; a curve clearly to the right indicates systematically larger debts.

*Output:* Figure `fig_icms_raw_value_ecdf`; Table `tab_icms_raw_ticket`.

In [ ]:
raw_values = collect(snap_icms.select("TIPO_DEBITO", pl.col("VALOR_SEM_HONORARIOS").alias("value"), "age", "ICMS_ORIGIN",
                                     "STATUS_AJUIZAMENTO_DEBITO", "SITUACAO_DEBITO", "ID_DEBITO", "CNPJ_ROOT"))
ticket_raw = raw_values.group_by("TIPO_DEBITO").agg(
    pl.col("value").mean().alias("mean"), pl.col("value").median().alias("median"),
    pl.col("value").quantile(0.25).alias("q25"), pl.col("value").quantile(0.75).alias("q75")).sort("mean", descending=True)
fig = ecdf(raw_values, "value", "TIPO_DEBITO", x_title="CDA value (R$, log scale)")
save_fig(fig, "fig_icms_raw_value_ecdf", width="double")
save_tab(ticket_raw, "tab_icms_raw_ticket", fmt="{:,.0f}")
fig

#### B.2.3 Age by raw type

**Method.** Distribution of CDA age by raw type.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** Box plots: the box spans the interquartile range, the inner line marks the median and the whiskers the range of non-outlying values; the vertical axis gives age in years.

**Interpretation.** Similar boxes indicate types enrolled over similar periods; a type concentrated at low ages was introduced or became frequent only recently.

*Output:* Figure `fig_icms_raw_age`; Table `tab_icms_raw_age`.

In [ ]:
age_raw = raw_values.group_by("TIPO_DEBITO").agg(
    pl.col("age").mean().alias("mean"), pl.col("age").median().alias("median"),
    pl.col("age").quantile(0.25).alias("q25"), pl.col("age").quantile(0.75).alias("q75")).sort("median")
fig = go.Figure([go.Box(y=d["age"].sample(min(d.height, 50_000), seed=SEED), name=k[0], marker_color=color(i), boxpoints=False)
                 for i, (k, d) in enumerate(raw_values.sort("TIPO_DEBITO").group_by("TIPO_DEBITO", maintain_order=True))])
fig.update_layout(yaxis_title="Age of CDA (years)")
save_fig(fig, "fig_icms_raw_age", width="double")
save_tab(age_raw, "tab_icms_raw_age")
fig

#### B.2.4 Recovery by raw type

**Method.** Annual non-accumulated recovery rate by raw type.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** The horizontal axis gives the year; the vertical axis gives the rate in percent on a logarithmic scale; one line per raw type, identified in the legend. Missing points mark years without payments.

**Interpretation.** Lines with similar level and shape support pooling; persistent gaps argue against it.

*Output:* Figure `fig_icms_raw_recovery`; Table `tab_icms_raw_recovery`.

In [ ]:
rec_raw = collect(panel.group_by("year", "TIPO_DEBITO").agg(
    (100 * pl.col("paid").sum() / pl.col("stock").sum()).alias("rate_pct"),
    pl.col("rate").filter(pl.col("stock") > 0).median().alias("median_cda_rate"),
    pl.col("rate").filter(pl.col("stock") > 0).mean().alias("mean_cda_rate")))
fig = lines(rec_raw.filter(pl.col("rate_pct") > 0), "year", "rate_pct", "TIPO_DEBITO", log_y=True,
            y_title="Recovery rate (% of stock, log scale)")
save_fig(fig, "fig_icms_raw_recovery", width="double")
save_tab(rec_raw.pivot(on="TIPO_DEBITO", index="year", values="rate_pct").sort("year"), "tab_icms_raw_recovery", fmt="{:.3f}")
fig

#### B.2.5 Payment behaviour by raw type

**Method.** Share of CDAs with any payment by raw type.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA. Payments from GARE since enrolment.

**How to read.** Horizontal bars; the horizontal axis gives the percentage of CDAs with at least one payment, printed at the end of each bar. The table adds time to first payment and the distribution of payments by age window.

**Interpretation.** Differences in the share paying reflect differences in payment propensity between types.

*Output:* Figure `fig_icms_raw_payment`; Table `tab_icms_raw_payment`.

In [ ]:
first_pay = collect(scan_gare().group_by("ID_DEBITO").agg(pl.col("DATA_ARRECADACAO_GARE").min().alias("first_payment"),
                                                          pl.col("VALOR_RECEITA").sum().alias("paid_total")))
cda = collect(attributes).join(first_pay, on="ID_DEBITO", how="left").with_columns(
    ((pl.col("first_payment") - pl.col("DATA_INSCRICAO")).dt.total_days() / 365.25).alias("years_to_first_payment"),
    pl.col("first_payment").is_not_null().alias("ever_paid"))
pay_raw = cda.group_by("TIPO_DEBITO").agg(
    (100 * pl.col("ever_paid").mean()).alias("pct_ever_paid"),
    pl.col("years_to_first_payment").median().alias("median_years_to_first_payment"),
    (pl.col("years_to_first_payment") < 1).mean().alias("share_first_payment_year0")).sort("pct_ever_paid", descending=True)
fig = bars(pay_raw, "TIPO_DEBITO", "pct_ever_paid", val_title="CDAs with any payment (%)")
save_fig(fig, "fig_icms_raw_payment", width="double")
save_tab(pay_raw, "tab_icms_raw_payment", fmt="{:.3f}")
fig

#### B.2.6 Judicial filing by raw type

**Method.** Share of CDAs and of stock filed in court by raw type.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** One row per raw type with both shares.

**Interpretation.** A share of stock above the share of CDAs indicates that filing concentrates on larger debts.

*Output:* Table `tab_icms_raw_filing`.

In [ ]:
filing_raw = raw_values.group_by("TIPO_DEBITO").agg(
    (100 * (pl.col("STATUS_AJUIZAMENTO_DEBITO") == "Ajuizado").mean()).alias("pct_cda_filed"),
    (100 * pl.col("value").filter(pl.col("STATUS_AJUIZAMENTO_DEBITO") == "Ajuizado").sum() / pl.col("value").sum()).alias("pct_stock_filed"))
save_tab(filing_raw, "tab_icms_raw_filing", fmt="{:.2f}")
filing_raw

#### B.2.7 Seasonality by raw type

**Method.** Seasonal profile of enrolments by raw type, normalised by each type's monthly mean so that types of very different size can be compared.

**Data.** ICMS extract (1.3), enrolments from 2016 onward; unit of observation: the type-calendar month.

**How to read.** The horizontal axis gives the calendar month; the vertical axis gives enrolments relative to the type's mean (1 = average month); one line per raw type, identified in the legend.

**Interpretation.** Values above 1 mark months in which enrolments of that type concentrate; a single large spike for a small type may reflect one batch rather than a seasonal pattern.

*Output:* Figure `fig_icms_raw_seasonality`.

In [ ]:
season_raw = (cda.filter(pl.col("DATA_INSCRICAO").dt.year() >= YEARS[0])
              .with_columns(pl.col("DATA_INSCRICAO").dt.year().alias("y"), pl.col("DATA_INSCRICAO").dt.month().alias("month"))
              .group_by("TIPO_DEBITO", "y", "month").agg(pl.len().alias("n"))
              .group_by("TIPO_DEBITO", "month").agg(pl.col("n").mean())
              .with_columns((pl.col("n") / pl.col("n").mean().over("TIPO_DEBITO")).alias("index")))
fig = lines(season_raw, "month", "index", "TIPO_DEBITO", y_title="Enrolments relative to type mean", x_title="Calendar month")
fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)))
save_fig(fig, "fig_icms_raw_seasonality", width="double")
fig

#### B.2.8 Tests of distributional homogeneity

**Method.** Rank-based tests of differences in value, age and recovery rate across raw types. The Kruskal–Wallis test is a nonparametric analogue of one-way analysis of variance that does not assume normality and is robust to heavy tails. It is followed by pairwise Mann–Whitney *U* tests and Dunn's test, with Benjamini–Hochberg adjustment. Effect sizes are epsilon-squared (global) and the rank-biserial correlation (pairwise). Whole distributions are compared with two-sample Kolmogorov–Smirnov and Anderson–Darling tests. Types with fewer than 1,000 CDAs are compared with exact or permutation tests.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA. Recovery rates at the CDA level.

**How to read.** The first table gives the global test for each variable; the second the pairwise tests with adjusted *p*-values and effect sizes; the third the distributional tests.

**Interpretation.** With samples of this size, negligible differences become significant; decisions therefore rely on effect sizes. As a guide, rank-biserial values below about 0.1 in absolute value indicate negligible differences and values above about 0.3 substantial ones.

*Output:* Tables `tab_icms_raw_kruskal`, `tab_icms_raw_pairwise`, `tab_icms_raw_ks`.

In [ ]:
RATE_CDA_SQL = """
SELECT ID_DEBITO, TIPO_DEBITO, SUM(paid) / AVG(stock) AS rate_cda
FROM read_parquet('{src}')
WHERE stock > 0 AND ID_DEBITO % {n} = {k}
GROUP BY ID_DEBITO, TIPO_DEBITO
"""
RATE_CDA = {f"b{k}": RATE_CDA_SQL.format(src=parts_glob("1.4:panel", PANEL, EXTRACT_PARAMS), n=BUCKETS, k=k)
            for k in range(BUCKETS)}
rate_cda = collect(cached_sql_parts("B.2.8:rate_cda", RATE_CDA, params=EXTRACT_PARAMS))

def homogeneity_tests() -> dict:
    frames = {"value": raw_values.select("TIPO_DEBITO", "value"), "age": raw_values.select("TIPO_DEBITO", "age"),
              "rate": rate_cda.select("TIPO_DEBITO", pl.col("rate_cda").alias("rate"))}
    kw, pw, dunn = [], [], []
    for var, f in frames.items():
        kw.append(kruskal_test(f, var, "TIPO_DEBITO"))
        pw.append(pairwise_tests(f, var, "TIPO_DEBITO"))
        sample = f.group_by("TIPO_DEBITO").agg(pl.col(var).sample(n=pl.min_horizontal(pl.len(), 100_000), seed=SEED)).explode(var)
        dunn.append(dunn_test(sample, var, "TIPO_DEBITO"))
    pairwise = pl.concat(pw).join(pl.concat(dunn).select("variable", "group_a", "group_b", pl.col("z").alias("dunn_z"),
                                                         pl.col("p_bh").alias("dunn_p_bh")),
                                  on=["variable", "group_a", "group_b"], how="left")
    return {"kruskal": kw, "pairwise": pairwise.to_dicts()}

homog = cached_json("B.2.8", homogeneity_tests, params={"snapshot": EXTRACT})
kruskal_tab = pl.DataFrame(homog["kruskal"])
pairwise_tab = pl.DataFrame(homog["pairwise"])
save_tab(kruskal_tab, "tab_icms_raw_kruskal", fmt="{:.4g}")
save_tab(pairwise_tab.select("variable", "group_a", "group_b", "rank_biserial", "p_value_bh", "dunn_p_bh"),
         "tab_icms_raw_pairwise", fmt="{:.4g}")
save_tab(pairwise_tab.select("variable", "group_a", "group_b", "ks_stat", "ks_p_bh", "ad_stat", "ad_p_bh"),
         "tab_icms_raw_ks", fmt="{:.4g}")
pairwise_tab

#### B.2.9 Tests of independence for payment and filing status

**Method.** Tests of independence between raw type and (i) paid versus open status and (ii) filed versus not filed: Pearson's chi-square test, or Fisher's exact test where expected counts are below five. Cramér's *V* (0 = independence, 1 = perfect association) measures strength, and standardised residuals locate the cells that drive the association.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** One row per test, with statistic, *p*-value, Cramér's *V* and the largest residuals.

**Interpretation.** Residuals above about 2 in absolute value mark cells clearly above or below the count expected under independence.

*Output:* Table `tab_icms_raw_chi2`.

In [ ]:
def independence_tests() -> list[dict]:
    ever = cda.select("ID_DEBITO", "ever_paid")
    d = raw_values.join(ever, on="ID_DEBITO", how="left").with_columns(pl.col("ever_paid").fill_null(False),
                                                                     (pl.col("STATUS_AJUIZAMENTO_DEBITO") == "Ajuizado").alias("filed"))
    out = []
    for status in ("ever_paid", "filed"):
        ct_df = d.group_by("TIPO_DEBITO", status).len().pivot(on=status, index="TIPO_DEBITO", values="len").fill_null(0).sort("TIPO_DEBITO")
        ct = ct_df.drop("TIPO_DEBITO").to_numpy()
        res = chi2_table(ct)
        expected = stats.contingency.expected_freq(ct)
        resid = (ct - expected) / np.sqrt(expected)
        out.append({"status": status, **res,
                    "max_abs_std_residual": float(np.abs(resid).max()),
                    "type_with_max_residual": ct_df["TIPO_DEBITO"][int(np.abs(resid).max(axis=1).argmax())]})
    return out

chi2_tab = pl.DataFrame(cached_json("B.2.9", independence_tests, params={"snapshot": EXTRACT}))
save_tab(chi2_tab, "tab_icms_raw_chi2", fmt="{:.4g}")
chi2_tab

#### B.2.10 Seasonal profiles

**Method.** Tests of equality of seasonal enrolment profiles across raw types: a chi-square test on the type × calendar-month table of counts, and a regression of the monthly series on type, month and their interaction, with a joint test of the interaction terms.

**Data.** ICMS extract (1.3), enrolments from 2016 onward.

**How to read.** One row per test, with statistic and *p*-value.

**Interpretation.** Rejection means that at least one type has a different seasonal profile; the figure in B.2.7 shows which.

*Output:* Table `tab_icms_raw_season_test`.

In [ ]:
import statsmodels.formula.api as smf

def season_tests() -> list[dict]:
    d = (cda.filter(pl.col("DATA_INSCRICAO").dt.year() >= YEARS[0])
         .with_columns(pl.col("DATA_INSCRICAO").dt.month().alias("month"), pl.col("DATA_INSCRICAO").dt.year().alias("y")))
    ct = d.group_by("TIPO_DEBITO", "month").len().pivot(on="month", index="TIPO_DEBITO", values="len").fill_null(0)
    res = chi2_table(ct.drop("TIPO_DEBITO").to_numpy())
    m = d.group_by("TIPO_DEBITO", "y", "month").len().with_columns(pl.col("len").log1p().alias("log_n")).to_pandas()
    full = smf.ols("log_n ~ C(TIPO_DEBITO) * C(month) + C(y)", data=m).fit()
    restricted = smf.ols("log_n ~ C(TIPO_DEBITO) + C(month) + C(y)", data=m).fit()
    f, p, df_diff = full.compare_f_test(restricted)
    return [{"test": "chi-square type x month", "statistic": res["chi2"], "df": res["df"], "p_value": res["p_value"],
             "effect": res["cramers_v"]},
            {"test": "F test type x month interaction", "statistic": float(f), "df": int(df_diff), "p_value": float(p),
             "effect": float(full.rsquared - restricted.rsquared)}]

season_tab = pl.DataFrame(cached_json("B.2.10", season_tests, params={"snapshot": EXTRACT}))
save_tab(season_tab, "tab_icms_raw_season_test", fmt="{:.4g}")
season_tab

#### B.2.11 Grouping decision

**Method.** Decision on the grouping of raw types into origins: *declared* (ICMS Declarado), *assessed* (ICMS Autuação and ICMS Autuação Simples Nacional) and *other* (ICMS Fecoep and ICMS Alteração de GIA). Assessed types are pooled only if B.2.8 to B.2.10 show small effect sizes between them; the *other* group is kept in totals but not interpreted on its own.

**Data.** Results of B.2.1 to B.2.10.

**How to read.** One row per criterion, with the test result and the decision.

**Interpretation.** The grouping is used in all later analyses by origin.

*Output:* Table `tab_icms_grouping_decision`.

In [ ]:
def effect(var: str, a: str, b: str) -> float:
    r = pairwise_tab.filter((pl.col("variable") == var) & (((pl.col("group_a") == a) & (pl.col("group_b") == b)) |
                                                          ((pl.col("group_a") == b) & (pl.col("group_b") == a))))
    return float(r["rank_biserial"].abs()[0]) if r.height else float("nan")

A, S, D = "ICMS Autuação", "ICMS Autuação Simples Nacional", "ICMS Declarado"
SMALL = 0.1
rows = []
for var in ("value", "age", "rate"):
    e_as, e_ad = effect(var, A, S), effect(var, A, D)
    rows.append({"criterion": f"{var}: assessed types (|r_rb|)", "result": f"{e_as:.3f}",
                 "decision": "pool" if e_as < SMALL else "pool with caution"})
    rows.append({"criterion": f"{var}: assessed vs declared (|r_rb|)", "result": f"{e_ad:.3f}",
                 "decision": "keep separate" if e_ad >= SMALL else "similar"})
for r in chi2_tab.iter_rows(named=True):
    rows.append({"criterion": f"{r['status']} by type (Cramér's V)", "result": f"{r['cramers_v']:.3f}",
                 "decision": "types differ" if r["cramers_v"] >= SMALL else "small association"})
grouping = pl.DataFrame(rows)
ORIGIN_MAP = {D: "declared", A: "assessed", S: "assessed", "ICMS Fecoep": "other", "ICMS Alteração de GIA": "other"}
save_tab(grouping, "tab_icms_grouping_decision")
grouping

### B.3 Declared versus assessed ICMS

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### B.3.1 Stock, count and value per CDA by origin

**Method.** Annual aggregates by origin.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** The horizontal axis gives the year; the vertical axis gives year-end stock in R$ billion on a logarithmic scale; one line per origin, identified in the legend. The table adds counts and value per CDA.

**Interpretation.** Parallel lines indicate a stable split of the stock between origins.

*Output:* Figure `fig_icms_type_stock`; Table `tab_icms_type_summary`.

In [ ]:
origin_year = collect(panel.group_by("year", "ICMS_ORIGIN").agg(
    pl.len().alias("n_cda"), (pl.col("stock").sum() / 1e9).alias("stock_bn"), (pl.col("paid").sum() / 1e9).alias("paid_bn"),
    pl.col("stock").mean().alias("mean_value"), pl.col("paid_flag").mean().alias("share_paying"))).sort("ICMS_ORIGIN", "year")
fig = lines(origin_year, "year", "stock_bn", "ICMS_ORIGIN", log_y=True, y_title="Year-end stock (R$ bn, log scale)")
save_fig(fig, "fig_icms_type_stock")
save_tab(origin_year, "tab_icms_type_summary", fmt="{:,.3f}")
fig

#### B.3.2 Recovery by origin

**Method.** Annual recovery rate by origin, with a Mann–Whitney comparison of CDA-level rates between declared and assessed debts.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** The horizontal axis gives the year; the vertical axis gives the rate in percent on a logarithmic scale; one line per origin, identified in the legend.

**Interpretation.** A persistent gap between declared and assessed debts indicates that origin is associated with recovery and should be controlled for.

*Output:* Figure `fig_recovery_by_icms_type`.

In [ ]:
origin_rate = origin_year.with_columns((100 * pl.col("paid_bn") / pl.col("stock_bn")).alias("rate_pct"))
rates_origin = rate_cda.join(cda.select("ID_DEBITO", "ICMS_ORIGIN"), on="ID_DEBITO")
mw_origin = mann_whitney(rates_origin.filter(pl.col("ICMS_ORIGIN") == "declared")["rate_cda"].to_numpy().astype(float),
                         rates_origin.filter(pl.col("ICMS_ORIGIN") == "assessed")["rate_cda"].to_numpy().astype(float))
fig = lines(origin_rate.filter(pl.col("rate_pct") > 0), "year", "rate_pct", "ICMS_ORIGIN", log_y=True,
            y_title="Recovery rate (% of stock, log scale)")
save_fig(fig, "fig_recovery_by_icms_type")
print(mw_origin)
fig

#### B.3.3 Reporting convention

From this point on, results are reported for the pooled ICMS portfolio and, where sample sizes allow, separately for declared and assessed debts.

### B.4 Concentration

*Terms introduced in this section:* Lorenz curve, Gini coefficient. Definitions are given in the Glossary.

#### B.4.1 Concentration of the stock across CDAs

**Method.** Concentration curve of the stock across CDAs: CDAs are ranked from the largest to the smallest and the curve gives the share of the stock held by the largest *x* percent. The Gini coefficient summarises inequality (0 = equal values, 1 = the whole stock in one CDA).

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** The horizontal axis gives the share of CDAs, ranked by value, in percent on a logarithmic scale, which spreads out the upper tail; the vertical axis gives the cumulative share of the stock; labelled points mark the top 0.1%, 1% and 10%.

**Interpretation.** A curve that rises steeply at the left means that a very small share of CDAs holds most of the stock; the table adds the Gini and the number of CDAs needed to reach half of the stock.

*Output:* Figure `fig_pareto_icms`; Table `tab_concentration`.

In [ ]:
values_now = raw_values["value"].drop_nulls().to_numpy()
share, cum, gini = lorenz(values_now)
idx = np.unique(np.geomspace(1, share.size, 400).astype(int)) - 1
fig = go.Figure(go.Scatter(x=share[idx] * 100, y=cum[idx] * 100, mode="lines", line_color=CEMEPI["deep_teal"]))
for f in (0.1, 1, 10):
    k = int(f / 100 * share.size)
    fig.add_annotation(x=np.log10(f), y=cum[k] * 100, text=f"top {f:g}%: {cum[k] * 100:.0f}%", showarrow=True, arrowhead=0, ax=30, ay=20)
fig.update_xaxes(type="log", title_text="Share of CDAs, ranked by value (%, log scale)")
fig.update_yaxes(title_text="Cumulative share of the stock (%)")
save_fig(fig, "fig_pareto_icms")
k50 = int(np.searchsorted(cum, 0.5)) + 1
concentration = pl.DataFrame([{"level": "CDA", "n": int(values_now.size), "gini": gini, **top_shares(values_now),
                               "n_for_50pct": k50, "pct_for_50pct": 100 * k50 / values_now.size}])
save_tab(concentration, "tab_concentration", fmt="{:,.4f}")
fig

#### B.4.2 Concentration across debtors

**Method.** The same concentration curve with debts aggregated by debtor (CNPJ root).

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA. Aggregated by CNPJ root.

**How to read.** Axes as in B.4.1; one line for CDAs and one for debtors, identified in the legend.

**Interpretation.** If the debtor curve lies to the left of the CDA curve at the top, a smaller share of firms than of CDAs holds the same share of the stock; the reverse means large debts are spread across many firms.

*Output:* Figure `fig_pareto_debtor`.

In [ ]:
debtor_values = raw_values.group_by("CNPJ_ROOT").agg(pl.col("value").sum())["value"].to_numpy()
share_d, cum_d, gini_d = lorenz(debtor_values)
idx_d = np.unique(np.geomspace(1, share_d.size, 400).astype(int)) - 1
fig = go.Figure([go.Scatter(x=share[idx] * 100, y=cum[idx] * 100, mode="lines", line=dict(color=CEMEPI["deep_teal"]), name="CDA"),
                 go.Scatter(x=share_d[idx_d] * 100, y=cum_d[idx_d] * 100, mode="lines", line=dict(color=CEMEPI["sand"], dash="dash"), name="Debtor (CNPJ root)")])
fig.update_layout(showlegend=True, legend=dict(orientation="h", y=-0.3))
fig.update_xaxes(type="log", title_text="Share of units, ranked by value (%, log scale)")
fig.update_yaxes(title_text="Cumulative share of the stock (%)")
concentration = pl.concat([concentration, pl.DataFrame([{"level": "Debtor", "n": int(debtor_values.size), "gini": gini_d,
                                                         **top_shares(debtor_values),
                                                         "n_for_50pct": int(np.searchsorted(cum_d, 0.5)) + 1,
                                                         "pct_for_50pct": 100 * (int(np.searchsorted(cum_d, 0.5)) + 1) / debtor_values.size}])])
save_fig(fig, "fig_pareto_debtor")
save_tab(concentration, "tab_concentration", fmt="{:,.4f}")
fig

#### B.4.3 Largest debts versus the tail

**Method.** Comparison of the largest debts with the remainder.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA. Payment history from GARE.

**How to read.** Rows give the top 0.1%, top 1% and remainder; columns give age, recovery and payment measures.

**Interpretation.** Systematic differences justify the separate study of the top set in B.5.

*Output:* Table `tab_large_vs_tail`.

In [ ]:
thr = {f: np.quantile(values_now, 1 - f) for f in (0.001, 0.01)}
strata = raw_values.join(cda.select("ID_DEBITO", "ever_paid"), on="ID_DEBITO", how="left").with_columns(
    pl.when(pl.col("value") >= thr[0.001]).then(pl.lit("top 0.1%"))
      .when(pl.col("value") >= thr[0.01]).then(pl.lit("top 0.1-1%")).otherwise(pl.lit("remainder")).alias("stratum"))
large_vs_tail = strata.group_by("stratum").agg(
    pl.len().alias("n_cda"), (pl.col("value").sum() / 1e9).alias("stock_bn"), pl.col("age").median().alias("median_age"),
    (100 * pl.col("ever_paid").fill_null(False).mean()).alias("pct_ever_paid"),
    (100 * (pl.col("STATUS_AJUIZAMENTO_DEBITO") == "Ajuizado").mean()).alias("pct_filed"),
    (100 * (pl.col("ICMS_ORIGIN") == "assessed").mean()).alias("pct_assessed")).sort("stock_bn", descending=True)
save_tab(large_vs_tail, "tab_large_vs_tail", fmt="{:,.2f}")
large_vs_tail

### B.5 The most concentrated CDAs
Because a very small fraction of CDAs holds a large share of the stock (B.4), this group is examined in depth: its definition and sensitivity, its characteristics and behaviour, the structure of the debtors behind it, the similarity among its members and its contrast with the rest of the portfolio. Debtor names are never used; CNPJ roots appear in outputs only as salted hashes.

*Terms introduced in this section:* Top 1% (top set), Jaccard index, accumulated recovery, Kaplan–Meier estimator, censoring, log-rank test, intraclass correlation, administrative situation, Gower distance, principal component analysis, t-SNE, agglomerative clustering, HDBSCAN, silhouette coefficient, adjusted Rand index, medoid, dynamic time warping. Definitions are given in the Glossary.

#### B.5.1 Definition of the top set

**Method.** Definition of four alternative top sets: the top 0.1% and top 1% of CDAs by value, and the smallest sets of CDAs that together hold 50% and 80% of the stock. The top 1% is the reference set.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** One row per set, with its size, share of the stock and value threshold.

**Interpretation.** Thresholds show how large a debt must be to enter each set.

*Output:* Table `tab_top_definitions`.

In [ ]:
ranked = raw_values.sort("value", descending=True).with_columns(
    (pl.col("value").cum_sum() / pl.col("value").sum()).alias("cum_share"), pl.int_range(1, pl.len() + 1).alias("rank"))
N = ranked.height
TOP_SETS = {"top_0.1pct": ranked["rank"] <= max(1, round(0.001 * N)),
            "top_1pct": ranked["rank"] <= max(1, round(0.01 * N)),
            "stock_50pct": ranked["cum_share"].shift(1, fill_value=0) < 0.5,
            "stock_80pct": ranked["cum_share"].shift(1, fill_value=0) < 0.8}
top_definitions = pl.DataFrame([{"set": k, "n_cda": int(m.sum()), "pct_cda": 100 * m.sum() / N,
                                 "pct_stock": 100 * ranked.filter(m)["value"].sum() / ranked["value"].sum(),
                                 "min_value": float(ranked.filter(m)["value"].min())} for k, m in TOP_SETS.items()])
top = ranked.filter(TOP_SETS["top_1pct"])
top_ids = set(top["ID_DEBITO"].to_list())
save_tab(top_definitions, "tab_top_definitions", fmt="{:,.3f}")
top_definitions

#### B.5.2 Sensitivity of the top set

**Method.** Sensitivity of the top set. The Jaccard index (intersection over union, 0 to 1) measures overlap between alternative sets; membership stability is the share of last year's top 1% that remains in the top 1%.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA. Year-end snapshots 2016–2026 for stability.

**How to read.** Heat map: rows and columns are the four sets, colour and printed value give the Jaccard index, with the colour bar from 0 to 1. Line chart: the horizontal axis gives the year and the vertical axis the percentage retained from the previous year.

**Interpretation.** Values near 1 on the heat map mean two definitions select the same CDAs; high retention means that results for the top set do not depend on the snapshot chosen, while a sharp drop in one year points to a change in the portfolio rather than gradual turnover.

*Output:* Figures `fig_top_overlap`, `fig_top_membership_stability`; Table `tab_top_overlap`.

In [ ]:
sets = {k: set(ranked.filter(m)["ID_DEBITO"].to_list()) for k, m in TOP_SETS.items()}
names = list(sets)
jac = np.array([[len(sets[a] & sets[b]) / len(sets[a] | sets[b]) for b in names] for a in names])
SET_NAMES = {"top_0.1pct": "Top 0.1%", "top_1pct": "Top 1%", "stock_50pct": "50% of stock", "stock_80pct": "80% of stock"}
overlap = pl.DataFrame({"set": [SET_NAMES.get(a, a) for a in names], **{SET_NAMES.get(b, b): jac[:, j] for j, b in enumerate(names)}})
fig_overlap = heatmap(overlap, "set", diverging=False, zmin=0, zmax=1, colorbar_title="Jaccard")
save_fig(fig_overlap, "fig_top_overlap")
fig_overlap.show()

ye_top = collect(year_end.select("year", "ID_DEBITO", "stock"))
members = {}
for (y,), d in ye_top.group_by("year"):
    k = max(1, round(0.01 * d.height))
    members[y] = set(d.sort("stock", descending=True).head(k)["ID_DEBITO"].to_list())
stability = pl.DataFrame([{"year": y, "retention": len(members[y] & members[y - 1]) / len(members[y - 1])}
                          for y in sorted(members) if y - 1 in members])
fig = lines(stability.with_columns(pl.col("retention") * 100), "year", "retention", y_title="Top 1% retained from previous year (%)")
save_fig(fig, "fig_top_membership_stability")
save_tab(overlap, "tab_top_overlap", fmt="{:.3f}")
fig

#### Characteristics

#### B.5.3 Composition by type and origin

**Method.** Composition of the top set by raw type compared with the portfolio.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Horizontal grouped bars; the horizontal axis gives the share of CDAs in percent; bars for the top 1% and for the whole portfolio are identified in the legend.

**Interpretation.** A raw type whose top-set bar exceeds its portfolio bar is over-represented among the largest debts.

*Output:* Figure `fig_top_composition`; Table `tab_top_composition`.

In [ ]:
def composition_share(frame: pl.DataFrame, label: str) -> pl.DataFrame:
    return frame.group_by("TIPO_DEBITO").agg(pl.len().alias("n"), pl.col("value").sum().alias("v")).with_columns(
        (100 * pl.col("n") / pl.col("n").sum()).alias("pct_cda"), (100 * pl.col("v") / pl.col("v").sum()).alias("pct_stock"),
        pl.lit(label).alias("group")).drop("n", "v")

top_comp = pl.concat([composition_share(top, "Top 1%"), composition_share(raw_values, "Portfolio")])
fig = go.Figure([go.Bar(y=d["TIPO_DEBITO"], x=d["pct_cda"], orientation="h", name=k[0], marker_color=color(i))
                 for i, (k, d) in enumerate(top_comp.group_by("group", maintain_order=True))])
fig.update_layout(barmode="group", xaxis_title="Share of CDAs (%)", showlegend=True, legend=dict(orientation="h", y=1.1))
save_fig(fig, "fig_top_composition", width="double")
save_tab(top_comp, "tab_top_composition")
fig

#### B.5.4 Age and year of enrolment

**Method.** Distribution of age and year of enrolment in the top set.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Left histogram: age in years; right histogram: year of enrolment; vertical axes give numbers of CDAs.

**Interpretation.** A concentration at old ages would indicate that large debts accumulate because they remain unpaid.

*Output:* Figure `fig_top_age_enrolment`.

In [ ]:
top_enrol = top.with_columns(pl.col("ID_DEBITO")).join(cda.select("ID_DEBITO", pl.col("DATA_INSCRICAO").dt.year().alias("enrol_year")), on="ID_DEBITO")
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
fig.add_trace(go.Histogram(x=top["age"], nbinsx=40, marker_color=CEMEPI["deep_teal"]), 1, 1)
counts = top_enrol.group_by("enrol_year").len().sort("enrol_year")
fig.add_trace(go.Bar(x=counts["enrol_year"], y=counts["len"], marker_color=CEMEPI["sand"]), 1, 2)
fig.update_xaxes(title_text="Age (years)", row=1, col=1)
fig.update_xaxes(title_text="Year of enrolment", row=1, col=2)
fig.update_yaxes(title_text="CDAs", row=1, col=1)
save_fig(fig, "fig_top_age_enrolment", width="double")
fig

#### B.5.5 Value distribution within the top set

**Method.** Distribution of value within the top set and its internal concentration.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Histogram of log₁₀ CDA value; the vertical axis gives the number of CDAs; the annotation gives the Gini coefficient within the set.

**Interpretation.** A high internal Gini indicates that concentration persists even among the largest debts.

*Output:* Figure `fig_top_value_dist`.

In [ ]:
v = np.sort(top["value"].to_numpy())[::-1]
fig = go.Figure(go.Histogram(x=np.log10(v), nbinsx=50, marker_color=CEMEPI["deep_teal"]))
fig.update_layout(xaxis_title="log10 CDA value (R$)", yaxis_title="CDAs")
_, _, gini_top = lorenz(v)
fig.add_annotation(x=0.98, y=0.95, xref="paper", yref="paper", text=f"Gini within top 1% = {gini_top:.2f}", showarrow=False, xanchor="right")
save_fig(fig, "fig_top_value_dist")
fig

#### B.5.6 Administrative situation and judicial filing

**Method.** Administrative situation and filing status of the top set.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Rows give categories; columns give shares for the top set and for the remainder.

**Interpretation.** Differences indicate whether the largest debts are handled differently by the administration or the courts.

*Output:* Table `tab_top_situation_filing`.

In [ ]:
situation_filing = pl.concat([
    frame.group_by(col).agg(pl.len().alias("n")).with_columns((100 * pl.col("n") / pl.col("n").sum()).alias("pct"),
                                                          pl.lit(label).alias("group"), pl.lit(col).alias("dimension")).rename({col: "category"})
    for frame, label in ((top, "Top 1%"), (raw_values, "Portfolio"))
    for col in ("SITUACAO_DEBITO", "STATUS_AJUIZAMENTO_DEBITO")]).select("dimension", "category", "group", "n", "pct")
save_tab(situation_filing, "tab_top_situation_filing", fmt="{:,.2f}")
situation_filing

#### Behaviour

#### B.5.7 Payments and recovery

**Method.** Annual recovery of the top set compared with the remainder.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026. Remainder from the panel.

**How to read.** The horizontal axis gives the year; the vertical axis gives the non-accumulated recovery rate in percent on a logarithmic scale; one line per group, identified in the legend.

**Interpretation.** A line persistently below the other on a log axis indicates proportionally lower recovery; parallel movements indicate common shocks.

*Output:* Figure `fig_top_recovery`; Table `tab_top_payments`.

In [ ]:
panel_top = collect(panel.filter(pl.col("ID_DEBITO").is_in(list(top_ids))))
TOP_IDS = pl.Series("ID_DEBITO", sorted(top_ids), dtype=pl.Int64)
top_payments = collect(panel.select("year", "ID_DEBITO", "stock", "paid", "paid_flag")
                       .with_columns(pl.col("ID_DEBITO").is_in(TOP_IDS.implode()).alias("is_top"))
                       .group_by("year", "is_top").agg(
    (100 * pl.col("paid").sum() / pl.col("stock").sum()).alias("rate_pct"), (100 * pl.col("paid_flag").mean()).alias("pct_paying"),
    (pl.col("paid").sum() / 1e9).alias("paid_bn"))).with_columns(
    pl.when(pl.col("is_top")).then(pl.lit("Top 1%")).otherwise(pl.lit("Remainder")).alias("group")).sort("year")
fig = lines(top_payments.filter(pl.col("rate_pct") > 0), "year", "rate_pct", "group", log_y=True, y_title="Recovery rate (%, log scale)")
save_fig(fig, "fig_top_recovery")
save_tab(top_payments.drop("is_top"), "tab_top_payments", fmt="{:,.3f}")
fig

#### B.5.8 Recovery trajectories

**Method.** Accumulated recovery trajectories of individual top CDAs, with the median and upper percentiles across all top CDAs.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** The horizontal axis gives years since enrolment; the vertical axis gives the cumulative amount paid as a percentage of the enrolled value. Thin grey lines are a random sample of 150 CDAs; the thick line is the median and the dashed and dotted lines the 75th and 90th percentiles, as identified in the legend.

**Interpretation.** A median at zero means that most top CDAs have not been paid at all at that horizon, so recovery of the set is driven by a minority.

*Output:* Figure `fig_top_trajectories`.

In [ ]:
gare_top = collect(scan_gare().filter(pl.col("ID_DEBITO").is_in(list(top_ids)))
                   .group_by("ID_DEBITO", pl.col("DATA_ARRECADACAO_GARE").dt.year().alias("pay_year")).agg(pl.col("VALOR_RECEITA").sum()))
enrol_val = top.select("ID_DEBITO", "value").join(cda.select("ID_DEBITO", pl.col("DATA_INSCRICAO").dt.year().alias("enrol_year"),
                                                          pl.col("max_value")), on="ID_DEBITO")
traj = (gare_top.join(enrol_val, on="ID_DEBITO").with_columns((pl.col("pay_year") - pl.col("enrol_year")).alias("h"))
        .filter(pl.col("h").is_between(0, 10)).group_by("ID_DEBITO", "h").agg(pl.col("VALOR_RECEITA").sum(), pl.col("max_value").first()))
grid = enrol_val.select("ID_DEBITO", "max_value", "enrol_year").join(pl.DataFrame({"h": list(range(11))}), how="cross") \
    .filter(pl.col("enrol_year") + pl.col("h") <= SNAP_YEAR)
traj = (grid.join(traj.select("ID_DEBITO", "h", "VALOR_RECEITA"), on=["ID_DEBITO", "h"], how="left")
        .with_columns(pl.col("VALOR_RECEITA").fill_null(0)).sort("ID_DEBITO", "h")
        .with_columns((pl.col("VALOR_RECEITA").cum_sum().over("ID_DEBITO") / pl.col("max_value")).alias("acc_rec")))
bands = traj.group_by("h").agg(pl.col("acc_rec").quantile(0.25).alias("q25"), pl.col("acc_rec").median().alias("q50"),
                               pl.col("acc_rec").quantile(0.75).alias("q75"), pl.col("acc_rec").quantile(0.9).alias("q90")).sort("h")
fig = go.Figure()
for i, sid in enumerate(traj["ID_DEBITO"].unique().sample(n=min(150, traj["ID_DEBITO"].n_unique()), seed=SEED)):
    d = traj.filter(pl.col("ID_DEBITO") == sid)
    fig.add_trace(go.Scatter(x=d["h"], y=d["acc_rec"] * 100, mode="lines", line=dict(color="rgba(138,148,153,0.35)", width=0.7),
                             hoverinfo="skip", name="Individual CDAs (random sample of 150)", legendgroup="cda", showlegend=i == 0))
fig.add_trace(go.Scatter(x=bands["h"], y=bands["q50"] * 100, mode="lines", line=dict(color=CEMEPI["deep_teal"], width=2.5), name="Median"))
fig.add_trace(go.Scatter(x=bands["h"], y=bands["q75"] * 100, mode="lines", line=dict(color=CEMEPI["teal"], dash="dash"), name="75th percentile"))
fig.add_trace(go.Scatter(x=bands["h"], y=bands["q90"] * 100, mode="lines", line=dict(color=CEMEPI["sand"], dash="dot"), name="90th percentile"))
fig.update_layout(xaxis_title="Years since enrolment", yaxis_title="Accumulated recovery (% of enrolled value)",
                  showlegend=True, legend=dict(orientation="h", y=-0.3))
save_fig(fig, "fig_top_trajectories", width="double")
fig

#### B.5.9 Time to first payment

**Method.** Kaplan–Meier estimation of the time from enrolment to first payment. CDAs without a payment by the last snapshot are right-censored: only a lower bound on their time is known. The estimator handles censoring without assuming a distribution for durations (`SurvfuncRight`, `statsmodels`), and the log-rank test (`survdiff`) compares the two curves.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026. The remainder is represented by a random sample of 200,000 CDAs.

**How to read.** The horizontal axis gives years since enrolment; the vertical axis gives the cumulative probability of a first payment (one minus the survival function); one line per group, identified in the legend.

**Interpretation.** A higher curve means faster first payment; the curves flatten where few further first payments occur. The test asks whether the groups differ over the whole horizon, not at a specific time.

*Output:* Figure `fig_top_km_first_payment`; Table `tab_top_logrank`.

In [ ]:
from statsmodels.duration.survfunc import SurvfuncRight, survdiff

censor_date = np.datetime64(f"{SNAP_YEAR}-{SNAP_MONTH:02d}-28")
surv = cda.filter(pl.col("DATA_INSCRICAO") >= pl.date(YEARS[0], 1, 1)).with_columns(
    pl.col("ID_DEBITO").is_in(list(top_ids)).cast(pl.Int8).alias("is_top"),
    pl.coalesce(pl.col("first_payment"), pl.lit(censor_date).cast(pl.Date)).alias("_end")).with_columns(
    ((pl.col("_end") - pl.col("DATA_INSCRICAO")).dt.total_days() / 365.25).clip(lower_bound=1 / 365).alias("t"),
    pl.col("ever_paid").cast(pl.Int8).alias("event"))
rest = surv.filter(pl.col("is_top") == 0)
surv_s = pl.concat([surv.filter(pl.col("is_top") == 1), rest.sample(n=min(200_000, rest.height), seed=SEED)])
fig = go.Figure()
for i, (g, label) in enumerate(((1, "Top 1%"), (0, "Remainder"))):
    d = surv_s.filter(pl.col("is_top") == g)
    sf = SurvfuncRight(d["t"].to_numpy(), d["event"].to_numpy())
    fig.add_trace(go.Scatter(x=sf.surv_times, y=1 - sf.surv_prob, mode="lines", line_shape="hv",
                             line=dict(color=color(i), dash=DASHES[i]), name=label))
chi2, pval = survdiff(surv_s["t"].to_numpy(), surv_s["event"].to_numpy(), surv_s["is_top"].to_numpy())
fig.update_layout(xaxis_title="Years since enrolment", yaxis_title="Cumulative probability of first payment", showlegend=True,
                  legend=dict(orientation="h", yanchor="top", y=-0.22, x=0.5, xanchor="center"))
logrank = pl.DataFrame([{"test": "log-rank", "chi2": float(chi2), "p_value": float(pval),
                         "n_top": int(surv_s["is_top"].sum()), "n_remainder_sample": int((surv_s["is_top"] == 0).sum())}])
save_fig(fig, "fig_top_km_first_payment")
save_tab(logrank, "tab_top_logrank", fmt="{:.4g}")
fig

#### B.5.10 Response to settlement programmes

**Method.** Response to settlement programmes: the mean monthly amount paid during each programme window divided by the mean monthly amount paid in the twelve months before it.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026. All GARE payments, monthly.

**How to read.** Grouped bars, one pair per programme; the vertical axis gives the ratio, printed above each bar (1 = no change); groups are identified in the legend.

**Interpretation.** A higher ratio for the top set means that large debts respond proportionally more strongly to programmes.

*Output:* Figure `fig_top_programme_response`; Table `tab_top_programme_response`.

In [ ]:
gare_month = collect(scan_gare().with_columns(pl.col("DATA_ARRECADACAO_GARE").dt.truncate("1mo").alias("month"),
                                             pl.col("ID_DEBITO").is_in(list(top_ids)).alias("is_top"))
                     .group_by("month", "is_top").agg(pl.col("VALOR_RECEITA").sum().alias("paid")))
rows = []
for name, start, end in PROGRAMMES:
    s, e = np.datetime64(start), np.datetime64(end)
    for is_top in (True, False):
        d = gare_month.filter(pl.col("is_top") == is_top)
        pre = d.filter((pl.col("month") >= s - np.timedelta64(365, "D")) & (pl.col("month") < s))["paid"].mean()
        during = d.filter((pl.col("month") >= s) & (pl.col("month") <= e))["paid"].mean()
        rows.append({"programme": name, "group": "Top 1%" if is_top else "Remainder", "pre_mean_mn": pre / 1e6,
                     "during_mean_mn": during / 1e6, "ratio": during / pre if pre else float("nan")})
prog_resp = pl.DataFrame(rows)
fig = go.Figure([go.Bar(x=d["programme"], y=d["ratio"], name=k[0], marker_color=color(i),
                        text=[f"{v:.1f}x" for v in d["ratio"]], textposition="outside")
                 for i, (k, d) in enumerate(prog_resp.group_by("group", maintain_order=True))])
fig.update_layout(barmode="group", yaxis_title="Monthly payments during / 12 months before", showlegend=True, legend=dict(orientation="h", y=1.1))
save_fig(fig, "fig_top_programme_response")
save_tab(prog_resp, "tab_top_programme_response")
fig

#### B.5.11 Persistence

**Method.** Persistence of payment status in the top set.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Transition matrix: rows give status in year *t* − 1, columns status in year *t*, cells the conditional probability.

**Interpretation.** Large diagonal entries mean that paying and non-paying CDAs tend to remain so.

*Output:* Table `tab_top_transition`.

In [ ]:
trans = panel_top.sort("ID_DEBITO", "year").with_columns(pl.col("paid_flag").shift(1).over("ID_DEBITO").alias("prev"),
                                                         pl.col("year").shift(1).over("ID_DEBITO").alias("prev_year")) \
    .filter(pl.col("prev_year") == pl.col("year") - 1)
transition = trans.group_by("prev", "paid_flag").len().with_columns((pl.col("len") / pl.col("len").sum().over("prev")).alias("prob")).sort("prev", "paid_flag")
acf_top = corr_pair(panel_top["rate"].to_numpy().astype(float), panel_top["rate_l1"].to_numpy().astype(float), "spearman")
transition = transition.with_columns(pl.lit(acf_top[0]).alias("spearman_rate_lag1"))
save_tab(transition, "tab_top_transition", fmt="{:.4f}")
transition

#### Debtor structure

#### B.5.12 CDAs per debtor

**Method.** Distribution of the number of CDAs per debtor.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** Scatter on logarithmic axes: the horizontal axis gives the number of CDAs per firm, the vertical axis the number of firms with that count. The table gives shares of debtors and of stock held by multi-CDA debtors, by full CNPJ and by CNPJ root.

**Interpretation.** An approximately straight declining pattern on log axes indicates a heavy-tailed distribution: most firms have one or a few CDAs and a few firms accumulate very many.

*Output:* Figure `fig_cdas_per_debtor`; Table `tab_multi_cda_debtors`.

In [ ]:
per_est = raw_values.group_by("CNPJ_ROOT").agg(pl.len().alias("n_cda"), pl.col("value").sum())
snap_cnpj = collect(snap_icms.select("ID_DEBITO", "CNPJ"))
per_full = raw_values.join(snap_cnpj, on="ID_DEBITO").group_by("CNPJ").agg(pl.len().alias("n_cda"), pl.col("value").sum())
def multi(frame: pl.DataFrame, level: str) -> dict:
    m = frame["n_cda"] > 1
    return {"level": level, "debtors": frame.height, "pct_multi_cda": 100 * m.mean(),
            "pct_stock_multi_cda": 100 * frame.filter(m)["value"].sum() / frame["value"].sum(),
            "max_cda_per_debtor": int(frame["n_cda"].max()), "median_cda_per_debtor": float(frame["n_cda"].median())}
multi_tab = pl.DataFrame([multi(per_full, "Establishment (full CNPJ)"), multi(per_est, "Firm (CNPJ root)")])
counts = per_est.group_by("n_cda").len().sort("n_cda")
fig = go.Figure(go.Scatter(x=counts["n_cda"], y=counts["len"], mode="markers", marker_color=CEMEPI["deep_teal"]))
fig.update_xaxes(type="log", title_text="CDAs per firm (log)")
fig.update_yaxes(type="log", title_text="Firms (log)")
save_fig(fig, "fig_cdas_per_debtor")
save_tab(multi_tab, "tab_multi_cda_debtors", fmt="{:,.2f}")
fig

#### B.5.13 Concentration at the debtor level

**Method.** Lorenz curves of the top-set stock across CDAs and across their debtors. Units are ordered from smallest to largest; each curve gives the cumulative share of the stock held by the smallest *x* percent of units.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026. Aggregated by CNPJ root for the debtor curve.

**How to read.** The horizontal axis gives the cumulative share of units in percent; the vertical axis the cumulative share of the stock; the legend gives the Gini coefficient of each curve.

**Interpretation.** The further a curve lies below the diagonal, the more unequal the distribution; a debtor curve below the CDA curve means that aggregation by firm increases concentration.

*Output:* Figure `fig_debtor_lorenz`.

In [ ]:
top_roots = top["CNPJ_ROOT"].unique()
dv = per_est.filter(pl.col("CNPJ_ROOT").is_in(top_roots))["value"].to_numpy()
s1, c1, g1 = lorenz(top["value"].to_numpy())
s2, c2, g2 = lorenz(dv)
fig = go.Figure([go.Scatter(x=s1 * 100, y=c1 * 100, mode="lines", line=dict(color=CEMEPI["deep_teal"]), name=f"Top 1% CDAs (Gini {g1:.2f})"),
                 go.Scatter(x=s2 * 100, y=c2 * 100, mode="lines", line=dict(color=CEMEPI["sand"], dash="dash"),
                            name=f"Their debtors, CNPJ root (Gini {g2:.2f})")])
fig.update_layout(xaxis_title="Cumulative share of units, ranked by value (%)", yaxis_title="Cumulative share of the stock (%)",
                  showlegend=True, legend=dict(orientation="h", y=-0.3))
save_fig(fig, "fig_debtor_lorenz")
fig

#### B.5.14 Establishments within firms

**Method.** Number of establishments per firm among top debtors and distribution of the stock across them.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Rows give firms grouped by number of establishments; columns give counts and the share of stock in the largest establishment.

**Interpretation.** A high share in the largest establishment means that the firm's debt is concentrated in one unit.

*Output:* Table `tab_establishments_per_root`.

In [ ]:
est_top = raw_values.join(snap_cnpj, on="ID_DEBITO").filter(pl.col("CNPJ_ROOT").is_in(top_roots)) \
    .group_by("CNPJ_ROOT", "CNPJ").agg(pl.col("value").sum()) \
    .group_by("CNPJ_ROOT").agg(pl.len().alias("establishments"), (pl.col("value").max() / pl.col("value").sum()).alias("share_largest"))
establishments = est_top.select(
    pl.len().alias("top_firms"), pl.col("establishments").mean().alias("mean_establishments"),
    pl.col("establishments").median().alias("median_establishments"), pl.col("establishments").max().alias("max_establishments"),
    (100 * (pl.col("establishments") > 1).mean()).alias("pct_multi_establishment"),
    pl.col("share_largest").median().alias("median_share_largest_establishment"))
save_tab(establishments, "tab_establishments_per_root", fmt="{:,.2f}")
establishments

#### B.5.15 Within-debtor homogeneity

**Method.** Within-debtor homogeneity: coefficient of variation of value and age, Shannon entropy of raw types and share of CDAs paid within each multi-CDA debtor, and the intraclass correlation of the recovery rate from a linear mixed model with a random intercept per debtor (`MixedLM`).

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026. Debtors with more than one CDA.

**How to read.** One row per measure, with its distribution across debtors.

**Interpretation.** An intraclass correlation near 1 means that CDAs of the same firm behave alike, so that the firm rather than the individual CDA is the natural unit for collection decisions.

*Output:* Table `tab_within_debtor_homogeneity`.

In [ ]:
from statsmodels.regression.mixed_linear_model import MixedLM

within = raw_values.filter(pl.col("CNPJ_ROOT").is_in(top_roots)).join(cda.select("ID_DEBITO", "ever_paid"), on="ID_DEBITO", how="left")
def entropy(s: pl.Series) -> float:
    p = s.value_counts()["count"].to_numpy() / s.len()
    return float(-(p * np.log(p)).sum())
homog_rows = []
for (root,), d in within.group_by("CNPJ_ROOT"):
    if d.height < 2:
        continue
    homog_rows.append({"cv_value": float(d["value"].std() / d["value"].mean()), "sd_age": float(d["age"].std()),
                       "entropy_type": entropy(d["TIPO_DEBITO"]), "share_paid": float(d["ever_paid"].fill_null(False).mean())})
h = pl.DataFrame(homog_rows)
icc_data = rate_cda.join(raw_values.select("ID_DEBITO", "CNPJ_ROOT"), on="ID_DEBITO").filter(pl.col("CNPJ_ROOT").is_in(top_roots)) \
    .filter(pl.col("CNPJ_ROOT").count().over("CNPJ_ROOT") > 1).with_columns(pl.col("rate_cda").clip(0, 1)).to_pandas()
mixed = MixedLM.from_formula("rate_cda ~ 1", groups="CNPJ_ROOT", data=icc_data).fit()
icc = float(mixed.cov_re.iloc[0, 0] / (mixed.cov_re.iloc[0, 0] + mixed.scale))
within_tab = h.select([pl.col(c).median().alias(f"median_{c}") for c in h.columns]).with_columns(
    pl.lit(h.height).alias("multi_cda_firms"), pl.lit(icc).alias("icc_recovery_rate"))
save_tab(within_tab, "tab_within_debtor_homogeneity", fmt="{:.3f}")
within_tab

#### B.5.16 Repeat enrolment

**Method.** Timing of repeat enrolments against the same top debtor.

**Data.** ICMS extract (1.3), CDAs of the firms in the top set, 2016–2026.

**How to read.** Left: histogram of the interval in years between successive enrolments (up to 15 years); right: bars of new CDAs per top firm by year of enrolment.

**Interpretation.** Mass near zero years indicates batch enrolment; a rising right panel indicates growing repeat delinquency.

*Output:* Figure `fig_repeat_enrolment`.

In [ ]:
repeat = cda.filter(pl.col("CNPJ_ROOT").is_in(top_roots)).sort("CNPJ_ROOT", "DATA_INSCRICAO").with_columns(
    ((pl.col("DATA_INSCRICAO") - pl.col("DATA_INSCRICAO").shift(1).over("CNPJ_ROOT")).dt.total_days() / 365.25).alias("gap_years"))
gaps = repeat["gap_years"].drop_nulls().to_numpy()
per_year = repeat.filter(pl.col("DATA_INSCRICAO").dt.year() >= YEARS[0]).group_by(pl.col("DATA_INSCRICAO").dt.year().alias("year")) \
    .agg(pl.len().alias("new_cda"), pl.col("CNPJ_ROOT").n_unique().alias("firms")).sort("year")
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
fig.add_trace(go.Histogram(x=gaps[gaps <= 15], nbinsx=60, marker_color=CEMEPI["deep_teal"]), 1, 1)
fig.add_trace(go.Bar(x=per_year["year"], y=per_year["new_cda"] / per_year["firms"], marker_color=CEMEPI["sand"]), 1, 2)
fig.update_xaxes(title_text="Years between successive enrolments", row=1, col=1)
fig.update_yaxes(title_text="Pairs of CDAs", row=1, col=1)
fig.update_xaxes(title_text="Year of enrolment", row=1, col=2)
fig.update_yaxes(title_text="New CDAs per top firm", row=1, col=2)
save_fig(fig, "fig_repeat_enrolment", width="double")
fig

#### Similarity and archetypes

#### B.5.17 Feature vectors and memory budget

**Method.** Construction of feature vectors and of a memory-bounded distance computation. Numerical features (log value, age, recovery rate, time to first payment, number of payments, programme response) are scaled by median and interquartile range; categorical features are origin, administrative situation and filing status. Distance matrices are computed in row blocks and written as single-precision memory-mapped files on the cache drive, so that only one block resides in memory at a time.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** One row per feature with summary statistics.

**Interpretation.** Robust scaling prevents a few extreme values from dominating the distances.

*Output:* Table `tab_feature_summary`.

In [ ]:
prog_top = collect(scan_gare().filter(pl.col("ID_DEBITO").is_in(list(top_ids)))
                   .with_columns(pl.any_horizontal([pl.col("DATA_ARRECADACAO_GARE").is_between(pl.lit(a).str.to_date(), pl.lit(b).str.to_date())
                                                    for _, a, b in PROGRAMMES]).alias("in_programme"))
                   .group_by("ID_DEBITO").agg(pl.len().alias("n_payments"), pl.col("VALOR_RECEITA").filter(pl.col("in_programme")).sum().alias("paid_programme"),
                                               pl.col("VALOR_RECEITA").sum().alias("paid_total")))
features = (top.select("ID_DEBITO", "CNPJ_ROOT", "value", "age", "ICMS_ORIGIN", "SITUACAO_DEBITO", "STATUS_AJUIZAMENTO_DEBITO")
            .join(cda.select("ID_DEBITO", "years_to_first_payment"), on="ID_DEBITO", how="left")
            .join(prog_top, on="ID_DEBITO", how="left")
            .with_columns(pl.col("n_payments").fill_null(0), pl.col("paid_total").fill_null(0), pl.col("paid_programme").fill_null(0))
            .with_columns(pl.col("value").log10().alias("log_value"), (pl.col("paid_total") / pl.col("value")).alias("recovery"),
                          (pl.col("paid_programme") / pl.col("paid_total")).fill_nan(0).fill_null(0).alias("programme_share"),
                          pl.col("years_to_first_payment").fill_null(pl.col("age")).alias("time_to_payment"))
            .sort("ID_DEBITO"))
NUM = ["log_value", "age", "recovery", "time_to_payment", "n_payments", "programme_share"]
CAT = ["ICMS_ORIGIN", "SITUACAO_DEBITO", "STATUS_AJUIZAMENTO_DEBITO"]
X_raw = features.select(NUM).to_numpy().astype(np.float32)
med, iqr = np.nanmedian(X_raw, axis=0), np.subtract(*np.nanpercentile(X_raw, [75, 25], axis=0))
X_num = np.nan_to_num((X_raw - med) / np.where(iqr == 0, 1, iqr)).astype(np.float32)
X_cat = np.column_stack([features[c].cast(pl.Categorical).to_physical().to_numpy() for c in CAT]).astype(np.int32)
feature_summary = features.select(NUM).describe()
save_tab(feature_summary, "tab_feature_summary", fmt="{:,.3f}")
print(f"{X_num.shape[0]:,} CDAs; full distance matrix = {X_num.shape[0] ** 2 * 4 / 1e9:.2f} GB (float32)")
feature_summary

#### B.5.18 Distance structure

**Method.** Gower distance and low-dimensional views. Gower distance averages, over features, the range-normalised absolute difference for numerical features and a 0/1 mismatch for categorical ones, giving a dissimilarity between 0 and 1. The structure is shown by principal components of the numerical features and by t-SNE computed on the Gower distances of a random subsample of 5,000 CDAs.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Left: scatter of the first two principal components; right: t-SNE embedding; points coloured by origin, identified in the legend.

**Interpretation.** Well-separated groups in t-SNE indicate distinct kinds of CDA. t-SNE preserves local neighbourhoods only: distances between groups and group sizes on the map carry no meaning.

*Output:* Figure `fig_top_embedding`.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def gower_blocks(num: np.ndarray, cat: np.ndarray, block: int = 512):
    rng = num.max(axis=0) - num.min(axis=0)
    rng[rng == 0] = 1.0
    numr = (num / rng).astype(np.float32)
    k = num.shape[1] + cat.shape[1]
    def build(mm: np.memmap) -> None:
        for start in range(0, numr.shape[0], block):
            stop = min(start + block, numr.shape[0])
            blk = np.abs(numr[start:stop, None, :] - numr[None, :, :]).sum(axis=2)
            blk += (cat[start:stop, None, :] != cat[None, :, :]).sum(axis=2)
            mm[start:stop] = blk / k
    return build

TOP_PARAMS = {"set": "top_1pct", "snapshot": EXTRACT, "features": NUM + CAT}
D = cached_memmap("B.5.18:gower", X_num.shape[0], gower_blocks(X_num, X_cat), params=TOP_PARAMS)
emb_pca = PCA(n_components=2, random_state=SEED).fit_transform(X_num)
N_EMBED = 5000
embed_idx = np.sort(np.random.default_rng(SEED).choice(D.shape[0], size=min(N_EMBED, D.shape[0]), replace=False))
emb_tsne = collect(cached("B.5.18:tsne", lambda: pl.DataFrame(
    TSNE(n_components=2, metric="precomputed", init="random", random_state=SEED).fit_transform(
        np.asarray(D[np.ix_(embed_idx, embed_idx)])), schema=["x", "y"], orient="row"),
    params={**TOP_PARAMS, "n_embed": N_EMBED}))
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08)
origin = features["ICMS_ORIGIN"].to_numpy()
for i, k in enumerate(features["ICMS_ORIGIN"].unique(maintain_order=True).to_list()):
    rows = np.where(origin == k)[0]
    sub_rows = np.where(origin[embed_idx] == k)[0]
    fig.add_trace(go.Scattergl(x=emb_pca[rows, 0], y=emb_pca[rows, 1], mode="markers", marker=dict(size=2, color=color(i)), name=k), 1, 1)
    fig.add_trace(go.Scattergl(x=emb_tsne["x"].to_numpy()[sub_rows], y=emb_tsne["y"].to_numpy()[sub_rows], mode="markers",
                               marker=dict(size=2, color=color(i)), showlegend=False), 1, 2)
fig.update_layout(showlegend=True, legend=dict(orientation="h", y=1.08))
fig.update_xaxes(title_text="PC1", row=1, col=1); fig.update_yaxes(title_text="PC2", row=1, col=1)
fig.update_xaxes(title_text="t-SNE 1", row=1, col=2); fig.update_yaxes(title_text="t-SNE 2", row=1, col=2)
save_fig(fig, "fig_top_embedding", width="double")
fig

#### B.5.19 Clustering

**Method.** Clustering on Gower distances. Agglomerative clustering with average linkage merges, at each step, the two clusters with the smallest average distance; HDBSCAN finds dense regions and labels sparse points as noise. Both are fitted on a random subsample of 5,000 CDAs and every other CDA receives the label of its nearest subsample member. The number of agglomerative clusters is chosen by the mean silhouette coefficient; stability is assessed by the adjusted Rand index over 20 pairs of independent subsamples.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** The horizontal axis gives the number of clusters; the vertical axis gives the mean silhouette coefficient (−1 to 1); the dotted line marks the selected number. The table gives stability and agreement indices.

**Interpretation.** Higher silhouettes mean more compact, better-separated clusters. Adjusted Rand values near 1 indicate stable partitions; values near 0 indicate agreement no better than chance.

*Output:* Figure `fig_cluster_selection`; Table `tab_cluster_stability`.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, HDBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

def agglomerative(M, k: int) -> np.ndarray:
    return AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="average").fit_predict(M)

def density(M, min_cluster_size: int = 50) -> np.ndarray:
    return HDBSCAN(min_cluster_size=min_cluster_size, metric="precomputed").fit_predict(M)

def submatrix(idx: np.ndarray) -> np.ndarray:
    return np.asarray(D[np.ix_(idx, idx)])

def choose_k(ks=range(2, 11), size: int = 5000) -> dict:
    idx = np.sort(np.random.default_rng(SEED).choice(D.shape[0], size=min(size, D.shape[0]), replace=False))
    Ds = submatrix(idx)
    return {str(k): float(silhouette_score(Ds, agglomerative(Ds, k), metric="precomputed")) for k in ks}

def subsample_stability(fit, n_rep: int = 20, size: int = 5000) -> list[float]:
    rng, n, scores = np.random.default_rng(SEED), D.shape[0], []
    for _ in range(n_rep):
        a = np.sort(rng.choice(n, size=min(size, n), replace=False))
        b = np.sort(rng.choice(n, size=min(size, n), replace=False))
        la, lb = fit(submatrix(a)), fit(submatrix(b))
        _, ia, ib = np.intersect1d(a, b, return_indices=True)
        scores.append(float(adjusted_rand_score(la[ia], lb[ib])))
    return scores

sil = cached_json("B.5.19:silhouette", choose_k, params=TOP_PARAMS)
K_STAR = int(max(sil, key=sil.get))
N_FIT = 5000
fit_idx = np.sort(np.random.default_rng(SEED + 1).choice(D.shape[0], size=min(N_FIT, D.shape[0]), replace=False))

def extend_labels(fit_labels: np.ndarray, block: int = 1024) -> np.ndarray:
    out = np.empty(D.shape[0], dtype=np.int64)
    for start in range(0, D.shape[0], block):
        stop = min(start + block, D.shape[0])
        out[start:stop] = fit_labels[np.asarray(D[start:stop][:, fit_idx]).argmin(axis=1)]
    out[fit_idx] = fit_labels
    return out

def full_labels() -> pl.DataFrame:
    Ds = submatrix(fit_idx)
    return pl.DataFrame({"ID_DEBITO": features["ID_DEBITO"],
                         "agglomerative": extend_labels(agglomerative(Ds, K_STAR)),
                         "hdbscan": extend_labels(density(Ds))})

labels = collect(cached("B.5.19:labels", full_labels, params={**TOP_PARAMS, "k": K_STAR, "n_fit": N_FIT}))
stab = cached_json("B.5.19:stability", lambda: {
    "agglomerative": subsample_stability(lambda M: agglomerative(M, K_STAR)),
    "hdbscan": subsample_stability(density)}, params={**TOP_PARAMS, "k": K_STAR})
stability_tab = pl.DataFrame([{"method": m, "mean_ari": float(np.mean(v)), "sd_ari": float(np.std(v)),
                               "p05_ari": float(np.percentile(v, 5))} for m, v in stab.items()]).with_columns(
    pl.lit(float(adjusted_rand_score(labels["agglomerative"], labels["hdbscan"]))).alias("agreement_ari"),
    pl.lit(K_STAR).alias("k_agglomerative"), pl.lit(int((labels["hdbscan"] == -1).sum())).alias("hdbscan_noise"))
fig = go.Figure(go.Scatter(x=[int(k) for k in sil], y=list(sil.values()), mode="lines+markers", line_color=CEMEPI["deep_teal"]))
fig.add_vline(x=K_STAR, line_dash="dot", line_color=CEMEPI["sand"], annotation_text=f"selected k = {K_STAR}",
              annotation_position="top right")
fig.update_layout(xaxis_title="Number of clusters (agglomerative, average linkage)", yaxis_title="Mean silhouette coefficient")
save_fig(fig, "fig_cluster_selection")
save_tab(stability_tab, "tab_cluster_stability", fmt="{:.3f}")
fig

#### B.5.20 Archetypes

**Method.** Profiles of the agglomerative clusters, interpreted as archetypes of large debt.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** Heat map: rows are clusters, columns are numerical features; colour and printed value give the cluster median standardised across clusters (z-score), with teal above and sand below the average across clusters. Empty cells correspond to features without variation across cluster medians.

**Interpretation.** Reading a row gives the profile of an archetype; the table adds size, share of the stock and modal categorical features.

*Output:* Figure `fig_archetype_profiles`; Table `tab_archetypes`.

In [ ]:
clustered = features.join(labels, on="ID_DEBITO")
medoids = {}
for (c,), d in clustered.with_row_index().group_by("agglomerative"):
    idx = d["index"].to_numpy()
    sub = idx if idx.size <= 3000 else np.random.default_rng(SEED).choice(idx, 3000, replace=False)
    medoids[c] = int(sub[np.asarray(D[np.ix_(sub, sub)]).sum(axis=1).argmin()])
archetypes = clustered.group_by("agglomerative").agg(
    pl.len().alias("n_cda"), (100 * pl.col("value").sum() / clustered["value"].sum()).alias("pct_stock"),
    *[pl.col(c).median().alias(c) for c in NUM], *[pl.col(c).mode().first().alias(c) for c in CAT]).sort("agglomerative")
prof = archetypes.select("agglomerative", *NUM)
z = prof.select([((pl.col(c) - pl.col(c).mean()) / pl.col(c).std()).alias(c) for c in NUM]).with_columns(
    pl.col("log_value").alias("log_value")).insert_column(0, prof["agglomerative"].map_elements(lambda c: f"Cluster {c}", return_dtype=pl.Utf8).alias("cluster"))
fig = heatmap(z, "cluster", diverging=True, zmin=-2, zmax=2, colorbar_title="Cluster median<br>(z-score)")
fig.update_layout(xaxis_title="Feature", yaxis_title="Agglomerative cluster")
save_fig(fig, "fig_archetype_profiles", width="double")
save_tab(archetypes, "tab_archetypes", fmt="{:,.3f}")
fig

#### B.5.21 Similarity of recovery trajectories

**Method.** Similarity of recovery trajectories. The main measure is the Euclidean distance between trajectories on a fixed grid of 0 to 10 years since enrolment. As a robustness check, dynamic time warping (DTW), which aligns trajectories that have similar shapes but different speeds, is computed for a stratified subsample of 1,000 trajectories. Trajectories are clustered hierarchically and a medoid is shown for each group.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** The horizontal axis gives years since enrolment; the vertical axis the accumulated recovery of each group's medoid trajectory in percent; group sizes appear in the legend. The annotation gives the Spearman correlation between grid and DTW distances.

**Interpretation.** A correlation near 1 means that both distance measures rank pairs of trajectories alike, so the grouping does not depend on the measure. A group with very few members represents an atypical trajectory rather than a pattern.

*Output:* Figure `fig_dtw_clusters`.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

T_wide = traj.pivot(on="h", index="ID_DEBITO", values="acc_rec").sort("ID_DEBITO")
T_ids = T_wide["ID_DEBITO"]
T_mat = T_wide.drop("ID_DEBITO").to_numpy().astype(np.float32)

def grid_blocks(X: np.ndarray, block: int = 2048):
    Xf = np.nan_to_num(X)
    sq = (Xf ** 2).sum(axis=1)
    def build(mm: np.memmap) -> None:
        for start in range(0, Xf.shape[0], block):
            stop = min(start + block, Xf.shape[0])
            mm[start:stop] = np.sqrt(np.maximum(sq[start:stop, None] + sq[None, :] - 2 * Xf[start:stop] @ Xf.T, 0))
    return build

def dtw(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    acc = np.full((a.size + 1, b.size + 1), np.inf)
    acc[0, 0] = 0.0
    for i in range(1, a.size + 1):
        for j in range(1, b.size + 1):
            acc[i, j] = abs(a[i - 1] - b[j - 1]) + min(acc[i - 1, j], acc[i, j - 1], acc[i - 1, j - 1])
    return float(acc[-1, -1])

D_grid = cached_memmap("B.5.21:grid", T_mat.shape[0], grid_blocks(T_mat), params={**TOP_PARAMS, "grid": "0-10y"})
traj_labels = labels.join(pl.DataFrame({"ID_DEBITO": T_ids}), on="ID_DEBITO", how="right")["agglomerative"].fill_null(-1).to_numpy()
rng = np.random.default_rng(SEED)
rep_idx = np.sort(np.concatenate([rng.choice(np.where(traj_labels == c)[0], size=min(int(1000 * (traj_labels == c).mean()) + 1, int((traj_labels == c).sum())), replace=False)
                                  for c in np.unique(traj_labels)]))

def dtw_matrix() -> list[list[float]]:
    X = T_mat[rep_idx]
    out = np.zeros((len(X), len(X)), dtype=np.float32)
    for i in range(len(X)):
        for j in range(i + 1, len(X)):
            out[i, j] = out[j, i] = dtw(X[i], X[j])
    return out.tolist()

D_dtw = np.array(cached_json("B.5.21:dtw", dtw_matrix, params={**TOP_PARAMS, "n_rep": int(rep_idx.size)}), dtype=np.float32)
D_grid_rep = np.asarray(D_grid[np.ix_(rep_idx, rep_idx)])
traj_groups = fcluster(linkage(squareform(D_grid_rep, checks=False), method="average"), t=K_STAR, criterion="maxclust")
corr_dtw_grid = stats.spearmanr(squareform(D_grid_rep, checks=False), squareform(D_dtw, checks=False))[0]
fig = go.Figure()
for i, g in enumerate(np.unique(traj_groups)):
    members = np.where(traj_groups == g)[0]
    med = members[D_grid_rep[np.ix_(members, members)].sum(axis=1).argmin()]
    y = T_mat[rep_idx][med] * 100
    fig.add_trace(go.Scatter(x=list(range(T_mat.shape[1])), y=y, mode="lines+markers", name=f"group {g} (n={members.size:,})",
                             line=dict(color=color(i), dash=DASHES[i % 5])))
fig.add_annotation(x=0.02, y=0.98, xref="paper", yref="paper", text=f"Spearman(grid, DTW) = {corr_dtw_grid:.2f}", showarrow=False, xanchor="left")
fig.update_layout(xaxis_title="Years since enrolment", yaxis_title="Accumulated recovery, medoid (%)", showlegend=True,
                  legend=dict(orientation="h", yanchor="top", y=-0.22, x=0.5, xanchor="center"))
save_fig(fig, "fig_dtw_clusters", width="double")
fig

#### Comparison with the tail

#### B.5.22 Top set versus the rest of the portfolio

**Method.** Comparison of the top set with the rest of the portfolio. Continuous variables use the Mann–Whitney *U* test and the rank-biserial correlation, defined as 1 − 2*U*/(*n*₁*n*₂) with the top set as the first group, so that negative values indicate larger values in the top set. Categorical variables use the chi-square test and Cramér's *V*. *p*-values are adjusted with the Benjamini–Hochberg procedure.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026. Remainder represented by a random sample of 200,000 CDAs.

**How to read.** Horizontal bars, one per variable; the horizontal axis gives the effect size; bars are coloured by type of effect size, identified in the legend.

**Interpretation.** Longer bars indicate larger differences. Cramér's *V* is non-negative and measures strength only; the sign of the rank-biserial correlation gives the direction.

*Output:* Figure `fig_top_vs_tail_effects`; Table `tab_top_vs_tail`.

In [ ]:
tail = raw_values.filter(~pl.col("ID_DEBITO").is_in(list(top_ids))).sample(n=200_000, seed=SEED)
cmp = pl.concat([top.select(raw_values.columns).with_columns(pl.lit(True).alias("is_top")),
                 tail.with_columns(pl.lit(False).alias("is_top"))]).join(
    cda.select("ID_DEBITO", "years_to_first_payment", "ever_paid"), on="ID_DEBITO", how="left").join(
    rate_cda.select("ID_DEBITO", "rate_cda"), on="ID_DEBITO", how="left")
rows = []
for var in ("value", "age", "rate_cda", "years_to_first_payment"):
    a = cmp.filter(pl.col("is_top"))[var].drop_nulls().to_numpy().astype(float)
    b = cmp.filter(~pl.col("is_top"))[var].drop_nulls().to_numpy().astype(float)
    r = mann_whitney(a, b)
    rows.append({"variable": var, "test": "Mann-Whitney", "effect": r["rank_biserial"], "effect_measure": "rank-biserial", "p_value": r["p_value"]})
for var in ("ever_paid", "STATUS_AJUIZAMENTO_DEBITO", "ICMS_ORIGIN", "SITUACAO_DEBITO"):
    ct = cmp.group_by("is_top", var).len().pivot(on=var, index="is_top", values="len").fill_null(0).drop("is_top").to_numpy()
    r = chi2_table(ct)
    rows.append({"variable": var, "test": "chi-square", "effect": r["cramers_v"], "effect_measure": "Cramér's V", "p_value": r["p_value"]})
top_vs_tail = pl.DataFrame(rows).with_columns(pl.Series("p_bh", multipletests([r["p_value"] for r in rows], method="fdr_bh")[1]))
d = top_vs_tail.sort("effect")
VAR_NAMES = {"value": "Value", "age": "Age", "rate_cda": "Recovery rate", "years_to_first_payment": "Years to first payment",
             "ever_paid": "Any payment", "STATUS_AJUIZAMENTO_DEBITO": "Filing status", "ICMS_ORIGIN": "ICMS origin",
             "SITUACAO_DEBITO": "Administrative situation"}
fig = go.Figure([go.Bar(y=[VAR_NAMES.get(v, v) for v in g["variable"]], x=g["effect"], orientation="h", marker_color=color(i), name=label,
                        text=[f"{v:.2f}" for v in g["effect"]], textposition="outside", cliponaxis=False)
                 for i, (m, label) in enumerate((("rank-biserial", "Rank-biserial correlation (continuous)"), ("Cramér's V", "Cramér's V (categorical)")))
                 for g in [d.filter(pl.col("effect_measure") == m)]])
fig.update_layout(xaxis_title="Effect size, top 1% versus remainder", showlegend=True, legend=dict(orientation="h", y=-0.3),
                  yaxis=dict(categoryorder="array", categoryarray=[VAR_NAMES.get(v, v) for v in d["variable"]]),
                  margin=dict(r=40))
save_fig(fig, "fig_top_vs_tail_effects")
save_tab(top_vs_tail, "tab_top_vs_tail", fmt="{:.4g}")
fig

#### Optional linkage with firm revenue

#### B.5.23 Linkage with masked revenue

**Method.** Linkage of the top debtors with masked revenue records (`FATURAMENTO_MASCARADO`) through the debtor identifier.

**Data.** CDAs in the top 1% by value in the March 2026 snapshot (B.5.1), with their panel and payment records, 2016–2026.

**How to read.** The table reports linkage coverage.

**Interpretation.** Ratios of debt to revenue are interpreted only if coverage is adequate.

*Output:* Table `tab_revenue_linkage_coverage`.

In [ ]:
def revenue_coverage() -> pl.DataFrame:
    con = duck()
    try:
        cols = [c for (c,) in con.execute("SELECT column_name FROM information_schema.columns WHERE table_name = 'FATURAMENTO_MASCARADO'").fetchall()]
        id_col = next((c for c in cols if "CNPJ" in c.upper()), None)
        if id_col is None:
            return pl.DataFrame({"columns": [", ".join(cols)], "linkable": [False]})
        roots = con.execute(f"SELECT DISTINCT LEFT(REGEXP_REPLACE(CAST({id_col} AS VARCHAR), '[^0-9]', '', 'g'), 8) AS root FROM FATURAMENTO_MASCARADO").pl()
    finally:
        con.close()
    linked = pl.DataFrame({"root": top_roots}).join(roots, on="root", how="inner").height
    return pl.DataFrame({"identifier_column": [id_col], "top_firms": [len(top_roots)], "linked_firms": [linked],
                         "coverage_pct": [100 * linked / len(top_roots)], "linkable": [linked > 0]})

coverage = collect(cached("B.5.23", revenue_coverage, params=TOP_PARAMS))
save_tab(coverage, "tab_revenue_linkage_coverage", fmt="{:,.2f}")
coverage

#### B.5.24 Anonymisation

**Method.** Pseudonymisation of debtor identifiers with a salted SHA-256 hash; the salt is kept outside the repository.

**Data.** CNPJ roots of the top debtors.

**How to read.** The cell defines the hashing function used in all debtor-level outputs.

**Interpretation.** No names or raw identifiers are written to figures or tables.

In [ ]:
SALT = os.environ.get("CEMEPI_HASH_SALT")
if not SALT:
    raise RuntimeError("CEMEPI_HASH_SALT must be set in .env to anonymise debtor identifiers.")

def anonymise(frame: pl.DataFrame, col: str = "CNPJ_ROOT") -> pl.DataFrame:
    hashed = frame[col].map_elements(lambda v: hashlib.sha256((SALT + v).encode()).hexdigest()[:12], return_dtype=pl.Utf8)
    return frame.with_columns(hashed.alias("debtor_hash")).drop([c for c in (col, "CNPJ") if c in frame.columns])

archetype_members = anonymise(clustered.select("CNPJ_ROOT", "agglomerative", "value"))
assert not any(c in archetype_members.columns for c in ("CNPJ", "CNPJ_ROOT"))
archetype_members.head()

### B.6 Age of CDAs

*Terms introduced in this section:* Age band. Definitions are given in the Glossary.

#### B.6.1 Age bands

**Method.** Distribution of CDAs and stock across age bands.

**Data.** ICMS CDA × year panel (1.4), legal entities only, year 2026; unit of observation: the CDA-year.

**How to read.** Grouped vertical bars, one pair per age band (ordered from youngest to oldest); the vertical axis gives the share in percent; the legend distinguishes the share of CDAs from the share of stock.

**Interpretation.** A band whose stock bar exceeds its count bar holds debts larger than average.

*Output:* Figure `fig_age_bands`; Table `tab_age_bands`.

In [ ]:
AGE_BANDS = [1, 3, 5, 10]
AGE_LABELS = ["<=1", "1-3", "3-5", "5-10", ">10"]
def band(col: str = "age") -> pl.Expr:
    return pl.col(col).cut(AGE_BANDS, labels=AGE_LABELS, left_closed=False).alias("age_band")

age_bands = collect(panel.filter(pl.col("year") == SNAP_YEAR).with_columns(band()).group_by("age_band").agg(
    pl.len().alias("n_cda"), (pl.col("stock").sum() / 1e9).alias("stock_bn"))).with_columns(
    (100 * pl.col("n_cda") / pl.col("n_cda").sum()).alias("pct_cda"), (100 * pl.col("stock_bn") / pl.col("stock_bn").sum()).alias("pct_stock")
).sort(pl.col("age_band").cast(pl.Utf8).replace_strict({l: i for i, l in enumerate(AGE_LABELS)}))
fig = go.Figure([go.Bar(x=age_bands["age_band"].cast(pl.Utf8), y=age_bands[c], name=n, marker_color=color(i))
                 for i, (c, n) in enumerate((("pct_cda", "CDAs"), ("pct_stock", "Stock")))])
fig.update_layout(barmode="group", xaxis_title="Age band (years)", yaxis_title="Share (%)", showlegend=True, legend=dict(orientation="h", y=1.1))
save_fig(fig, "fig_age_bands")
save_tab(age_bands, "tab_age_bands")
fig

#### B.6.2 Exposure by age

**Method.** CDA-years at risk by age, used as denominators for age-specific rates.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** One row per year of age, with the number of CDA-years and total paid.

**Interpretation.** Dividing by exposure ensures that differences across ages reflect behaviour rather than the number of CDAs at each age.

*Output:* Table `tab_exposure_age`.

In [ ]:
exposure = collect(panel.with_columns(pl.col("age").floor().cast(pl.Int32).clip(0, 30).alias("age_year"))
                   .group_by("age_year").agg(pl.len().alias("cda_years"), pl.col("paid_flag").sum().alias("cda_years_with_payment"),
                                             pl.col("paid").sum().alias("paid"), pl.col("stock").sum().alias("stock"))
                   .with_columns((pl.col("cda_years_with_payment") / pl.col("cda_years")).alias("payment_hazard"),
                                 (pl.col("paid") / pl.col("stock")).alias("rate")).sort("age_year"))
save_tab(exposure, "tab_exposure_age", fmt="{:,.4f}")
exposure

#### B.6.3 Age and recovery

**Method.** Pearson and Spearman correlations between age and recovery rate and between age and stock. Pearson measures linear association; Spearman is the Pearson correlation of ranks and measures monotone association, robust to outliers.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year. A 20% random sample of CDA-years with positive stock.

**How to read.** One row per pair of variables, with both coefficients.

**Interpretation.** A large divergence between the coefficients indicates a nonlinear or outlier-driven relation; the formal choice is made in C.2.

*Output:* Table `tab_corr_age`.

In [ ]:
pr = collect(panel.filter(pl.col("stock") > 0).select("age", "rate", "stock")).sample(fraction=0.2, seed=SEED)
rows = []
for y in ("rate", "stock"):
    for m in ("pearson", "spearman"):
        r, p = corr_pair(pr["age"].to_numpy().astype(float), pr[y].to_numpy().astype(float), m)
        rows.append({"x": "age", "y": y, "method": m, "r": r, "p_value": p, "n": pr.height})
corr_age = pl.DataFrame(rows)
save_tab(corr_age, "tab_corr_age", fmt="{:.4g}")
corr_age

### B.7 Recovery

*Terms introduced in this section:* Enrolment cohort, recoverable stock. Definitions are given in the Glossary.

#### B.7.1 Non-accumulated and accumulated recovery

Two recovery measures are used throughout. The *non-accumulated* recovery rate of period $t$ is $r_t = P_t / S_t$, the amount paid in $t$ divided by the stock. The *accumulated* recovery of an enrolment cohort $c$ at horizon $h$ is $R_{c,h} = \sum_{k=0}^{h} P_{c,k} / S_{c,0}$, the cumulative amount paid up to $h$ divided by the stock at enrolment. The first describes the yearly performance of the portfolio; the second describes how much of a cohort's debt is eventually recovered. Axis titles state which measure is shown.

#### B.7.2 Annual portfolio recovery rate

**Method.** Annual non-accumulated recovery rate of the portfolio.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** The horizontal axis gives the year; the vertical axis the rate in percent on a logarithmic scale; labelled points mark the programme years.

**Interpretation.** On a log axis, equal vertical distances correspond to equal proportional changes; the final year covers January to March only.

*Output:* Figure `fig_taxa_recuperacao_anual`.

In [ ]:
fig = go.Figure(go.Scatter(x=annual["year"], y=annual["portfolio_rate"] * 100, mode="lines+markers", line_color=CEMEPI["deep_teal"]))
for name, a, _ in PROGRAMMES:
    y = int(a[:4])
    v = annual.filter(pl.col("year") == y)["portfolio_rate"]
    if v.len():
        fig.add_annotation(x=y, y=np.log10(v[0] * 100), text=name, showarrow=True, arrowhead=0, ay=-25)
fig.update_yaxes(type="log", title_text="Non-accumulated recovery rate (%, log scale)")
save_fig(fig, "fig_taxa_recuperacao_anual")
fig

#### B.7.3 Recovery by age

**Method.** Annual non-accumulated recovery rate by age band.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year. CDA-years with a valid age.

**How to read.** The horizontal axis gives the year; the vertical axis the rate in percent on a logarithmic scale; one line per age band, identified in the legend.

**Interpretation.** If recovery declines with age, the lines are ordered from the youngest band at the top to the oldest at the bottom.

*Output:* Figure `fig_recovery_age_log`.

In [ ]:
rec_age = collect(panel.with_columns(band()).group_by("year", "age_band").agg(
    (100 * pl.col("paid").sum() / pl.col("stock").sum()).alias("rate_pct"))).with_columns(pl.col("age_band").cast(pl.Utf8))
fig = lines(rec_age.filter((pl.col("rate_pct") > 0) & pl.col("age_band").is_not_null()), "year", "rate_pct", "age_band", log_y=True,
            y_title="Non-accumulated recovery rate (%, log scale)", order=AGE_LABELS,
            names={l: f"{l} years" for l in AGE_LABELS})
save_fig(fig, "fig_recovery_age_log", width="double")
fig

#### B.7.4 Accumulated recovery by cohort

**Method.** Accumulated recovery curves by enrolment cohort.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year. Cohorts enrolled from 2016 onward.

**How to read.** The horizontal axis gives years since enrolment; the vertical axis the cumulative amount paid as a percentage of the enrolled value, on a logarithmic scale; one line per cohort, identified in the legend.

**Interpretation.** Curves that flatten indicate that additional recovery becomes rare after a few years; later cohorts have shorter curves because fewer horizons are observed.

*Output:* Figure `fig_recovery_cumulative_log`.

In [ ]:
def cohort_curves() -> pl.DataFrame:
    enrolled = attributes.filter(pl.col("DATA_INSCRICAO").dt.year() >= YEARS[0]).select(
        "ID_DEBITO", pl.col("DATA_INSCRICAO").dt.year().alias("cohort"), "max_value")
    base = enrolled.group_by("cohort").agg(pl.col("max_value").sum().alias("enrolled_value"))
    paid = (scan_gare().join(enrolled, on="ID_DEBITO")
            .with_columns((pl.col("DATA_ARRECADACAO_GARE").dt.year() - pl.col("cohort")).alias("h"))
            .filter(pl.col("h") >= 0).group_by("cohort", "h").agg(pl.col("VALOR_RECEITA").sum().alias("paid")))
    return collect(paid.join(base, on="cohort").sort("cohort", "h").with_columns(
        (pl.col("paid").cum_sum().over("cohort") / pl.col("enrolled_value")).alias("acc_rec")))

curves = collect(cached("B.7.4", cohort_curves, params=EXTRACT_PARAMS)).with_columns(pl.col("cohort").cast(pl.Utf8))
fig = lines(curves.with_columns(pl.col("acc_rec") * 100), "h", "acc_rec", "cohort", log_y=True,
            y_title="Accumulated recovery (% of enrolled value, log scale)", x_title="Years since enrolment")
save_fig(fig, "fig_recovery_cumulative_log", width="double")
fig

#### B.7.5 Distribution of CDA recovery rates

**Method.** Descriptive statistics of CDA-level recovery.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** One row per variable (recovery rate, amount paid, payment indicator) with moments, quantiles and share of zeros.

**Interpretation.** A large share of zeros means that recovery is a mixture of non-paying CDAs and a skewed distribution among payers, which motivates modelling the probability of payment separately (D.6.1).

*Output:* Table `tab_descriptives`.

In [ ]:
descriptives = collect(panel.filter(pl.col("stock") > 0).select(
    *[pl.struct(variable=pl.lit(c), n=pl.col(c).count(), mean=pl.col(c).mean(), median=pl.col(c).median(),
                sd=pl.col(c).std(), p90=pl.col(c).quantile(0.9), max=pl.col(c).max(), share_zero=(pl.col(c) == 0).mean()).alias(c)
      for c in ("rate", "paid", "paid_flag", "stock", "age")])).transpose().unnest("column_0")
save_tab(descriptives, "tab_descriptives", fmt="{:,.4f}")
descriptives

#### B.7.6 Recovered over recoverable stock

**Method.** Recovery relative to two denominators: total year-end stock and recoverable stock. Recoverable stock keeps only debts in the enrolled administrative situation and excludes those older than ten years that have never recorded a payment.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** The horizontal axis gives the year; the vertical axis the amount recovered as a percentage of each denominator, on a logarithmic scale; one line per denominator, identified in the legend.

**Interpretation.** The ratio to recoverable stock is always the higher one; the vertical distance between the lines reflects the share of stock classified as irrecoverable. The indicator depends on the criterion adopted.

*Output:* Figure `fig_recoverable_ratio`.

In [ ]:
RECOVERABLE_MAX_AGE = 10
def recoverable_ratio() -> pl.DataFrame:
    ever_paid_ids = scan_gare().select("ID_DEBITO").unique()
    flagged = panel.join(ever_paid_ids.with_columns(pl.lit(True).alias("has_paid")), on="ID_DEBITO", how="left").with_columns(
        (~((pl.col("age") > RECOVERABLE_MAX_AGE) & pl.col("has_paid").is_null()) &
         (pl.col("SITUACAO_DEBITO") == "Inscrito")).alias("recoverable"))
    return collect(flagged.group_by("year").agg(pl.col("paid").sum(), pl.col("stock").sum().alias("stock"),
                                                pl.col("stock").filter(pl.col("recoverable")).sum().alias("recoverable_stock")).sort("year"))

rr = collect(cached("B.7.6", recoverable_ratio, params={**EXTRACT_PARAMS, "max_age": RECOVERABLE_MAX_AGE})).with_columns(
    (100 * pl.col("paid") / pl.col("stock")).alias("recovered_over_stock"),
    (100 * pl.col("paid") / pl.col("recoverable_stock")).alias("recovered_over_recoverable"))
long = rr.unpivot(index="year", on=["recovered_over_stock", "recovered_over_recoverable"], variable_name="measure", value_name="pct")
fig = lines(long, "year", "pct", "measure", log_y=True, y_title="Amount recovered (% of denominator, log scale)",
            names={"recovered_over_stock": "Over total year-end stock", "recovered_over_recoverable": "Over recoverable stock"})
save_fig(fig, "fig_recoverable_ratio", width="double")
fig

### B.8 Payments over the life of the debt

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### B.8.1 Open and paid CDAs

**Method.** Cross-tabulation of payment in the current year and payment since enrolment.

**Data.** ICMS CDA × year panel (1.4), legal entities only, year 2026; unit of observation: the CDA-year. Payment history from GARE.

**How to read.** Rows combine the two payment indicators; columns give CDA counts and stock.

**Interpretation.** CDAs never paid but still in the stock form the core of non-recovery.

*Output:* Table `tab_open_paid`.

In [ ]:
open_paid = collect(panel.filter(pl.col("year") == SNAP_YEAR).join(
    scan_gare().select("ID_DEBITO").unique().with_columns(pl.lit(True).alias("ever_paid")), on="ID_DEBITO", how="left")
    .with_columns(pl.col("ever_paid").fill_null(False))
    .group_by(pl.col("paid_flag").cast(pl.Boolean).alias("paid_this_year"), "ever_paid")
    .agg(pl.len().alias("n_cda"), (pl.col("stock").sum() / 1e9).alias("stock_bn")).sort("paid_this_year", "ever_paid"))
save_tab(open_paid, "tab_open_paid", fmt="{:,.2f}")
open_paid

#### B.8.2 Open and paid CDAs by enrolment year

**Method.** Size and payment experience of enrolment cohorts.

**Data.** ICMS extract (1.3), CDAs enrolled from 1990 onward; unit of observation: the enrolment year.

**How to read.** Bars (left axis) give CDAs enrolled per year in thousands; the line (right axis) gives the percentage with any payment; the legend states which axis each series uses.

**Interpretation.** Read the shape of the line rather than its height relative to the bars; recent cohorts have had less time to pay, so their percentage is mechanically lower.

*Output:* Figure `fig_open_paid_by_cohort`.

In [ ]:
by_cohort = cda.filter(pl.col("DATA_INSCRICAO").dt.year() >= 1990).group_by(pl.col("DATA_INSCRICAO").dt.year().alias("cohort")).agg(
    pl.len().alias("n_cda"), (100 * pl.col("ever_paid").mean()).alias("pct_ever_paid")).sort("cohort")
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=by_cohort["cohort"], y=by_cohort["n_cda"] / 1e3, marker_color="rgba(21,76,86,0.35)",
                     name="CDAs enrolled (left axis)"), secondary_y=False)
fig.add_trace(go.Scatter(x=by_cohort["cohort"], y=by_cohort["pct_ever_paid"], mode="lines+markers", line_color=CEMEPI["sand"],
                         name="CDAs with any payment, % (right axis)"), secondary_y=True)
fig.update_yaxes(title_text="CDAs (thousand)", secondary_y=False, rangemode="tozero")
fig.update_yaxes(title_text="With any payment (%)", secondary_y=True, showgrid=False, rangemode="tozero", tickformat=".0f")
fig.update_layout(showlegend=True, legend=dict(orientation="h", y=-0.3))
fig.update_xaxes(title_text="Year of enrolment")
save_fig(fig, "fig_open_paid_by_cohort", width="double")
fig

#### B.8.3 Paying population by age

**Method.** Paying population by age band.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** One row per age band with numbers of CDAs, of paying CDAs and the share paying.

**Interpretation.** A declining share with age indicates that older debts are less likely to have ever been paid.

*Output:* Table `tab_payers_age`.

In [ ]:
payers_age = cda.with_columns(((pl.date(SNAP_YEAR, SNAP_MONTH, 1) - pl.col("DATA_INSCRICAO")).dt.total_days() / 365.25).alias("age")) \
    .with_columns(band()).group_by("age_band").agg(pl.len().alias("n_cda"), pl.col("ever_paid").sum().alias("n_paid")) \
    .with_columns((100 * pl.col("n_paid") / pl.col("n_cda")).alias("pct_paid"), pl.col("age_band").cast(pl.Utf8)).sort("age_band")
save_tab(payers_age, "tab_payers_age", fmt="{:,.2f}")
payers_age

#### B.8.4 Time to first payment

**Method.** Delay between enrolment and first payment among paying CDAs.

**Data.** ICMS extract (1.3), CDAs enrolled from 2016 onward with at least one payment.

**How to read.** Vertical bars by whole years from enrolment to first payment, the last grouping ten years or more; the vertical axis gives the share of paying CDAs, printed above each bar.

**Interpretation.** Mass in the first bar means that, among CDAs that are paid at all, payment usually starts soon after enrolment.

*Output:* Figure `fig_delay_first_gare`.

In [ ]:
delay = cda.filter(pl.col("ever_paid") & (pl.col("DATA_INSCRICAO") >= pl.date(YEARS[0], 1, 1))) \
    .with_columns(pl.col("years_to_first_payment").floor().cast(pl.Int32).clip(0, 10).alias("delay")).group_by("delay").len().sort("delay")
delay = delay.with_columns((100 * pl.col("len") / pl.col("len").sum()).alias("pct"))
fig = go.Figure(go.Bar(x=delay["delay"], y=delay["pct"], marker_color=CEMEPI["deep_teal"], text=[f"{v:.0f}%" for v in delay["pct"]], textposition="outside"))
fig.update_layout(xaxis_title="Years from enrolment to first payment (10 = 10 or more)", yaxis_title="Share of paying CDAs (%)")
save_fig(fig, "fig_delay_first_gare")
fig

#### B.8.5 Time to settlement

**Method.** Time to settlement, approximated by the exit of a CDA from the snapshots after at least one payment, because settlement is not recorded.

**Data.** ICMS extract (1.3), CDAs with a payment that left the snapshots before March 2026.

**How to read.** Histogram; the horizontal axis gives years from enrolment to exit, the vertical axis numbers of CDAs.

**Interpretation.** Exits can also result from cancellation, so the proxy is an upper bound on settlement times.

*Output:* Figure `fig_settlement_proxy`.

In [ ]:
last_snap = SNAP_YEAR * 100 + SNAP_MONTH
settled = cda.filter((pl.col("last_snapshot") < last_snap) & pl.col("ever_paid")).with_columns(
    ((pl.date(pl.col("last_snapshot") // 100, pl.col("last_snapshot") % 100, 1) - pl.col("DATA_INSCRICAO")).dt.total_days() / 365.25).alias("years_to_exit"))
fig = go.Figure(go.Histogram(x=settled["years_to_exit"].clip(0, 20), nbinsx=40, marker_color=CEMEPI["deep_teal"]))
fig.update_layout(xaxis_title="Years from enrolment to exit after payment (settlement proxy)", yaxis_title="CDAs")
save_fig(fig, "fig_settlement_proxy")
fig

#### B.8.6 Age windows of payments

**Method.** Distribution of payments by age of the CDA at the time of payment, and the payment hazard per exposed CDA-year.

**Data.** GARE payments linked to ICMS CDAs, 2016–2026; exposure from the panel.

**How to read.** Grouped bars by age band (youngest to oldest); the vertical axis gives the share of payment records and of the amount paid, identified in the legend. The table adds the hazard.

**Interpretation.** A band whose value bar exceeds its count bar receives fewer but larger payments.

*Output:* Figure `fig_gare_window_life`; Table `tab_gare_window`.

In [ ]:
def gare_windows() -> pl.DataFrame:
    g = scan_gare().join(attributes.select("ID_DEBITO", "DATA_INSCRICAO"), on="ID_DEBITO").with_columns(
        ((pl.col("DATA_ARRECADACAO_GARE") - pl.col("DATA_INSCRICAO")).dt.total_days() / 365.25).alias("age"))
    g = g.with_columns(band())
    pay = collect(g.group_by("age_band").agg(pl.len().alias("payments"), pl.col("ID_DEBITO").n_unique().alias("cda_paying"),
                                             pl.col("VALOR_RECEITA").sum().alias("value")))
    expo = collect(panel.with_columns(band()).group_by("age_band").agg(pl.len().alias("cda_years")))
    return pay.join(expo, on="age_band").with_columns(
        (100 * pl.col("payments") / pl.col("payments").sum()).alias("pct_payments"),
        (100 * pl.col("value") / pl.col("value").sum()).alias("pct_value"),
        (pl.col("cda_paying") / pl.col("cda_years")).alias("hazard"), pl.col("age_band").cast(pl.Utf8))

windows = collect(cached("B.8.6", gare_windows, params=EXTRACT_PARAMS)).sort("age_band")
fig = go.Figure([go.Bar(x=windows["age_band"], y=windows[c], name=n, marker_color=color(i)) for i, (c, n) in
                 enumerate((("pct_payments", "Payments"), ("pct_value", "Value paid")))])
fig.update_layout(barmode="group", xaxis_title="Age of CDA at payment (years)", yaxis_title="Share (%)", showlegend=True, legend=dict(orientation="h", y=1.1))
save_fig(fig, "fig_gare_window_life")
save_tab(windows, "tab_gare_window", fmt="{:,.4f}")
fig

#### B.8.7 Cohort by horizon

**Method.** Cumulative share of CDAs with a payment by cohort and horizon.

**Data.** ICMS extract (1.3), cohorts 2016–2026.

**How to read.** Heat map: rows are enrolment cohorts, columns years since enrolment; colour and printed values give the percentage with any payment, as shown by the colour bar.

**Interpretation.** Read along a row for the speed of payment of a cohort and down a column to compare cohorts at the same horizon; the triangle arises because later cohorts are observed for fewer years.

*Output:* Figure `fig_cohort_horizon`.

In [ ]:
horizon = (cda.filter(pl.col("DATA_INSCRICAO").dt.year() >= YEARS[0])
           .with_columns(pl.col("DATA_INSCRICAO").dt.year().alias("cohort"),
                         (pl.col("first_payment").dt.year() - pl.col("DATA_INSCRICAO").dt.year()).alias("h")))
size = horizon.group_by("cohort").len()
cells = []
for h in range(0, SNAP_YEAR - YEARS[0] + 1):
    c = horizon.group_by("cohort").agg((pl.col("h") <= h).sum().alias("n")).with_columns(pl.lit(h).alias("h"))
    cells.append(c)
grid = pl.concat(cells).join(size, on="cohort").with_columns(
    pl.when(pl.col("cohort") + pl.col("h") <= SNAP_YEAR).then(100 * pl.col("n") / pl.col("len")).alias("pct"))
mat = grid.pivot(on="h", index="cohort", values="pct").sort("cohort").with_columns(pl.col("cohort").cast(pl.Utf8))
fig = heatmap(mat, "cohort", diverging=False, zmin=0, fmt="{:.0f}", colorbar_title="% with payment")
fig.update_layout(xaxis_title="Years since enrolment", yaxis_title="Enrolment cohort")
save_fig(fig, "fig_cohort_horizon", width="double")
fig

#### B.8.8 Annual collection

**Method.** Annual GARE collection on ICMS CDAs.

**Data.** GARE payments linked to ICMS CDAs, 2016–2026.

**How to read.** Vertical bars by year; the vertical axis gives R$ billion, printed above each bar.

**Interpretation.** The final year covers January to March only and is not comparable with full years.

*Output:* Figure `fig_gare_annual`.

In [ ]:
gare_annual = collect(scan_gare().group_by(pl.col("DATA_ARRECADACAO_GARE").dt.year().alias("year"))
                      .agg((pl.col("VALOR_RECEITA").sum() / 1e9).alias("collected_bn")).sort("year"))
fig = go.Figure(go.Bar(x=gare_annual["year"], y=gare_annual["collected_bn"], marker_color=CEMEPI["deep_teal"],
                       text=[f"{v:.2f}" for v in gare_annual["collected_bn"]], textposition="outside"))
fig.update_layout(yaxis_title="GARE collection on ICMS CDAs (R$ bn)")
save_fig(fig, "fig_gare_annual")
fig

### B.9 Judicial filing

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### B.9.1 Filing status of the portfolio

**Method.** Composition of the stock by judicial filing status. The Portuguese categories are translated; records whose status code does not match a documented category are grouped as *unrecognised code*.

**Data.** Year-end snapshots of the ICMS extract, 2016–2026; unit of observation: the status-year.

**How to read.** The horizontal axis gives the year; the vertical axis the share of year-end stock in percent; one line per status, identified in the legend.

**Interpretation.** A stable dominant share for *filed* indicates that most of the stock is under judicial collection.

*Output:* Figure `fig_filing_status`; Table `tab_filing_status`.

In [ ]:
filing_year = collect(year_end.with_columns(filing_label()).group_by("year", "STATUS_AJUIZAMENTO_DEBITO").agg(
    pl.len().alias("n_cda"), (pl.col("stock").sum() / 1e9).alias("stock_bn"))).with_columns(
    (100 * pl.col("stock_bn") / pl.col("stock_bn").sum().over("year")).alias("pct_stock")).sort("year")
fig = lines(filing_year, "year", "pct_stock", "STATUS_AJUIZAMENTO_DEBITO", y_title="Share of year-end stock (%)",
            order=list(FILING_LABELS.values()) + ["Unrecognised code"])
save_fig(fig, "fig_filing_status", width="double")
save_tab(filing_year.filter(pl.col("year") == SNAP_YEAR), "tab_filing_status", fmt="{:,.2f}")
fig

#### B.9.2 Filing by age and value

**Method.** Filing by age band and by value decile.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** Left: bars of the share of CDAs filed by age band. Right: bars of the share of CDAs filed by value decile (1 = smallest tenth, 10 = largest) and a line for the share of stock filed within each decile; the legend identifies the series.

**Interpretation.** Rising bars indicate that older and larger debts are more likely to be filed.

*Output:* Figure `fig_filing_age_value`.

In [ ]:
fv = raw_values.with_columns(band(), (pl.col("value").rank() * 10 / pl.len()).ceil().cast(pl.Int32).alias("decile"),
                             (pl.col("STATUS_AJUIZAMENTO_DEBITO") == "Ajuizado").alias("filed"))
by_age = fv.group_by("age_band").agg((100 * pl.col("filed").mean()).alias("pct")).with_columns(pl.col("age_band").cast(pl.Utf8)).sort("age_band")
by_dec = fv.group_by("decile").agg((100 * pl.col("filed").mean()).alias("pct"),
                                   (100 * pl.col("value").filter(pl.col("filed")).sum() / pl.col("value").sum()).alias("pct_value")).sort("decile")
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)
fig.add_trace(go.Bar(x=by_age["age_band"], y=by_age["pct"], marker_color=CEMEPI["deep_teal"], name="Share of CDAs filed", legendgroup="n"), 1, 1)
fig.add_trace(go.Bar(x=by_dec["decile"], y=by_dec["pct"], marker_color=CEMEPI["deep_teal"], name="Share of CDAs filed", legendgroup="n",
                     showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=by_dec["decile"], y=by_dec["pct_value"], mode="lines+markers", line_color=CEMEPI["sand"],
                         name="Share of stock filed"), 1, 2)
fig.update_yaxes(title_text="Filed in court (%)", row=1, col=2)
fig.update_xaxes(tickmode="array", tickvals=list(range(1, 11)), row=1, col=2)
fig.update_layout(showlegend=True, legend=dict(orientation="h", y=-0.3))
fig.update_xaxes(title_text="Age band (years)", row=1, col=1)
fig.update_xaxes(title_text="Value decile", row=1, col=2)
fig.update_yaxes(title_text="Filed in court (%)", row=1, col=1)
save_fig(fig, "fig_filing_age_value", width="double")
fig

#### B.9.3 Time to filing

**Method.** Time from enrolment to filing for CDAs whose status changes to filed during the observation window.

**Data.** ICMS extract (1.3), 2016–2026.

**How to read.** Histogram; the horizontal axis gives years from enrolment to the first snapshot as filed (capped at 15), the vertical axis numbers of CDAs.

**Interpretation.** CDAs already filed at their first observation are excluded, so the distribution describes recent filing practice.

*Output:* Figure `fig_time_to_filing`.

In [ ]:
filed_later = cda.filter(pl.col("first_filed_snapshot").is_not_null() & (pl.col("first_filed_snapshot") > pl.col("first_snapshot"))) \
    .with_columns(((pl.date(pl.col("first_filed_snapshot") // 100, pl.col("first_filed_snapshot") % 100, 1) - pl.col("DATA_INSCRICAO"))
                   .dt.total_days() / 365.25).alias("years_to_filing"))
fig = go.Figure(go.Histogram(x=filed_later["years_to_filing"].clip(0, 15), nbinsx=45, marker_color=CEMEPI["deep_teal"]))
fig.update_layout(xaxis_title="Years from enrolment to judicial filing", yaxis_title="CDAs")
save_fig(fig, "fig_time_to_filing")
fig

#### B.9.4 Payments and recovery by filing status

**Method.** Amount paid by age band and filing status.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** Stacked vertical bars by age band (youngest to oldest); the vertical axis gives the amount paid 2016–2026 in R$ billion; segments correspond to filing statuses, identified in the legend.

**Interpretation.** If payments on older debts come mainly from filed CDAs, this is consistent with judicial collection supporting recovery at advanced ages; the table adds payment shares and rates.

*Output:* Figure `fig_recovery_by_filing`; Table `tab_recovery_by_filing`.

In [ ]:
rec_filing = collect(panel.with_columns(band(), filing_label()).group_by("STATUS_AJUIZAMENTO_DEBITO", "age_band").agg(
    pl.len().alias("cda_years"), (100 * pl.col("paid_flag").mean()).alias("pct_paying"),
    (100 * pl.col("paid").sum() / pl.col("stock").sum()).alias("rate_pct"), (pl.col("paid").sum() / 1e9).alias("paid_bn"))
    .with_columns(pl.col("age_band").cast(pl.Utf8)).sort("STATUS_AJUIZAMENTO_DEBITO", "age_band"))
fig = go.Figure([go.Bar(x=d["age_band"], y=d["paid_bn"], name=k[0], marker_color=color(i))
                 for i, (k, d) in enumerate(rec_filing.group_by("STATUS_AJUIZAMENTO_DEBITO", maintain_order=True))])
fig.update_layout(barmode="stack", xaxis_title="Age band (years)", yaxis_title="Amount paid 2016-2026 (R$ bn)", showlegend=True, legend=dict(orientation="h", y=1.12))
save_fig(fig, "fig_recovery_by_filing", width="double")
save_tab(rec_filing, "tab_recovery_by_filing", fmt="{:,.3f}")
fig

#### B.9.5 Payments around the filing date

**Method.** Event-time profile of payments around filing: CDAs are aligned on the month in which their status changes to filed (month 0).

**Data.** ICMS extract (1.3) and GARE payments, CDAs filed during the window.

**How to read.** The horizontal axis gives months relative to filing (−12 to 12); the vertical axis the percentage of CDAs with a payment in that month; the dotted line marks month 0.

**Interpretation.** A rise after month 0 is consistent with filing prompting payment, but alignment alone does not establish causation, since filing may target debts likely to pay.

*Output:* Figure `fig_payment_around_filing`.

In [ ]:
ev = filed_later.select("ID_DEBITO", (pl.col("first_filed_snapshot") // 100 * 12 + pl.col("first_filed_snapshot") % 100).alias("filed_m"))
pay_m = collect(scan_gare().join(ev.lazy(), on="ID_DEBITO").with_columns(
    (pl.col("DATA_ARRECADACAO_GARE").dt.year() * 12 + pl.col("DATA_ARRECADACAO_GARE").dt.month() - pl.col("filed_m")).alias("k"))
    .filter(pl.col("k").is_between(-12, 12)).group_by("k").agg(pl.col("ID_DEBITO").n_unique().alias("paying")))
pay_m = pay_m.with_columns((100 * pl.col("paying") / ev.height).alias("hazard")).sort("k")
fig = go.Figure(go.Scatter(x=pay_m["k"], y=pay_m["hazard"], mode="lines+markers", line_color=CEMEPI["deep_teal"]))
fig.add_vline(x=0, line_dash="dot", line_color=CEMEPI["sand"])
fig.update_layout(xaxis_title="Months relative to judicial filing", yaxis_title="CDAs with a payment (%)")
save_fig(fig, "fig_payment_around_filing")
fig

#### B.9.6 Filing by origin and debtor size

**Method.** Filing by origin and by concentration stratum.

**Data.** ICMS extract (1.3), March 2026 snapshot, legal entities only; unit of observation: the CDA.

**How to read.** One row per origin and stratum (top 0.1%, top 1%, remainder), with shares of CDAs and stock filed.

**Interpretation.** Differences indicate whether filing priorities depend on origin or debt size.

*Output:* Table `tab_filing_origin_size`.

In [ ]:
filing_origin_size = strata.group_by("ICMS_ORIGIN", "stratum").agg(
    pl.len().alias("n_cda"), (100 * (pl.col("STATUS_AJUIZAMENTO_DEBITO") == "Ajuizado").mean()).alias("pct_filed"),
    (100 * pl.col("value").filter(pl.col("STATUS_AJUIZAMENTO_DEBITO") == "Ajuizado").sum() / pl.col("value").sum()).alias("pct_stock_filed")
).sort("ICMS_ORIGIN", "stratum")
save_tab(filing_origin_size, "tab_filing_origin_size", fmt="{:,.2f}")
filing_origin_size

### B.10 Seasonality

*Terms introduced in this section:* Seasonality, STL. Definitions are given in the Glossary.

#### B.10.1 Seasonality of enrolments

**Method.** Seasonality of enrolments: mean number of new CDAs by calendar month across years, with a 95% interval (1.96 standard errors across years).

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Full years only.

**How to read.** Vertical bars by calendar month; the vertical axis gives new CDAs in thousands; error bars give the interval; the dotted line marks the overall monthly mean.

**Interpretation.** Months whose interval lies entirely above or below the line show a systematic seasonal deviation.

*Output:* Figure `fig_seasonality_enrolment`.

In [ ]:
season = monthly_all.with_columns(pl.col("month").dt.month().alias("m"), pl.col("month").dt.year().alias("y")) \
    .filter(pl.col("y") < SNAP_YEAR).group_by("m").agg(pl.col("n_new").mean().alias("mean"), pl.col("n_new").std().alias("sd"),
                                                      pl.len().alias("n")).sort("m")
season = season.with_columns((1.96 * pl.col("sd") / pl.col("n").sqrt()).alias("ci"))
fig = go.Figure(go.Bar(x=season["m"], y=season["mean"] / 1e3, marker_color=CEMEPI["deep_teal"],
                       error_y=dict(type="data", array=(season["ci"] / 1e3).to_list(), color=CEMEPI["sand"])))
fig.add_hline(y=season["mean"].mean() / 1e3, line_dash="dot", line_color=CEMEPI["grey"])
fig.update_layout(xaxis=dict(title="Calendar month", tickmode="array", tickvals=list(range(1, 13))), yaxis_title="New ICMS CDAs (thousand, mean)")
save_fig(fig, "fig_seasonality_enrolment")
fig

#### B.10.2 Monthly collection

**Method.** Monthly collection with peaks highlighted.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** The horizontal axis gives the month; the vertical axis collection in R$ billion; months above the 95th percentile are labelled with their date; cream bands mark programme windows.

**Interpretation.** Peaks inside the bands are associated with programmes; isolated peaks outside them point to other events.

*Output:* Figure `fig_monthly_collection`.

In [ ]:
coll = monthly_all.select("month", "collection").with_columns(bn(pl.col("collection")).alias("bn"))
p95 = coll["bn"].quantile(0.95)
fig = go.Figure(go.Scatter(x=coll["month"], y=coll["bn"], mode="lines", line_color=CEMEPI["deep_teal"]))
for j, r in enumerate(coll.filter(pl.col("bn") > p95).iter_rows(named=True)):
    fig.add_annotation(x=r["month"], y=r["bn"], text=r["month"].strftime("%b %Y"), showarrow=False, yshift=8, font_size=9,
                       xanchor="right" if j % 2 == 0 else "left")
shade_programmes(fig)
fig.update_layout(yaxis_title="Monthly collection (R$ bn)")
save_fig(fig, "fig_monthly_collection", width="double")
fig

#### B.10.3 Seasonal decomposition

**Method.** Robust STL decomposition (Seasonal-Trend decomposition using Loess; Cleveland et al., 1990) of log monthly collection into trend, a twelve-month seasonal component that may evolve slowly, and a remainder. The robust version down-weights outliers so that programme peaks do not distort trend and seasonality.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Full years only.

**How to read.** Decomposition figure: panels from top to bottom give the observed series, trend, seasonal component and remainder. Sub-series figure: one panel per calendar month, with that month's values across years and a horizontal line at their mean; the legend identifies the two elements.

**Interpretation.** Differences between the monthly means describe the seasonal pattern; trends within panels show whether it changes over time; large spikes in the remainder are irregular events.

*Output:* Figures `fig_stl`, `fig_seasonal_subseries`.

In [ ]:
from statsmodels.tsa.seasonal import STL

y = monthly_all.filter(pl.col("month") < pl.date(SNAP_YEAR, 1, 1)).select("month", "collection")
stl = STL(np.log(y["collection"].to_numpy()), period=12, robust=True).fit()
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.03)
for r, (series, name) in enumerate(((stl.observed, "log collection"), (stl.trend, "trend"), (stl.seasonal, "seasonal"), (stl.resid, "remainder")), start=1):
    fig.add_trace(go.Scatter(x=y["month"], y=series, mode="lines", line_color=color(r - 1)), r, 1)
    fig.update_yaxes(title_text=name, row=r, col=1)
save_fig(fig, "fig_stl", width="double", ratio=0.9)
fig.show()

sub = y.with_columns(pl.col("month").dt.month().alias("m"), pl.col("month").dt.year().alias("yr"), pl.col("collection").log().alias("lc"))
fig = make_subplots(rows=1, cols=12, shared_yaxes=True, horizontal_spacing=0.005)
for m in range(1, 13):
    d = sub.filter(pl.col("m") == m).sort("yr")
    fig.add_trace(go.Scatter(x=d["yr"], y=d["lc"], mode="lines", line_color=CEMEPI["deep_teal"], name="Collection in that month, by year",
                             legendgroup="y", showlegend=m == 1), 1, m)
    fig.add_trace(go.Scatter(x=[d["yr"].min(), d["yr"].max()], y=[d["lc"].mean()] * 2, mode="lines", line_color=CEMEPI["sand"],
                             name="Mean across years", legendgroup="m", showlegend=m == 1), 1, m)
    fig.update_xaxes(showticklabels=False, title_text=str(m), row=1, col=m)
fig.update_yaxes(title_text="Monthly collection (log R$)", row=1, col=1)
fig.update_layout(showlegend=True, legend=dict(orientation="h", y=-0.3),
                  annotations=[*fig.layout.annotations, dict(text="Calendar month", x=0.5, y=-0.12, xref="paper", yref="paper", showarrow=False)])
save_fig(fig, "fig_seasonal_subseries", width="double")
fig

#### B.10.4 Seasonality by origin

**Method.** Seasonal profile of enrolments by origin, normalised by each origin's monthly mean.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Declared and assessed debts.

**How to read.** The horizontal axis gives the calendar month; the vertical axis enrolments relative to the origin's mean (1 = average month); one line per origin, identified in the legend.

**Interpretation.** Different profiles would be consistent with assessed debts following the calendar of administrative appeals rather than that of declarations.

*Output:* Figure `fig_seasonality_by_type`.

In [ ]:
season_origin = monthly.filter(pl.col("ICMS_ORIGIN").is_in(["declared", "assessed"])).with_columns(
    pl.col("month").dt.month().alias("m")).group_by("ICMS_ORIGIN", "m").agg(pl.col("n_new").mean()).with_columns(
    (pl.col("n_new") / pl.col("n_new").mean().over("ICMS_ORIGIN")).alias("index"))
fig = lines(season_origin, "m", "index", "ICMS_ORIGIN", y_title="Enrolments relative to origin mean", x_title="Calendar month")
fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)))
save_fig(fig, "fig_seasonality_by_type")
fig

### B.11 Settlement programmes

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### B.11.1 Programme calendar

**Method.** Calendar of settlement programmes.

**Data.** Programme regulations.

**How to read.** One row per programme, with dates and main terms.

**Interpretation.** The windows defined here are used throughout the notebook.

*Output:* Table `tab_editais`.

In [ ]:
editais = pl.DataFrame([
    ("PEP 2019", "Programa Especial de Parcelamento do ICMS", "2019", "Reduced fines and interest; instalments for ICMS debts"),
    ("Acordo Paulista 2024", "Transação tributária (Lei 17.843/2023)", "2024", "Negotiated settlements with discounts by recoverability class"),
], schema=["programme", "instrument", "year", "main_terms"], orient="row").with_columns(
    pl.Series("window_start", [p[1] for p in PROGRAMMES]), pl.Series("window_end", [p[2] for p in PROGRAMMES]))
save_tab(editais, "tab_editais")
editais

#### B.11.2 Programme windows

**Method.** Collection around programme start, indexed to the pre-programme mean.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** The horizontal axis gives months relative to the start (0 = first month, dotted line); the vertical axis collection indexed so that the pre-programme mean equals 100; one line per programme, identified in the legend.

**Interpretation.** Indexing removes differences in level, so the curves compare the relative size and timing of each programme's effect.

*Output:* Figure `fig_edital_windows`.

In [ ]:
rows = []
for name, start, _ in PROGRAMMES:
    s = np.datetime64(start, "M")
    d = monthly_all.with_columns(((pl.col("month").dt.year() - int(start[:4])) * 12 + pl.col("month").dt.month() - int(start[5:7])).alias("k")) \
        .filter(pl.col("k").is_between(-12, 12))
    base = d.filter(pl.col("k") < 0).select(pl.col("collection").mean(), pl.col("n_payers").mean())
    rows.append(d.with_columns((100 * pl.col("collection") / base["collection"][0]).alias("collection_idx"),
                               (100 * pl.col("n_payers") / base["n_payers"][0]).alias("payers_idx"), pl.lit(name).alias("programme")))
win = pl.concat(rows)
fig = lines(win, "k", "collection_idx", "programme", y_title="Collection (pre-programme mean = 100)", x_title="Months relative to programme start")
fig.add_vline(x=0, line_dash="dot", line_color=CEMEPI["grey"])
save_fig(fig, "fig_edital_windows", width="double")
fig

#### B.11.3 Arrears before programmes

**Method.** New enrolments before each programme, indexed to months −24 to −13.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** The horizontal axis gives months before the start; the vertical axis new enrolments indexed so that the mean of months −24 to −13 equals 100; one line per programme, identified in the legend.

**Interpretation.** A rise in the final months would be consistent with arrears building up in anticipation of favourable terms; isolated spikes may reflect batch enrolments.

*Output:* Figure `fig_pre_programme`.

In [ ]:
pre = []
for name, start, _ in PROGRAMMES:
    d = monthly_all.with_columns(((pl.col("month").dt.year() - int(start[:4])) * 12 + pl.col("month").dt.month() - int(start[5:7])).alias("k")) \
        .filter(pl.col("k").is_between(-24, 0))
    base = d.filter(pl.col("k") <= -13)["n_new"].mean()
    pre.append(d.with_columns((100 * pl.col("n_new") / base).alias("new_idx"), pl.lit(name).alias("programme")))
fig = lines(pl.concat(pre), "k", "new_idx", "programme", y_title="New enrolments (months -24 to -13 = 100)", x_title="Months before programme start")
save_fig(fig, "fig_pre_programme", width="double")
fig

#### B.11.4 The 2024 peak

**Method.** Decomposition of the change in collection from 2023 to 2024 by origin, age band and debtor size.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year. Years 2023 and 2024.

**How to read.** One row per category within each dimension, with the increase in R$ billion and its share of the total increase within the dimension.

**Interpretation.** Categories with large shares drive the 2024 peak; negative shares indicate categories whose collection fell.

*Output:* Table `tab_2024_peak`.

In [ ]:
peak = collect(panel.filter(pl.col("year").is_in([2023, 2024])).with_columns(band()).with_columns(pl.col("age_band").cast(pl.Utf8))
               .join(pl.DataFrame({"ID_DEBITO": list(top_ids)}).lazy().with_columns(pl.lit("top 1%").alias("size")), on="ID_DEBITO", how="left")
               .with_columns(pl.col("size").fill_null("remainder")))
rows = []
for dim in ("ICMS_ORIGIN", "age_band", "size"):
    g = peak.group_by(dim, "year").agg(pl.col("paid").sum()).pivot(on="year", index=dim, values="paid").rename({dim: "category"})
    rows.append(g.select(pl.col("category").cast(pl.Utf8), pl.col("2023"), pl.col("2024")).with_columns(pl.lit(dim).alias("dimension")))
peak_tab = pl.concat(rows).with_columns(((pl.col("2024") - pl.col("2023")) / 1e9).alias("increase_bn"))
peak_tab = peak_tab.with_columns((100 * pl.col("increase_bn") / pl.col("increase_bn").sum().over("dimension")).alias("pct_of_increase"))
save_tab(peak_tab.select("dimension", "category", "increase_bn", "pct_of_increase"), "tab_2024_peak", fmt="{:,.3f}")
peak_tab

#### B.11.5 Peaks of the observed series

**Method.** Monthly collection as a scatter with peaks highlighted.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** Each point is one month; the vertical axis gives collection in R$ billion; highlighted points above the 95th percentile (dotted line) are labelled with their date; cream bands mark programme windows.

**Interpretation.** The proportion of highlighted points inside the bands indicates how much of the extreme collection is programme-driven.

*Output:* Figure `fig_peaks_scatter`.

In [ ]:
peaks = coll.with_columns((pl.col("bn") > p95).alias("peak"))
fig = go.Figure([go.Scatter(x=peaks.filter(~pl.col("peak"))["month"], y=peaks.filter(~pl.col("peak"))["bn"], mode="markers",
                            marker=dict(size=5, color=CEMEPI["grey"])),
                 go.Scatter(x=peaks.filter(pl.col("peak"))["month"], y=peaks.filter(pl.col("peak"))["bn"], mode="markers+text",
                            marker=dict(size=8, color=CEMEPI["sand"]), text=[m.strftime("%b %Y") for m in peaks.filter(pl.col("peak"))["month"]],
                            textposition="top center", textfont_size=9)])
fig.add_hline(y=p95, line_dash="dot", line_color=CEMEPI["deep_teal"], annotation_text="95th percentile")
shade_programmes(fig)
fig.update_layout(yaxis_title="Monthly collection (R$ bn)")
save_fig(fig, "fig_peaks_scatter", width="double")
fig

## Part C. Statistical properties
The properties are examined in a fixed order because each step conditions the next. The distribution of the variables (C.1) determines whether moment-based statistics are reliable and which transformation is required. Linearity (C.2) then determines, before any association is inspected, whether Pearson's or Spearman's coefficient is appropriate, which avoids choosing the measure after seeing the results. Stationarity and autocorrelation (C.3) must be addressed because correlations between trending or autocorrelated series are spurious (Granger and Newbold, 1974) and their naive *p*-values invalid. Finally, the persistence of recovery (C.4) informs the lag structure of the models in Part D.

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

### C.1 Distribution

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### C.1.1 Moments and zero inflation

**Method.** Moments and zero inflation of the main variables.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. CDA-year variables from a random sample of one million CDA-years with positive stock (1.4).

**How to read.** One row per variable and level (monthly or CDA-year): mean, median, standard deviation, skewness, excess kurtosis and share of zeros.

**Interpretation.** Skewness far from zero and excess kurtosis far above zero indicate departures from normality that make moment-based statistics, such as Pearson's correlation, sensitive to a few observations.

*Output:* Table `tab_moments`.

In [ ]:
Y_VARS = ["collection", "n_payers", "n_new", "stock"]
mm = monthly_all.select("month", *Y_VARS).join(macro_m, on="month", how="left").sort("month")
def moments(x: np.ndarray, name: str, level: str) -> dict:
    x = x[~np.isnan(x)]
    return {"variable": name, "level": level, "n": x.size, "mean": x.mean(), "median": np.median(x), "sd": x.std(ddof=1),
            "skewness": stats.skew(x), "kurtosis": stats.kurtosis(x), "share_zero": float((x == 0).mean())}
cda_sample = collect(panel.filter(pl.col("stock") > 0).select("rate", "paid", "stock")).sample(n=1_000_000, seed=SEED)
moments_tab = pl.DataFrame([moments(mm[c].to_numpy().astype(float), c, "monthly") for c in Y_VARS] +
                           [moments(cda_sample[c].to_numpy().astype(float), c, "CDA-year") for c in ("rate", "paid", "stock")])
save_tab(moments_tab, "tab_moments", fmt="{:,.4g}")
moments_tab

#### C.1.2 Normality

**Method.** Normality tests (Shapiro–Wilk and Jarque–Bera) and normal Q–Q plots, for each series in levels, logarithms and inverse hyperbolic sine.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** Q–Q plots: one panel per series and transformation; the horizontal axis gives standard normal quantiles, the vertical axis standardised sample quantiles; the dotted line is the 45-degree reference. The table gives test statistics and *p*-values.

**Interpretation.** Points on the line indicate normality; points above it at the right end and below it at the left end indicate heavy tails. A transformation that brings points onto the line is preferred.

*Output:* Figure `fig_qq`; Table `tab_normality`.

In [ ]:
TRANSFORMS = {"level": lambda x: x, "log": lambda x: np.log(x[x > 0]), "asinh": np.arcsinh}
rows = []
for c in Y_VARS:
    x = mm[c].drop_nulls().to_numpy().astype(float)
    for t, f in TRANSFORMS.items():
        v = f(x)
        sw, jb = stats.shapiro(v), stats.jarque_bera(v)
        rows.append({"variable": c, "transform": t, "shapiro_W": sw.statistic, "shapiro_p": sw.pvalue,
                     "jb_stat": jb.statistic, "jb_p": jb.pvalue})
normality = pl.DataFrame(rows)
fig = make_subplots(rows=2, cols=len(Y_VARS), horizontal_spacing=0.06, vertical_spacing=0.12,
                    subplot_titles=[f"{c} ({t})" for t in ("level", "log") for c in Y_VARS])
for j, c in enumerate(Y_VARS, start=1):
    for i, t in enumerate(("level", "log"), start=1):
        v = np.sort(TRANSFORMS[t](mm[c].drop_nulls().to_numpy().astype(float)))
        q = stats.norm.ppf((np.arange(1, v.size + 1) - 0.5) / v.size)
        z = (v - v.mean()) / v.std()
        fig.add_trace(go.Scatter(x=q, y=z, mode="markers", marker=dict(size=3, color=CEMEPI["deep_teal"])), i, j)
        fig.add_trace(go.Scatter(x=[-3, 3], y=[-3, 3], mode="lines", line=dict(color=CEMEPI["sand"], dash="dot")), i, j)
fig.update_annotations(font_size=9)
save_fig(fig, "fig_qq", width="double", ratio=0.6)
save_tab(normality, "tab_normality", fmt="{:.4g}")
fig

#### C.1.3 Transformations

**Method.** Choice of variance-stabilising transformations: the Box–Cox parameter λ is estimated by maximum likelihood (λ = 0 corresponds to the logarithm, λ = 1 to no transformation); the inverse hyperbolic sine is used for series with zeros.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** One row per series with the estimated λ and the transformation adopted.

**Interpretation.** λ close to 0 supports the logarithm; λ far from both 0 and 1 suggests a power transformation.

*Output:* Table `tab_transforms`.

In [ ]:
from scipy.stats import boxcox
rows = []
for c in Y_VARS:
    x = mm[c].drop_nulls().to_numpy().astype(float)
    best = min(normality.filter(pl.col("variable") == c).iter_rows(named=True), key=lambda r: -r["shapiro_p"])
    lam = boxcox(x[x > 0])[1] if (x > 0).all() else float("nan")
    rows.append({"variable": c, "boxcox_lambda": lam, "has_zeros": bool((x == 0).any()),
                 "selected": "asinh" if (x == 0).any() else best["transform"]})
transforms = pl.DataFrame(rows)
SELECTED = dict(zip(transforms["variable"], transforms["selected"]))
save_tab(transforms, "tab_transforms", fmt="{:.3f}")
transforms

### C.2 Linearity and the choice of correlation coefficient

*Terms introduced in this section:* Pearson *r*, Spearman ρ, Kendall τ, LOWESS, RESET, Harvey–Collier. Definitions are given in the Glossary.

#### C.2.1 Visual assessment

**Method.** Visual check of functional form with LOWESS smoothers, which fit weighted local regressions over a moving window and trace the relation without assuming a functional form.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** Grid of scatter plots: rows are portfolio series, columns macroeconomic indicators; each panel shows the points and the smoother.

**Interpretation.** A smoother close to a straight line supports a linear relation; marked curvature or a flat smoother argues against Pearson's coefficient.

*Output:* Figure `fig_lowess_grid`.

In [ ]:
import statsmodels.api as sm
lowess = sm.nonparametric.lowess
X_SHOW = ["selic", "gdp_monthly", "credit_firms", "unemployment", "ipca", "hh_indebtedness"]
fig = make_subplots(rows=len(Y_VARS), cols=len(X_SHOW), horizontal_spacing=0.02, vertical_spacing=0.03)
for i, yv in enumerate(Y_VARS, start=1):
    for j, xv in enumerate(X_SHOW, start=1):
        d = mm.select(xv, yv).drop_nulls()
        x, y = d[xv].to_numpy().astype(float), TRANSFORMS[SELECTED[yv]](d[yv].to_numpy().astype(float))
        fit = lowess(y, x[: y.size], frac=0.6)
        fig.add_trace(go.Scatter(x=x[: y.size], y=y, mode="markers", marker=dict(size=2.5, color=CEMEPI["grey"])), i, j)
        fig.add_trace(go.Scatter(x=fit[:, 0], y=fit[:, 1], mode="lines", line=dict(color=CEMEPI["deep_teal"], width=1.5)), i, j)
        if i == len(Y_VARS):
            fig.update_xaxes(title_text=xv, row=i, col=j, title_font_size=9)
        if j == 1:
            fig.update_yaxes(title_text=yv, row=i, col=j, title_font_size=9)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
save_fig(fig, "fig_lowess_grid", width="double", ratio=0.75)
fig

#### C.2.2 Tests of functional form

**Method.** Formal tests of linearity: Ramsey's RESET test (joint significance of powers of the fitted values), the Harvey–Collier test (mean of recursive residuals) and the comparison of linear and spline fits by the Akaike information criterion.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** One row per pair of variables, with test statistics, *p*-values and AIC difference.

**Interpretation.** Rejection by the tests or a lower AIC for the spline indicates a nonlinear relation.

*Output:* Table `tab_linearity`.

In [ ]:
from statsmodels.stats.diagnostic import linear_reset, linear_harvey_collier
from statsmodels.gam.api import GLMGam, BSplines
rows = []
for yv in Y_VARS:
    for xv in MACRO_VARS:
        d = mm.select(xv, yv).drop_nulls()
        if d.height < 30:
            continue
        x = d[xv].to_numpy().astype(float)
        y = np.arcsinh(d[yv].to_numpy().astype(float)) if SELECTED[yv] == "asinh" else TRANSFORMS[SELECTED[yv]](d[yv].to_numpy().astype(float))
        X = sm.add_constant(x)
        ols = sm.OLS(y, X).fit()
        reset = linear_reset(ols, power=3, use_f=True)
        try:
            hc_p = float(linear_harvey_collier(ols)[1])
        except Exception:
            hc_p = float("nan")
        gam = GLMGam(y, exog=np.ones((y.size, 1)), smoother=BSplines(x[:, None], df=[6], degree=[3])).fit()
        rows.append({"y": yv, "x": xv, "reset_p": float(reset.pvalue), "harvey_collier_p": hc_p,
                     "aic_linear": ols.aic, "aic_spline": gam.aic, "spline_better": bool(gam.aic + 2 < ols.aic)})
linearity = pl.DataFrame(rows)
save_tab(linearity, "tab_linearity", fmt="{:.4g}")
linearity

#### C.2.3 Decision rule

**Method.** Pre-specified rule for the correlation coefficient: Spearman's ρ (with Kendall's τ as robustness check) whenever linearity is rejected at the 5% level or the variables are heavy-tailed, Pearson's *r* otherwise. Setting the rule before inspecting correlations avoids choosing the coefficient on the basis of the results.

**Data.** Results of C.1 and C.2.

**How to read.** One row per pair, with the rule's outcome and the difference between coefficients.

**Interpretation.** Large differences between coefficients confirm that the choice matters for that pair.

*Output:* Table `tab_corr_method_decision`.

In [ ]:
heavy = {r["variable"]: abs(r["kurtosis"]) > 3 for r in moments_tab.filter(pl.col("level") == "monthly").iter_rows(named=True)}
decision = linearity.with_columns(
    ((pl.col("reset_p") < 0.05) | pl.col("spline_better") | pl.col("y").replace_strict(heavy, default=False)).alias("nonlinear_or_heavy"))
decision = decision.with_columns(pl.when(pl.col("nonlinear_or_heavy")).then(pl.lit("spearman")).otherwise(pl.lit("pearson")).alias("method"))
div = []
for r in decision.iter_rows(named=True):
    d = mm.select(r["x"], r["y"]).drop_nulls()
    a, b = d[r["x"]].to_numpy().astype(float), d[r["y"]].to_numpy().astype(float)
    div.append(abs(corr_pair(a, b, "pearson")[0] - corr_pair(a, b, "spearman")[0]))
decision = decision.with_columns(pl.Series("abs_pearson_minus_spearman", div))
METHOD = {(r["y"], r["x"]): r["method"] for r in decision.iter_rows(named=True)}
GLOBAL_METHOD = "spearman" if decision["method"].eq("spearman").mean() >= 0.5 else "pearson"
save_tab(decision.select("y", "x", "reset_p", "spline_better", "method", "abs_pearson_minus_spearman"), "tab_corr_method_decision", fmt="{:.3f}")
print(f"Share of pairs using Spearman: {decision['method'].eq('spearman').mean():.0%}; global method: {GLOBAL_METHOD}")
decision

### C.3 Stationarity and autocorrelation

*Terms introduced in this section:* Stationarity, unit root, ADF, KPSS, Zivot–Andrews, ACF, PACF, Ljung–Box, pre-whitening. Definitions are given in the Glossary.

#### C.3.1 Unit roots

**Method.** Unit-root and stationarity tests. The augmented Dickey–Fuller (ADF) test has a unit root as null hypothesis, the KPSS test has stationarity as null; the Zivot–Andrews test allows one structural break at an unknown date, which matters here because programmes may produce breaks that ADF would mistake for a unit root.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** One row per series and test, with statistic, *p*-value and, for Zivot–Andrews, the estimated break date.

**Interpretation.** ADF rejecting and KPSS not rejecting indicates stationarity; the reverse indicates a unit root; disagreement indicates a borderline case to be differenced conservatively.

*Output:* Table `tab_stationarity`.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss, zivot_andrews
import warnings
def unit_root_row(x: np.ndarray, name: str) -> dict:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adf = adfuller(x, autolag="AIC")
        kp = kpss(x, regression="c", nlags="auto")
        za = zivot_andrews(x, regression="c", autolag="AIC")
    return {"series": name, "adf_stat": adf[0], "adf_p": adf[1], "kpss_stat": kp[0], "kpss_p": kp[1],
            "za_stat": za[0], "za_p": za[1], "za_break_index": int(za[4]),
            "stationary": bool(adf[1] < 0.05 and kp[1] > 0.05)}
series = {c: TRANSFORMS[SELECTED[c]](mm[c].drop_nulls().to_numpy().astype(float)) for c in Y_VARS}
series |= {c: mm[c].drop_nulls().to_numpy().astype(float) for c in MACRO_VARS}
stationarity = pl.DataFrame([unit_root_row(v, k) for k, v in series.items()] +
                            [unit_root_row(np.diff(v), f"d_{k}") for k, v in series.items()])
save_tab(stationarity, "tab_stationarity", fmt="{:.4g}")
stationarity

#### C.3.2 Seasonal unit roots

**Method.** Seasonal unit-root tests: an OCSB-type regression testing the coefficient on the lagged seasonal level, and a Canova–Hansen-type test of the stability of seasonal dummies across the two halves of the sample.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** One row per series with statistics and *p*-values.

**Interpretation.** Evidence of a seasonal unit root would require seasonal differencing before association analysis.

*Output:* Table `tab_seasonal_unitroot`.

In [ ]:
def ocsb_like(x: np.ndarray, m: int = 12) -> dict:
    # Seasonal-difference regression test (OCSB form): significance of the seasonal unit-root coefficient
    dx = np.diff(x)
    sdx = x[m:] - x[:-m]
    y = sdx[1:] - sdx[:-1]
    X = sm.add_constant(np.column_stack([dx[m - 1:-1], sdx[:-1]]))
    res = sm.OLS(y, X).fit()
    return {"ocsb_t_seasonal": float(res.tvalues[2]), "ocsb_p": float(res.pvalues[2])}

def canova_hansen_proxy(x: np.ndarray, m: int = 12) -> dict:
    # Stability of seasonal dummies across halves of the sample (Chow-type F test)
    n = x.size
    months = np.arange(n) % m
    D = np.eye(m)[months]
    half = (np.arange(n) >= n // 2).astype(float)[:, None]
    full = sm.OLS(x, np.column_stack([D, D * half])).fit()
    restricted = sm.OLS(x, D).fit()
    f, p, _ = full.compare_f_test(restricted)
    return {"seasonal_stability_F": float(f), "seasonal_stability_p": float(p)}

seasonal_ur = pl.DataFrame([{"series": c, **ocsb_like(series[c]), **canova_hansen_proxy(series[c])} for c in Y_VARS])
save_tab(seasonal_ur, "tab_seasonal_unitroot", fmt="{:.4g}")
seasonal_ur

#### C.3.3 Autocorrelation

**Method.** Autocorrelation (ACF) and partial autocorrelation (PACF) functions, and Ljung–Box tests. The ACF gives the correlation between a series and its own lags; the PACF gives the same after removing shorter lags.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month.

**How to read.** One row of panels per series, ACF on the left and PACF on the right; the horizontal axis gives the lag in months, bars give the coefficients and dotted lines the approximate 95% bounds ±1.96/√n. The table gives Ljung–Box tests at lags 6, 12 and 24.

**Interpretation.** Bars beyond the bounds indicate significant autocorrelation. A slowly decaying ACF is typical of a trending or non-stationary series; a PACF that cuts off after a few lags suggests an autoregressive process of that order; spikes at lag 12 indicate seasonality.

*Output:* Figure `fig_acf_pacf`; Table `tab_ljungbox`.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
LAGS = 24
fig = make_subplots(rows=len(Y_VARS), cols=2, shared_xaxes=True, vertical_spacing=0.04, column_titles=["ACF", "PACF"])
lb_rows = []
for i, c in enumerate(Y_VARS, start=1):
    x = series[c]
    band95 = 1.96 / np.sqrt(x.size)
    for j, vals in enumerate((acf(x, nlags=LAGS)[1:], pacf(x, nlags=LAGS)[1:]), start=1):
        fig.add_trace(go.Bar(x=list(range(1, LAGS + 1)), y=vals, marker_color=CEMEPI["deep_teal"]), i, j)
        fig.add_hline(y=band95, line_dash="dot", line_color=CEMEPI["sand"], row=i, col=j)
        fig.add_hline(y=-band95, line_dash="dot", line_color=CEMEPI["sand"], row=i, col=j)
    fig.update_yaxes(title_text=c, row=i, col=1, title_font_size=9)
    lb = acorr_ljungbox(x, lags=[6, 12, 24], return_df=True)
    lb_rows += [{"series": c, "lag": int(l), "Q": float(q), "p_value": float(p)} for l, q, p in zip(lb.index, lb["lb_stat"], lb["lb_pvalue"])]
fig.update_xaxes(title_text="Lag (months)", row=len(Y_VARS))
fig.add_annotation(text="Dotted lines: ±1.96/√n bounds", x=1, y=1.04, xref="paper", yref="paper", showarrow=False, xanchor="right", font_size=9)
save_fig(fig, "fig_acf_pacf", width="double", ratio=0.9)
save_tab(pl.DataFrame(lb_rows), "tab_ljungbox", fmt="{:.4g}")
fig

#### C.3.4 Transformations for association analysis

**Method.** Transformations for association analysis: differencing where C.3.1 indicates a unit root, followed by pre-whitening, in which an autoregressive model is fitted to each macroeconomic series and the same filter is applied to the portfolio series, so that the remaining cross-correlations are not driven by shared autocorrelation.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** One row per series with the differencing order and the filter applied.

**Interpretation.** Associations computed after these steps are less exposed to spurious correlation between trending series.

*Output:* Table `tab_transform_plan`.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
stat_map = dict(zip(stationarity["series"], stationarity["stationary"]))
plan, PREPARED = [], {}
for k, v in series.items():
    diff = not stat_map.get(k, False)
    z = np.diff(v) if diff else v
    plan.append({"series": k, "transform": SELECTED.get(k, "level"), "differenced": diff})
    PREPARED[k] = z
prepared_df = mm.select("month").slice(1).with_columns([pl.Series(k, (v if v.size == mm.height - 1 else v[1:]).astype(float))
                                                          for k, v in PREPARED.items() if v.size >= mm.height - 1])

def prewhiten(x: np.ndarray, y: np.ndarray, order=(2, 0, 0)) -> tuple[np.ndarray, np.ndarray]:
    m = ~(np.isnan(x) | np.isnan(y))
    fit = ARIMA(x[m], order=order).fit()
    phi = np.r_[1, -fit.arparams]
    fx = np.convolve(x[m], phi, mode="valid")
    fy = np.convolve(y[m], phi, mode="valid")
    return fx, fy

save_tab(pl.DataFrame(plan), "tab_transform_plan")
pl.DataFrame(plan)

### C.4 Persistence of recovery

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### C.4.1 Persistence at the CDA level

**Method.** Persistence of recovery at the CDA level: correlation between a CDA's recovery rate in year *t* and in years *t* − 1 to *t* − 3, with Pearson and Spearman coefficients.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** Heat map: rows are coefficients, columns lags, colour and value give the correlation. Bar chart: the same values as grouped bars, identified in the legend.

**Interpretation.** A Spearman coefficient well above Pearson indicates persistence in ranks (paying CDAs keep paying) weakened in the linear measure by extreme rates.

*Output:* Figures `fig_corr_pearson_lags`, `fig_autocorr_taxa_lags`.

In [ ]:
pl_sample = collect(panel.filter(pl.col("stock") > 0).select("rate", "rate_l1", "rate_l2", "rate_l3")).sample(n=2_000_000, seed=SEED)
rows = []
for m in ("pearson", "spearman"):
    for k in (1, 2, 3):
        r, p = corr_pair(pl_sample["rate"].to_numpy().astype(float), pl_sample[f"rate_l{k}"].to_numpy().astype(float), m)
        rows.append({"method": m, "lag": f"t-{k}", "r": r})
lag_corr = pl.DataFrame(rows)
fig = heatmap(lag_corr.pivot(on="lag", index="method", values="r"), "method")
save_fig(fig, "fig_corr_pearson_lags")
fig.show()
fig = go.Figure([go.Bar(x=d["lag"], y=d["r"], name=k[0], marker_color=color(i)) for i, (k, d) in enumerate(lag_corr.group_by("method", maintain_order=True))])
fig.update_layout(barmode="group", yaxis_title="Correlation with rate in year t", showlegend=True, legend=dict(orientation="h", y=1.1))
save_fig(fig, "fig_autocorr_taxa_lags")
fig

#### C.4.2 Persistence at the portfolio level

**Method.** Autocorrelation of the portfolio recovery rate. The monthly rate is collection in month *t* divided by the stock at the end of month *t* − 1, computed only when that stock is positive; the annual rate is from B.1.1.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Annual rates 2016–2026.

**How to read.** Left: monthly autocorrelations (lags in months); right: annual autocorrelations (lags in years); dotted lines give the approximate 95% bounds.

**Interpretation.** With about ten annual observations, the annual coefficients are imprecise and should not be read individually.

*Output:* Figure `fig_portfolio_acf`.

In [ ]:
monthly_rate = monthly_all.sort("month").with_columns(pl.when(pl.col("stock").shift(1) > 0).then(pl.col("collection") / pl.col("stock").shift(1)).alias("rate")) \
    .filter(pl.col("rate").is_not_null() & pl.col("rate").is_finite())
ac_m = acf(monthly_rate["rate"].to_numpy(), nlags=24)[1:]
ac_a = acf(annual["portfolio_rate"].drop_nulls().to_numpy(), nlags=min(5, annual.height - 2))[1:]
fig = make_subplots(rows=1, cols=2, column_titles=["Monthly", "Annual"], horizontal_spacing=0.1)
fig.add_trace(go.Bar(x=list(range(1, 25)), y=ac_m, marker_color=CEMEPI["deep_teal"]), 1, 1)
fig.add_trace(go.Bar(x=list(range(1, ac_a.size + 1)), y=ac_a, marker_color=CEMEPI["sand"]), 1, 2)
fig.add_hline(y=1.96 / np.sqrt(monthly_rate.height), line_dash="dot", row=1, col=1)
fig.add_hline(y=-1.96 / np.sqrt(monthly_rate.height), line_dash="dot", row=1, col=1)
fig.add_hline(y=1.96 / np.sqrt(annual.height), line_dash="dot", row=1, col=2)
fig.add_hline(y=-1.96 / np.sqrt(annual.height), line_dash="dot", row=1, col=2)
fig.update_xaxes(title_text="Lag (months)", row=1, col=1)
fig.update_xaxes(title_text="Lag (years)", row=1, col=2)
fig.update_yaxes(title_text="Autocorrelation of the portfolio recovery rate", row=1, col=1)
fig.add_annotation(text="Dotted lines: ±1.96/√n bounds", x=1, y=1.08, xref="paper", yref="paper", showarrow=False, xanchor="right", font_size=9)
save_fig(fig, "fig_portfolio_acf", width="double")
fig

## Part D. Macroeconomic conditions

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

### D.1 Monthly correlations

*Terms introduced in this section:* Cross-correlation function. Definitions are given in the Glossary.

#### D.1.1 Contemporaneous and short lags

**Method.** Correlation matrices between monthly portfolio series and macroeconomic indicators at lags of 0 to 3 months, using the coefficient selected in C.2.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** One heat map per lag: rows are portfolio series (levels and first differences, prefixed d_), columns indicators; colour and printed value give the correlation on a diverging scale fixed on [−1, 1] (teal positive, sand negative, white zero).

**Interpretation.** Strong correlations in levels that vanish in first differences typically reflect common trends rather than short-run relations; no inference is drawn here without the corrections of D.5.

*Output:* Figures `fig_corr_macro_lag0_mensal`, `fig_corr_macro_lag1_mensal`, `fig_corr_macro_lag2_mensal`, `fig_corr_macro_lag3_mensal`.

In [ ]:
Y_MONTHLY = ["collection", "n_payers", "n_new", "d_collection", "d_n_payers", "d_n_new"]
mmd = monthly_all.select("month", *Y_MONTHLY).join(macro_m, on="month", how="left").sort("month")
def corr_matrix(frame: pl.DataFrame, ys, xs, lag: int, freq_col: str) -> pl.DataFrame:
    rows = []
    for yv in ys:
        row = {"y": yv}
        for xv in xs:
            x = frame[xv].shift(lag).to_numpy().astype(float)
            row[xv] = corr_pair(x, frame[yv].to_numpy().astype(float), METHOD.get((yv.removeprefix("d_"), xv), GLOBAL_METHOD))[0]
        rows.append(row)
    return pl.DataFrame(rows)
for lag in range(4):
    fig = heatmap(corr_matrix(mmd, Y_MONTHLY, MACRO_VARS, lag, "month"), "y")
    save_fig(fig, f"fig_corr_macro_lag{lag}_mensal", width="double")
    if lag < 3:
        fig.show()
fig

#### D.1.2 Lags of 1 to 12 months

**Method.** Cross-correlation functions between pre-whitened series (C.3.4) at lags of 1 to 12 months.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** One heat-map panel per portfolio series; rows are indicators, columns lags in months; the colour bar gives the correlation on [−1, 1].

**Interpretation.** Because pre-whitening removes each series' own autocorrelation, an isolated dark cell indicates that past values of the indicator are associated with the portfolio series beyond common persistence.

*Output:* Figure `fig_ccf_monthly`; Table `tab_ccf_monthly`.

In [ ]:
rows = []
for yv in ("collection", "n_payers", "n_new"):
    for xv in MACRO_VARS:
        if xv not in PREPARED or yv not in PREPARED:
            continue
        x, y = PREPARED[xv], PREPARED[yv]
        n = min(x.size, y.size)
        fx, fy = prewhiten(x[-n:], y[-n:])
        for lag in range(1, 13):
            r, p = corr_pair(fx[:-lag], fy[lag:], METHOD.get((yv, xv), GLOBAL_METHOD))
            rows.append({"y": yv, "x": xv, "lag_months": lag, "r": r, "p_value": p, "n": int(fx.size - lag)})
ccf_m = pl.DataFrame(rows)
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, column_titles=["collection", "n_payers", "n_new"], horizontal_spacing=0.03)
for j, yv in enumerate(("collection", "n_payers", "n_new"), start=1):
    mat = ccf_m.filter(pl.col("y") == yv).pivot(on="lag_months", index="x", values="r")
    fig.add_trace(go.Heatmap(z=mat.drop("x").to_numpy(), x=list(range(1, 13)), y=mat["x"].to_list(), colorscale=DIV, zmin=-1, zmax=1, zmid=0,
                             showscale=j == 3), 1, j)
fig.update_xaxes(title_text="Lag (months)")
save_fig(fig, "fig_ccf_monthly", width="double")
save_tab(ccf_m, "tab_ccf_monthly", fmt="{:.4g}")
fig

### D.2 Quarterly correlations

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### D.2.1 Lags of 1 to 4 quarters

**Method.** Correlations at quarterly frequency between collection and indicators lagged one to four quarters.

**Data.** Quarterly aggregates of the monthly series (1.5, 1.6).

**How to read.** Heat map: rows are indicators, columns lags in quarters; colour bar as in D.1.1.

**Interpretation.** The quarterly resolution is intermediate between noisy monthly data and the short annual sample.

*Output:* Figure `fig_corr_quarterly`; Table `tab_corr_quarterly`.

In [ ]:
q = monthly_all.with_columns(pl.col("month").dt.truncate("1q").alias("quarter")).group_by("quarter").agg(
    pl.col("collection").sum(), pl.col("n_new").sum(), pl.col("n_payers").sum(), pl.col("stock").last()).join(macro_q, on="quarter").sort("quarter")
q = q.with_columns([pl.col(c).diff().alias(f"d_{c}") for c in ("collection", "n_new", "n_payers", "stock", *MACRO_VARS)])
rows = []
for yv in ("collection", "n_payers", "n_new", "stock"):
    for xv in MACRO_VARS:
        for lag in range(1, 5):
            r, p = corr_pair(q[f"d_{xv}"].shift(lag).to_numpy().astype(float), q[f"d_{yv}"].to_numpy().astype(float), METHOD.get((yv, xv), GLOBAL_METHOD))
            rows.append({"y": yv, "x": xv, "lag_quarters": lag, "r": r, "p_value": p})
corr_q = pl.DataFrame(rows)
mat = corr_q.filter(pl.col("y") == "collection").pivot(on="lag_quarters", index="x", values="r")
fig = heatmap(mat.rename({c: f"t-{c}" for c in mat.columns if c != "x"}), "x")
save_fig(fig, "fig_corr_quarterly")
save_tab(corr_q, "tab_corr_quarterly", fmt="{:.4g}")
fig

### D.3 Annual correlations

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### D.3.1 Annual portfolio aggregates

**Method.** Correlations between annual portfolio aggregates and indicators at lags of 0 to 3 years.

**Data.** Annual aggregates (B.1.1) and annual macroeconomic series, 2016–2026.

**How to read.** One heat map per lag, rows portfolio aggregates, columns indicators; colour bar as in D.1.1.

**Interpretation.** With about ten observations the sampling error of each coefficient is large; values are reported for completeness rather than inference.

*Output:* Figures `fig_corr_macro_lag0_anual`, `fig_corr_macro_lag1_anual`, `fig_corr_macro_lag2_anual`, `fig_corr_macro_lag3_anual`.

In [ ]:
macro_a = macro_m.with_columns(pl.col("month").dt.year().alias("year")).group_by("year").agg(
    [pl.col(c).last() if c in LOG_VARS else pl.col(c).mean() for c in MACRO_VARS]).sort("year")
ann = annual.select("year", "stock", "collected", "portfolio_rate", "share_paying", "n_cda").join(new_year.select("year", "n_new"), on="year", how="left")
ann_m = ann.join(macro_a, on="year", how="left").sort("year")
Y_ANNUAL = ["portfolio_rate", "share_paying", "collected", "stock", "n_new"]
for lag in range(4):
    fig = heatmap(corr_matrix(ann_m, Y_ANNUAL, MACRO_VARS, lag, "year"), "y")
    save_fig(fig, f"fig_corr_macro_lag{lag}_anual", width="double")
    if lag < 3:
        fig.show()
fig

#### D.3.2 Annual recovery rate

**Method.** Correlations between the annual portfolio recovery rate and each indicator at lags of 0 to 3 years.

**Data.** As in D.3.1.

**How to read.** Heat map: rows are indicators, columns lags; colour bar as in D.1.1.

**Interpretation.** The same caveat on sample size as in D.3.1 applies.

*Output:* Figure `fig_corr_taxa_recuperacao_macro`.

In [ ]:
rate_macro = pl.DataFrame([{"x": xv, **{f"t-{lag}": corr_pair(ann_m[xv].shift(lag).to_numpy().astype(float),
                                                                   ann_m["portfolio_rate"].to_numpy().astype(float), GLOBAL_METHOD)[0]
                                        for lag in range(4)}} for xv in MACRO_VARS])
fig = heatmap(rate_macro, "x")
save_fig(fig, "fig_corr_taxa_recuperacao_macro")
fig

#### D.3.3 New enrolments and stock

**Method.** Correlations between annual new enrolments, year-end stock and the indicators.

**Data.** As in D.3.1.

**How to read.** One row per pair with the coefficient.

**Interpretation.** The same caveat on sample size as in D.3.1 applies.

*Output:* Table `tab_corr_proxies_annual`.

In [ ]:
rows = []
for yv in ("n_new", "stock"):
    for xv in MACRO_VARS:
        for lag in range(4):
            r, p = corr_pair(ann_m[xv].shift(lag).to_numpy().astype(float), ann_m[yv].to_numpy().astype(float), GLOBAL_METHOD)
            rows.append({"y": yv, "x": xv, "lag_years": lag, "r": r, "p_value": p, "n": int(ann_m.height - lag)})
corr_proxies = pl.DataFrame(rows)
save_tab(corr_proxies, "tab_corr_proxies_annual", fmt="{:.4g}")
corr_proxies

### D.4 Panel correlations

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### D.4.1 Full panel

**Method.** Correlation matrix of CDA-level variables in the full panel, with pairwise-complete observations, computed out of core in DuckDB; Spearman coefficients are Pearson correlations of average ranks.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year.

**How to read.** Matrix of coefficients for recovery rate, its lag, stock, age and payment indicator.

**Interpretation.** These coefficients describe the whole panel and serve as the benchmark for the estimation sample.

*Output:* Table `tab_corr_full`.

In [ ]:
PANEL_VARS = ["rate", "rate_l1", "stock", "age", "paid_flag"]
def panel_corr(frame: pl.DataFrame) -> pl.DataFrame:
    rows = []
    for a in PANEL_VARS:
        row = {"variable": a}
        for b in PANEL_VARS:
            row[b] = corr_pair(frame[a].to_numpy().astype(float), frame[b].to_numpy().astype(float), GLOBAL_METHOD)[0]
        rows.append(row)
    return pl.DataFrame(rows)
PANEL_SRC = parts_glob("1.4:panel", PANEL, EXTRACT_PARAMS)
PAIR_SQL = """
WITH d AS (
    SELECT CAST({a} AS DOUBLE) AS a, CAST({b} AS DOUBLE) AS b
    FROM (SELECT rate, rate_l1, LN(stock) AS stock, age, paid_flag FROM read_parquet('{src}') WHERE stock > 0)
    WHERE {a} IS NOT NULL AND {b} IS NOT NULL AND NOT isnan({a}) AND NOT isnan({b})
), r AS (
    -- average ranks for ties, as in scipy.stats.spearmanr
    SELECT a, b,
           RANK() OVER (ORDER BY a) + (COUNT(*) OVER (PARTITION BY a) - 1) / 2.0 AS ra,
           RANK() OVER (ORDER BY b) + (COUNT(*) OVER (PARTITION BY b) - 1) / 2.0 AS rb
    FROM d
)
SELECT '{a}' AS var_a, '{b}' AS var_b, COUNT(*) AS n, CORR(a, b) AS pearson, CORR(ra, rb) AS spearman FROM r
"""
PAIRS = {f"{a}__{b}": PAIR_SQL.format(a=a, b=b, src=PANEL_SRC)
         for i, a in enumerate(PANEL_VARS) for b in PANEL_VARS[i + 1:]}
pair_corr = collect(cached_sql_parts("D.4.1", PAIRS, params=EXTRACT_PARAMS))
lookup = {(r["var_a"], r["var_b"]): r[GLOBAL_METHOD] for r in pair_corr.iter_rows(named=True)}
corr_full = pl.DataFrame([{"variable": a, **{b: 1.0 if a == b else lookup.get((a, b), lookup.get((b, a))) for b in PANEL_VARS}}
                          for a in PANEL_VARS])
save_tab(corr_full, "tab_corr_full", fmt="{:.3f}")
corr_full

#### D.4.2 Estimation sample

**Method.** The same matrix in the estimation sample.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year. Random sample of one million CDA-years used in D.6.

**How to read.** Matrix as in D.4.1.

**Interpretation.** Close agreement with D.4.1 shows that the sample is representative for these variables.

*Output:* Table `tab_corr_sample`.

In [ ]:
est_sample = collect(cached("D.4.2", lambda: panel.filter((pl.col("stock") > 0) & pl.col("rate_l1").is_not_null())
                             .select("ID_DEBITO", "year", "rate", "rate_l1", "stock", "age", "paid_flag", "ICMS_ORIGIN")
                             .collect(engine="streaming").sample(n=1_000_000, seed=SEED), params={**EXTRACT_PARAMS, "n": 1_000_000}))
corr_sample = panel_corr(est_sample.with_columns(pl.col("stock").log()))
save_tab(corr_sample, "tab_corr_sample", fmt="{:.3f}")
corr_sample

### D.5 Statistical significance

*Terms introduced in this section:* Newey–West (HAC) standard errors; moving-block bootstrap. Definitions are given in the Glossary.

#### D.5.1 Robust inference

**Method.** Robust inference for correlations between autocorrelated series: the moving-block bootstrap resamples contiguous blocks of observations and so preserves short-run dependence; results are checked against Newey–West (HAC) standard errors.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** One row per variable pair and lag, with the coefficient, bootstrap confidence interval and *p*-value.

**Interpretation.** Intervals that include zero indicate associations not distinguishable from zero once dependence is accounted for.

In [ ]:
from statsmodels.stats.sandwich_covariance import cov_hac

def block_bootstrap_corr(x: np.ndarray, y: np.ndarray, method: str, block: int = 12, n_boot: int = 999) -> tuple[float, float, float, float]:
    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]
    n = x.size
    r0 = corr_pair(x, y, method)[0]
    rng = np.random.default_rng(SEED)
    starts = np.arange(n - block + 1)
    boots = []
    for _ in range(n_boot):
        idx = np.concatenate([np.arange(s, s + block) for s in rng.choice(starts, size=int(np.ceil(n / block)))])[:n]
        boots.append(corr_pair(x[idx], y[idx], method)[0])
    boots = np.array(boots)
    p = float(2 * min((boots <= 0).mean(), (boots >= 0).mean()))
    return r0, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)), p

def robust_monthly() -> list[dict]:
    rows = []
    for r in ccf_m.iter_rows(named=True):
        x, y = PREPARED[r["x"]], PREPARED[r["y"]]
        n = min(x.size, y.size)
        lag = r["lag_months"]
        est, lo, hi, p = block_bootstrap_corr(x[-n:][:-lag], y[-n:][lag:], METHOD.get((r["y"], r["x"]), GLOBAL_METHOD))
        rows.append({"y": r["y"], "x": r["x"], "lag": lag, "frequency": "monthly", "r": est, "ci_low": lo, "ci_high": hi, "p_boot": p})
    return rows

robust = pl.DataFrame(cached_json("D.5.1", robust_monthly, params={"block": 12, "n_boot": 999}))
robust.sort("p_boot").head(10)

#### D.5.2 Multiple testing

**Method.** Control of the false discovery rate across all variable–lag combinations with the Benjamini–Hochberg procedure at *q* = 0.05: among the associations declared significant, the expected share of false positives is at most 5%.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** Heat map: rows are indicators, columns lags in months; colour and printed value give the correlation with monthly collection; an asterisk marks combinations significant after correction. The table lists the significant associations.

**Interpretation.** Isolated significant cells at scattered lags are weaker evidence than a consistent block across adjacent lags.

*Output:* Figure `fig_heatmap_significant`; Table `tab_assoc_significance`.

In [ ]:
robust = robust.with_columns(pl.Series("q_bh", multipletests(robust["p_boot"].to_numpy(), method="fdr_bh")[1]))
robust = robust.with_columns((pl.col("q_bh") < 0.05).alias("significant"))
sig = robust.filter(pl.col("y") == "collection").sort("lag")
mat = sig.pivot(on="lag", index="x", values="r", sort_columns=True)
flag = sig.pivot(on="lag", index="x", values="significant", sort_columns=True)
cols = [c for c in mat.columns if c != "x"]
marks = [[("" if v is None else f"{v:.2f}" + ("*" if f else "")) for v, f in zip(rv, fv)]
         for rv, fv in zip(mat.select(cols).iter_rows(), flag.select(cols).iter_rows())]
fig = heatmap(mat, "x", colorbar_title="Correlation with<br>collection", text=marks)
fig.add_annotation(text="* significant after Benjamini–Hochberg correction (q < 0.05)", x=0, y=-0.25, xref="paper", yref="paper",
                   showarrow=False, xanchor="left", font_size=9)
fig.update_layout(xaxis_title="Lag (months)")
save_fig(fig, "fig_heatmap_significant")
save_tab(robust.filter(pl.col("significant")).sort("q_bh"), "tab_assoc_significance", fmt="{:.4g}")
fig

### D.6 CDA-level and portfolio-level models

*Terms introduced in this section:* Logit model, odds ratio, fixed effects, variance inflation factor, AR(1). Definitions are given in the Glossary.

#### D.6.1 Model 1: probability of payment

**Method.** Logit model of the probability that a CDA records a payment in a year, as a function of log stock, age and lagged recovery rate, with year fixed effects that absorb all common shocks, including programmes and macroeconomic conditions.

**Data.** ICMS CDA × year panel (1.4), legal entities only, 2016–2026; unit of observation: the CDA-year. Random sample of one million CDA-years with positive stock and an observed lag.

**How to read.** One row per regressor with coefficient, robust standard error, *p*-value and odds ratio.

**Interpretation.** An odds ratio above 1 means higher values of the regressor are associated with higher odds of payment, other regressors held constant.

*Output:* Table `tab_model1_logit`.

In [ ]:
import statsmodels.formula.api as smf
est = est_sample.with_columns(pl.col("stock").log().alias("log_stock"), pl.col("year").cast(pl.Utf8)).to_pandas()

def coef_table(res, label: str, odds: bool = False) -> pl.DataFrame:
    ci = res.conf_int()
    out = pl.DataFrame({"term": res.params.index.tolist(), "coef": res.params.values, "se": res.bse.values,
                        "p_value": res.pvalues.values, "ci_low": ci[0].values, "ci_high": ci[1].values}).with_columns(pl.lit(label).alias("model"))
    return out.with_columns(pl.col("coef").exp().alias("odds_ratio")) if odds else out

m1 = smf.logit("paid_flag ~ log_stock + age + rate_l1 + C(year)", data=est).fit(disp=False, cov_type="HC1")
tab_m1 = coef_table(m1, "logit", odds=True)
save_tab(tab_m1.filter(~pl.col("term").str.starts_with("C(year)")), "tab_model1_logit", fmt="{:.4g}")
print(f"McFadden pseudo-R2 = {m1.prsquared:.4f}; n = {int(m1.nobs):,}")
tab_m1

#### D.6.2 Model 2: CDA recovery rate

**Method.** OLS regression of the CDA recovery rate on its lag, log stock and age, with year fixed effects and heteroskedasticity-robust (HC1) standard errors; CDA-years with rates above one are excluded.

**Data.** As in D.6.1.

**How to read.** One row per regressor with coefficient, standard error and *p*-value; R² and sample size are printed.

**Interpretation.** The coefficient on the lagged rate measures persistence of recovery at the CDA level.

*Output:* Table `tab_model2_ols`.

In [ ]:
m2 = smf.ols("rate ~ rate_l1 + log_stock + age + C(year)", data=est[est["rate"] <= 1]).fit(cov_type="HC1")
tab_m2 = coef_table(m2, "ols")
save_tab(tab_m2.filter(~pl.col("term").str.starts_with("C(year)")), "tab_model2_ols", fmt="{:.4g}")
print(f"R2 = {m2.rsquared:.4f}; n = {int(m2.nobs):,}")
tab_m2

#### D.6.3 Model 3: portfolio recovery rate

**Method.** First-order autoregression of the portfolio recovery rate, r_t = α + ρ r_{t−1} + ε_t, at annual and monthly frequency, with Newey–West standard errors (12 lags) at monthly frequency; only finite rates enter the estimation.

**Data.** Annual rates (B.1.1) and monthly rates (C.4.2).

**How to read.** One row per frequency and term, with coefficient, standard error, *p*-value and number of observations.

**Interpretation.** ρ near 0 indicates that shocks to recovery do not carry over; ρ near 1 indicates highly persistent recovery.

*Output:* Table `tab_model3_ar1`.

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
y_a = annual["portfolio_rate"].drop_nulls().to_numpy()
y_m = monthly_rate["rate"].drop_nulls().to_numpy()
ar_a = AutoReg(y_a[np.isfinite(y_a)], lags=1).fit()
ar_m = AutoReg(y_m[np.isfinite(y_m)], lags=1).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
tab_m3 = pl.DataFrame([{"frequency": f, "term": t, "coef": float(r.params[i]), "se": float(r.bse[i]), "p_value": float(r.pvalues[i]), "n": int(r.nobs)}
                       for f, r in (("annual", ar_a), ("monthly", ar_m)) for i, t in enumerate(("const", "rate_lag1"))])
save_tab(tab_m3, "tab_model3_ar1", fmt="{:.4g}")
tab_m3

#### D.6.4 Model 4: probability of payment and macroeconomic conditions

**Method.** Model 1 with year fixed effects replaced by five annual macroeconomic indicators (Selic, IPCA, IBC-Br, credit to firms, unemployment), each standardised to zero mean and unit variance; collinearity is assessed with variance inflation factors.

**Data.** As in D.6.1, merged with annual macroeconomic series.

**How to read.** Coefficient table as in D.6.1, plus a table of VIFs.

**Interpretation.** Odds ratios refer to a one-standard-deviation change in the indicator. VIFs above about 10 mean that individual coefficients are poorly identified.

*Output:* Tables `tab_model4_logit_macro`, `tab_vif`.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
macro_std = macro_a.with_columns([((pl.col(c) - pl.col(c).mean()) / pl.col(c).std()).alias(c) for c in MACRO_VARS])
est_m = est_sample.with_columns(pl.col("stock").log().alias("log_stock")).join(macro_std, on="year", how="left").drop_nulls(MACRO_VARS).to_pandas()
MAIN_MACRO = ["selic", "ipca", "gdp_monthly", "credit_firms", "unemployment"]
m4 = smf.logit("paid_flag ~ log_stock + age + rate_l1 + " + " + ".join(MAIN_MACRO), data=est_m).fit(disp=False, cov_type="HC1")
tab_m4 = coef_table(m4, "logit_macro", odds=True)
Xv = macro_std.select(MAIN_MACRO).drop_nulls().to_numpy()
vif = pl.DataFrame({"variable": MAIN_MACRO, "vif": [variance_inflation_factor(np.column_stack([np.ones(len(Xv)), Xv]), i + 1) for i in range(len(MAIN_MACRO))]})
save_tab(tab_m4, "tab_model4_logit_macro", fmt="{:.4g}")
save_tab(vif, "tab_vif", fmt="{:.2f}")
tab_m4

#### D.6.5 Model 4b: principal components

**Method.** Model 4 estimated on the principal components of the standardised indicators, which are uncorrelated by construction.

**Data.** As in D.6.4.

**How to read.** Loadings table: weight of each indicator in each component; coefficient table as in D.6.1.

**Interpretation.** Interpret a component's coefficient through its loadings, for example as a general activity or price factor.

*Output:* Tables `tab_pca_loadings`, `tab_model4b_pcs`.

In [ ]:
from sklearn.decomposition import PCA as _PCA
Z = macro_std.drop_nulls(MACRO_VARS)
pca_macro = _PCA(n_components=3, random_state=SEED).fit(Z.select(MACRO_VARS).to_numpy())
loadings = pl.DataFrame({"variable": MACRO_VARS, **{f"PC{i + 1}": pca_macro.components_[i] for i in range(3)}})
pcs = pl.DataFrame({"year": Z["year"], **{f"PC{i + 1}": pca_macro.transform(Z.select(MACRO_VARS).to_numpy())[:, i] for i in range(3)}})
est_pc = est_sample.with_columns(pl.col("stock").log().alias("log_stock")).join(pcs, on="year", how="inner").to_pandas()
m4b = smf.logit("paid_flag ~ log_stock + age + rate_l1 + PC1 + PC2 + PC3", data=est_pc).fit(disp=False, cov_type="HC1")
tab_m4b = coef_table(m4b, "logit_pcs", odds=True)
save_tab(loadings.with_columns(pl.lit(None).alias("_")).drop("_"), "tab_pca_loadings", fmt="{:.3f}")
save_tab(tab_m4b, "tab_model4b_pcs", fmt="{:.4g}")
print("Explained variance:", np.round(pca_macro.explained_variance_ratio_, 3))
tab_m4b

#### D.6.6 Model comparison

**Method.** Comparison of specifications by McFadden's pseudo-R² and likelihood-ratio tests between nested models.

**Data.** Models of D.6.1, D.6.4 and D.6.5.

**How to read.** One row per model or comparison.

**Interpretation.** The fixed-effects model is the upper benchmark for specifications based on annual common variables, because year dummies absorb all of their variation.

*Output:* Table `tab_model_comparison`.

In [ ]:
base = smf.logit("paid_flag ~ log_stock + age + rate_l1", data=est_m).fit(disp=False)
fe = smf.logit("paid_flag ~ log_stock + age + rate_l1 + C(year)", data=est_m.assign(year=est_m["year"].astype(str))).fit(disp=False)
macro_fit = smf.logit("paid_flag ~ log_stock + age + rate_l1 + " + " + ".join(MAIN_MACRO), data=est_m).fit(disp=False)
pc_fit = smf.logit("paid_flag ~ log_stock + age + rate_l1 + PC1 + PC2 + PC3", data=est_pc).fit(disp=False)
def lr(a, b) -> tuple[float, float]:
    stat = 2 * (b.llf - a.llf)
    return float(stat), float(stats.chi2.sf(stat, b.df_model - a.df_model))
comparison = pl.DataFrame([{"model": n, "pseudo_r2": float(r.prsquared), "aic": float(r.aic), "df_model": int(r.df_model),
                            "lr_vs_base": lr(base, r)[0] if n != "CDA variables only" else float("nan"),
                            "p_lr": lr(base, r)[1] if n != "CDA variables only" else float("nan")}
                           for n, r in (("CDA variables only", base), ("Year fixed effects", fe), ("Macro indicators", macro_fit), ("Macro PCs", pc_fit))])
save_tab(comparison, "tab_model_comparison", fmt="{:.4g}")
comparison

### D.7 Dynamic models

*Terms introduced in this section:* ARDL, Granger causality. Definitions are given in the Glossary.

#### D.7.1 Autoregressive distributed-lag models

**Method.** Autoregressive distributed-lag (ARDL) models of each stationary monthly portfolio series on its own lags and on current and lagged indicators, with programme dummies; lag orders selected by an information criterion; Newey–West standard errors.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency. Pooled portfolio and by origin.

**How to read.** One row per model and regressor with coefficient, standard error and *p*-value.

**Interpretation.** Significant lagged indicators suggest delayed associations; coefficients are conditional on the programme dummies.

*Output:* Table `tab_ardl`.

In [ ]:
from statsmodels.tsa.ardl import ardl_select_order
PROG_DUMMY = np.zeros(prepared_df.height)
months = prepared_df["month"].to_numpy()
for _, a, b in PROGRAMMES:
    PROG_DUMMY[(months >= np.datetime64(a)) & (months <= np.datetime64(b))] = 1
rows = []
for yv in ("collection", "n_payers"):
    if yv not in prepared_df.columns:
        continue
    X = prepared_df.select([c for c in MAIN_MACRO if c in prepared_df.columns]).to_pandas()
    sel = ardl_select_order(prepared_df[yv].to_numpy(), maxlag=3, exog=X, maxorder=6, ic="aic", trend="c",
                            fixed=PROG_DUMMY[:, None], seasonal=True, period=12)
    res = sel.model.fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    rows += [{"y": yv, "term": t, "coef": float(c), "se": float(s), "p_value": float(p)}
             for t, c, s, p in zip(res.params.index, res.params.values, res.bse.values, res.pvalues.values)]
ardl = pl.DataFrame(rows)
save_tab(ardl.filter(~pl.col("term").str.contains("s\\(")), "tab_ardl", fmt="{:.4g}")
ardl

#### D.7.2 Granger causality

**Method.** Granger tests: an indicator Granger-causes a portfolio series if its past values improve the prediction of the series beyond the series' own past. The test is an *F* test of the lagged indicator in a vector autoregression with lag length selected by AIC; *q*-values apply the Benjamini–Hochberg correction.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** One row per pair with selected lags, *F* statistic, *p*-value and *q*-value.

**Interpretation.** A significant result means predictive precedence, not causation.

*Output:* Table `tab_granger`.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR
rows = []
for yv in ("collection", "n_payers"):
    for xv in MAIN_MACRO:
        d = prepared_df.select(yv, xv).drop_nulls().to_numpy()
        p_sel = max(1, VAR(d).select_order(maxlags=12).aic)
        res = grangercausalitytests(d, maxlag=[p_sel])[p_sel][0]["ssr_ftest"]
        rows.append({"y": yv, "x": xv, "lags": int(p_sel), "F": float(res[0]), "p_value": float(res[1])})
granger = pl.DataFrame(rows).with_columns(pl.Series("q_bh", multipletests([r["p_value"] for r in rows], method="fdr_bh")[1]))
save_tab(granger, "tab_granger", fmt="{:.4g}")
granger

#### D.7.3 The Selic rate and tax non-payment as financing

**Method.** Interpretation of the Selic coefficients for stock and recovery under the hypothesis that firms use unpaid taxes as financing when credit is expensive (H4b), with and without credit to firms as a control.

**Data.** Monthly ICMS series (1.5), January 2016 to March 2026; unit of observation: the month. Macroeconomic series from 1.6 at the same frequency.

**How to read.** One row per outcome and specification with the Selic coefficient and its robust standard error.

**Interpretation.** Under H4b, a higher Selic would be associated with a larger stock and lower recovery; robustness to the credit control indicates that the association is not merely a credit-cycle effect.

*Output:* Table `tab_selic`.

In [ ]:
selic_rows = []
for yv in ("collection", "stock", "n_payers"):
    if yv not in prepared_df.columns:
        continue
    for spec, extra in (("Selic", []), ("Selic + credit to firms", ["credit_firms"])):
        d = prepared_df.select(yv, "selic", *extra).drop_nulls().to_pandas()
        for lag in (0, 3, 6, 12):
            dd = d.assign(selic_l=d["selic"].shift(lag)).dropna()
            res = smf.ols(f"{yv} ~ selic_l" + "".join(f" + {e}" for e in extra), data=dd).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
            selic_rows.append({"y": yv, "specification": spec, "lag_months": lag, "coef_selic": float(res.params["selic_l"]),
                               "se": float(res.bse["selic_l"]), "p_value": float(res.pvalues["selic_l"]), "n": int(res.nobs)})
selic_tab = pl.DataFrame(selic_rows)
save_tab(selic_tab, "tab_selic", fmt="{:.4g}")
selic_tab

## Part E. Discussion

*No new terms are introduced in this section; terms used here are defined in the Glossary.*

#### E.1 Synthesis

The answers to the research questions are summarised in the manuscript on the basis of the results above.

#### E.2 Limitations

The analysis relies on a single frozen extraction; stocks are reconstructed from monthly snapshots; settlement is only approximated; the annual sample is short; settlement programmes introduce structural breaks; firm-level revenue and indebtedness are not available; and the restriction to ICMS limits generalisation to other taxes.

#### E.3 Ethical considerations

The study uses only data on legal entities. Records whose debtor is identified by a CPF (natural persons) are excluded at extraction, and no name or personal identifier is retained.

#### E.4 Future research

Directions for further work include the recovered-over-recoverable indicator under a validated criterion; explicit modelling of settlement programmes through their attributes, with separate components for regular and programme-driven collection; instalment agreements and their breach; protest; the distinction between inability to pay and concealment of assets; firm indebtedness; and heterogeneity across sectors.

## Correspondence with the manuscript and the dissertation
The structure of this notebook corresponds to the paper and to the master's dissertation prepared with the School of Technology (FT-Unicamp) thesis template.

| Notebook | Paper | Dissertation |
|---|---|---|
| Research questions | Introduction | Ch. 1 Introduction |
| 0–1 | Data | Ch. 3 Data and methods |
| Part A | The portfolio across tax types | Ch. 4 Exploratory analysis |
| Part B | The ICMS portfolio (incl. the most concentrated CDAs) | Ch. 4 Exploratory analysis |
| Part C | Statistical properties (Methods) | Ch. 3 Data and methods |
| Part D | Macroeconomic conditions | Ch. 5 Macroeconomic associations |
| Part E | Discussion, limitations, conclusion | Ch. 6 Discussion; Ch. 7 Conclusion |
| A.1.2, A.3.1, 1.7 | Appendix | Appendices |

Figures (`fig_*.pdf`) and tables (`tab_*.tex`) are shared by the paper and the dissertation.